<a href="https://colab.research.google.com/github/Rohil121/bharat-portfolio-lab/blob/v0.6-ml-trading-research/notebooks/06_ml_trading_research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!printf "Y\n\n" | env BROWSER=echo stdbuf -oL -eL gh auth login \
  --hostname github.com \
  --git-protocol https \
  --web



! First copy your one-time code: 4AFD-66C1
Open this URL to continue in your web browser: https://github.com/login/device
✓ Authentication complete.
- gh config set -h github.com git_protocol https
✓ Configured git protocol
! Authentication credentials saved in plain text
✓ Logged in as Rohil121


# Bharat Portfolio Lab v0.6

## Machine Learning and Trading Research

### Objective

Test whether machine-learning signals can improve the risk-adjusted performance of an Indian equity portfolio after transaction costs, turnover and strict out-of-sample validation.

The initial research universe will be the India 10 portfolio, while the final production modules will support arbitrary eligible Indian listed equities.

## Fixed v0.6 Scope

### Prediction tasks

1. Predict each stock’s forward 21-trading-day return.
2. Estimate the probability of a positive forward return.
3. Estimate the probability of outperforming the Nifty 50 over the same period.

### Candidate features

- 1-month, 3-month, 6-month and 12-month momentum
- 21-day and 63-day realised volatility
- Recent drawdown and distance from rolling high
- Moving-average trend indicators
- Relative strength versus the Nifty 50
- Rolling beta and benchmark correlation
- Nifty 50 trend, volatility and market regime
- Lagged stock returns
- Volume-based indicators where reliable

### Candidate models

- Historical-mean baseline
- Momentum baseline
- Linear regression
- Ridge and Lasso regression
- Logistic regression
- Random forest
- Gradient boosting

### Trading strategies

- Top-ranked long-only portfolio
- Probability-weighted portfolio
- ML signal with inverse-volatility sizing
- Regime-aware ML portfolio
- Equal-weight and India 10 benchmarks

### Evaluation principles

- Expanding-window walk-forward validation
- No random train-test split
- No future information in model features
- One-day signal execution lag
- Monthly rebalancing
- Transaction costs and turnover
- CAGR, volatility, Sharpe ratio and maximum drawdown
- Hit rate, prediction error and rank information coefficient
- Performance comparison across market regimes

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd


# Project folders
REPO_ROOT = Path("/content/bharat-portfolio-lab")

PROCESSED_DATA_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "ml_trading"
)

OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "ml_trading"
)

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Research assumptions
RANDOM_SEED = 42
TRADING_DAYS_PER_YEAR = 252
FORWARD_HORIZON_DAYS = 21
MINIMUM_TRAINING_DAYS = 756

RISK_FREE_RATE = 0.065
ONE_WAY_TRANSACTION_COST = 0.0015

BENCHMARK_TICKER = "^NSEI"
BENCHMARK_NAME = "Nifty 50"


# India 10 research universe
INDIA_10_TICKERS = [
    "HDFCBANK.NS",
    "TCS.NS",
    "HINDUNILVR.NS",
    "SUNPHARMA.NS",
    "POWERGRID.NS",
    "BHARTIARTL.NS",
    "LT.NS",
    "M&M.NS",
    "BEL.NS",
    "TRENT.NS",
]

np.random.seed(
    RANDOM_SEED
)

print("v0.6 research configuration initialised.")
print("Research universe:", len(INDIA_10_TICKERS), "stocks")
print("Prediction horizon:", FORWARD_HORIZON_DAYS, "trading days")
print("Minimum training history:", MINIMUM_TRAINING_DAYS, "trading days")
print("Benchmark:", BENCHMARK_NAME)
print("Transaction cost:", f"{ONE_WAY_TRANSACTION_COST:.2%}")

v0.6 research configuration initialised.
Research universe: 10 stocks
Prediction horizon: 21 trading days
Minimum training history: 756 trading days
Benchmark: Nifty 50
Transaction cost: 0.15%


## Market Data Collection

Download daily adjusted prices and trading volumes for the India 10 stocks and the Nifty 50 benchmark.

The dataset begins in 2015 to provide enough history for:

- 12-month momentum features
- Three-year minimum training windows
- Expanding-window validation
- Multiple market regimes
- Strict out-of-sample testing

In [3]:
import yfinance as yf

DATA_START_DATE = "2015-01-01"
DATA_END_DATE = "2026-07-31"

ALL_TICKERS = (
    INDIA_10_TICKERS
    + [BENCHMARK_TICKER]
)

raw_market_data = yf.download(
    tickers=ALL_TICKERS,
    start=DATA_START_DATE,
    end=DATA_END_DATE,
    auto_adjust=True,
    progress=False,
    group_by="column",
    threads=True,
)

if raw_market_data.empty:
    raise RuntimeError(
        "No market data were downloaded."
    )

if not isinstance(
    raw_market_data.columns,
    pd.MultiIndex,
):
    raise RuntimeError(
        "Unexpected yFinance column format."
    )


# Extract adjusted close prices and volumes
close_prices = (
    raw_market_data["Close"]
    .reindex(
        columns=ALL_TICKERS
    )
    .sort_index()
)

trading_volume = (
    raw_market_data["Volume"]
    .reindex(
        columns=ALL_TICKERS
    )
    .sort_index()
)


# Remove dates on which every security is missing
close_prices = close_prices.dropna(
    how="all"
)

trading_volume = trading_volume.reindex(
    close_prices.index
)


# Create a data-quality summary
quality_summary = pd.DataFrame(
    {
        "First Valid Date": (
            close_prices.apply(
                lambda series: series.first_valid_index()
            )
        ),
        "Last Valid Date": (
            close_prices.apply(
                lambda series: series.last_valid_index()
            )
        ),
        "Price Observations": (
            close_prices.notna().sum()
        ),
        "Missing Prices": (
            close_prices.isna().sum()
        ),
        "Volume Observations": (
            trading_volume.notna().sum()
        ),
    }
)

quality_summary[
    "Price Coverage"
] = (
    quality_summary[
        "Price Observations"
    ]
    / len(close_prices)
)

quality_summary[
    "Sufficient for ML"
] = (
    quality_summary[
        "Price Observations"
    ]
    >= (
        MINIMUM_TRAINING_DAYS
        + 252
    )
)


print("ML MARKET DATA CHECK")
print("=" * 65)
print(
    "Downloaded period:",
    close_prices.index.min().date(),
    "to",
    close_prices.index.max().date(),
)
print(
    "Trading dates:",
    len(close_prices),
)
print(
    "India 10 stocks:",
    len(INDIA_10_TICKERS),
)
print(
    "Benchmark included:",
    BENCHMARK_TICKER in close_prices.columns,
)
print(
    "All securities sufficient for ML:",
    quality_summary[
        "Sufficient for ML"
    ].all(),
)

display(
    quality_summary
)

ML MARKET DATA CHECK
Downloaded period: 2015-01-01 to 2026-07-30
Trading dates: 2861
India 10 stocks: 10
Benchmark included: True
All securities sufficient for ML: True


,First Valid Date,Last Valid Date,Price Observations,Missing Prices,Volume Observations,Price Coverage,Sufficient for ML
Ticker,,,,,,,
HDFCBANK.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
TCS.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
HINDUNILVR.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
SUNPHARMA.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
POWERGRID.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
BHARTIARTL.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
LT.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
M&M.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
BEL.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True


## Leakage-Safe Feature and Target Dataset

Each observation represents one stock on one trading date.

Features use only information available on or before that date. The prediction targets measure:

- Forward 21-trading-day stock return
- Whether the forward return is positive
- Forward excess return versus the Nifty 50
- Whether the stock outperforms the Nifty 50

The final trading signal will be executed with a one-day lag during backtesting.

In [5]:
# Align all securities to valid Nifty 50 trading dates

original_date_count = len(close_prices)

valid_benchmark_dates = (
    close_prices[
        BENCHMARK_TICKER
    ].notna()
)

removed_dates = (
    close_prices.index[
        ~valid_benchmark_dates
    ]
)

close_prices = (
    close_prices
    .loc[
        valid_benchmark_dates
    ]
    .copy()
)

trading_volume = (
    trading_volume
    .reindex(
        close_prices.index
    )
    .copy()
)

assert close_prices[
    BENCHMARK_TICKER
].notna().all()

assert close_prices[
    INDIA_10_TICKERS
].notna().all().all()

print("TRADING CALENDAR ALIGNMENT")
print("=" * 65)
print("Original dates:", original_date_count)
print("Benchmark-valid dates:", len(close_prices))
print("Dates removed:", len(removed_dates))
print(
    "Aligned period:",
    close_prices.index.min().date(),
    "to",
    close_prices.index.max().date(),
)
print("Benchmark missing values:", int(
    close_prices[
        BENCHMARK_TICKER
    ].isna().sum()
))

TRADING CALENDAR ALIGNMENT
Original dates: 2861
Benchmark-valid dates: 2849
Dates removed: 12
Aligned period: 2015-01-02 to 2026-07-30
Benchmark missing values: 0


In [7]:
# Daily return series
stock_daily_returns = (
    close_prices[
        INDIA_10_TICKERS
    ]
    .pct_change(
        fill_method=None
    )
)

benchmark_price = (
    close_prices[
        BENCHMARK_TICKER
    ]
)

benchmark_daily_return = (
    benchmark_price
    .pct_change(
        fill_method=None
    )
)


# Benchmark-wide features repeated for every stock
benchmark_features = pd.DataFrame(
    index=close_prices.index
)

benchmark_features[
    "benchmark_return_1d"
] = benchmark_daily_return

benchmark_features[
    "benchmark_momentum_21d"
] = (
    benchmark_price
    .pct_change(
        21,
        fill_method=None
    )
)

benchmark_features[
    "benchmark_momentum_63d"
] = (
    benchmark_price
    .pct_change(
        63,
        fill_method=None
    )
)

benchmark_features[
    "benchmark_momentum_126d"
] = (
    benchmark_price
    .pct_change(
        126,
        fill_method=None
    )
)

benchmark_features[
    "benchmark_volatility_21d"
] = (
    benchmark_daily_return
    .rolling(
        21
    )
    .std()
    * np.sqrt(
        TRADING_DAYS_PER_YEAR
    )
)

benchmark_features[
    "benchmark_volatility_63d"
] = (
    benchmark_daily_return
    .rolling(
        63
    )
    .std()
    * np.sqrt(
        TRADING_DAYS_PER_YEAR
    )
)

benchmark_features[
    "benchmark_drawdown_252d"
] = (
    benchmark_price
    / benchmark_price
    .rolling(
        252
    )
    .max()
    - 1
)

benchmark_features[
    "benchmark_ma_gap_200d"
] = (
    benchmark_price
    / benchmark_price
    .rolling(
        200
    )
    .mean()
    - 1
)

benchmark_features[
    "benchmark_bull_regime"
] = (
    benchmark_price
    > benchmark_price
    .rolling(
        200
    )
    .mean()
).astype(float)


# Forward benchmark target
benchmark_forward_return_21d = (
    benchmark_price.shift(
        -FORWARD_HORIZON_DAYS
    )
    / benchmark_price
    - 1
)


# Build one feature frame per stock
stock_feature_frames = {}

for ticker in INDIA_10_TICKERS:

    stock_price = (
        close_prices[
            ticker
        ]
    )

    stock_return = (
        stock_daily_returns[
            ticker
        ]
    )

    stock_volume = (
        trading_volume[
            ticker
        ]
    )

    stock_frame = pd.DataFrame(
        index=close_prices.index
    )

    # Recent returns and momentum
    stock_frame[
        "return_1d"
    ] = stock_return

    stock_frame[
        "return_5d"
    ] = (
        stock_price
        .pct_change(
            5,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_21d"
    ] = (
        stock_price
        .pct_change(
            21,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_63d"
    ] = (
        stock_price
        .pct_change(
            63,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_126d"
    ] = (
        stock_price
        .pct_change(
            126,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_252d"
    ] = (
        stock_price
        .pct_change(
            252,
            fill_method=None
        )
    )

    # Risk and drawdown
    stock_frame[
        "volatility_21d"
    ] = (
        stock_return
        .rolling(
            21
        )
        .std()
        * np.sqrt(
            TRADING_DAYS_PER_YEAR
        )
    )

    stock_frame[
        "volatility_63d"
    ] = (
        stock_return
        .rolling(
            63
        )
        .std()
        * np.sqrt(
            TRADING_DAYS_PER_YEAR
        )
    )

    stock_frame[
        "drawdown_252d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            252
        )
        .max()
        - 1
    )

    # Trend indicators
    stock_frame[
        "ma_gap_21d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            21
        )
        .mean()
        - 1
    )

    stock_frame[
        "ma_gap_63d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            63
        )
        .mean()
        - 1
    )

    stock_frame[
        "ma_gap_200d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            200
        )
        .mean()
        - 1
    )

    # Relative strength versus Nifty 50
    stock_frame[
        "relative_strength_63d"
    ] = (
        stock_frame[
            "momentum_63d"
        ]
        - benchmark_features[
            "benchmark_momentum_63d"
        ]
    )

    stock_frame[
        "relative_strength_126d"
    ] = (
        stock_frame[
            "momentum_126d"
        ]
        - benchmark_features[
            "benchmark_momentum_126d"
        ]
    )

    # Rolling market sensitivity
    rolling_covariance = (
        stock_return
        .rolling(
            63
        )
        .cov(
            benchmark_daily_return
        )
    )

    rolling_benchmark_variance = (
        benchmark_daily_return
        .rolling(
            63
        )
        .var()
    )

    stock_frame[
        "beta_63d"
    ] = (
        rolling_covariance
        / rolling_benchmark_variance
    )

    stock_frame[
        "benchmark_correlation_63d"
    ] = (
        stock_return
        .rolling(
            63
        )
        .corr(
            benchmark_daily_return
        )
    )

    # Volume features
    stock_frame[
        "volume_ratio_21d"
    ] = (
        stock_volume
        / stock_volume
        .rolling(
            21
        )
        .mean()
        - 1
    )

    stock_frame[
        "volume_trend_21_63d"
    ] = (
        stock_volume
        .rolling(
            21
        )
        .mean()
        / stock_volume
        .rolling(
            63
        )
        .mean()
        - 1
    )

    # Add market-wide features
    stock_frame = stock_frame.join(
        benchmark_features
    )

    # Forward targets
    forward_return = (
        stock_price.shift(
            -FORWARD_HORIZON_DAYS
        )
        / stock_price
        - 1
    )

    forward_excess_return = (
        forward_return
        - benchmark_forward_return_21d
    )

    stock_frame[
        "forward_return_21d"
    ] = forward_return

    stock_frame[
        "forward_excess_return_21d"
    ] = forward_excess_return

    stock_frame[
        "positive_return_target"
    ] = (
        forward_return
        .gt(0)
        .where(
            forward_return.notna()
        )
        .astype(float)
    )

    stock_frame[
        "outperform_target"
    ] = (
        forward_excess_return
        .gt(0)
        .where(
            forward_excess_return.notna()
        )
        .astype(float)
    )

    stock_feature_frames[
        ticker
    ] = stock_frame


# Convert into one stock-date panel
ml_panel_raw = (
    pd.concat(
        stock_feature_frames,
        names=[
            "Ticker",
            "Date",
        ],
    )
    .reset_index()
)


feature_columns = [
    "return_1d",
    "return_5d",
    "momentum_21d",
    "momentum_63d",
    "momentum_126d",
    "momentum_252d",
    "volatility_21d",
    "volatility_63d",
    "drawdown_252d",
    "ma_gap_21d",
    "ma_gap_63d",
    "ma_gap_200d",
    "relative_strength_63d",
    "relative_strength_126d",
    "beta_63d",
    "benchmark_correlation_63d",
    "volume_ratio_21d",
    "volume_trend_21_63d",
    "benchmark_return_1d",
    "benchmark_momentum_21d",
    "benchmark_momentum_63d",
    "benchmark_momentum_126d",
    "benchmark_volatility_21d",
    "benchmark_volatility_63d",
    "benchmark_drawdown_252d",
    "benchmark_ma_gap_200d",
    "benchmark_bull_regime",
]

target_columns = [
    "forward_return_21d",
    "forward_excess_return_21d",
    "positive_return_target",
    "outperform_target",
]


# Retain only observations with complete features and targets
ml_panel = (
    ml_panel_raw
    .dropna(
        subset=(
            feature_columns
            + target_columns
        )
    )
    .sort_values(
        [
            "Date",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# Store classification targets as integers
ml_panel[
    "positive_return_target"
] = (
    ml_panel[
        "positive_return_target"
    ]
    .astype(int)
)

ml_panel[
    "outperform_target"
] = (
    ml_panel[
        "outperform_target"
    ]
    .astype(int)
)


# Save a reproducible research dataset
feature_dataset_path = (
    PROCESSED_DATA_DIR
    / "india10_ml_feature_panel.csv"
)

ml_panel.to_csv(
    feature_dataset_path,
    index=False,
)


# Validation
assert ml_panel[
    feature_columns
].notna().all().all()

assert ml_panel[
    target_columns
].notna().all().all()

assert set(
    ml_panel[
        "Ticker"
    ].unique()
) == set(
    INDIA_10_TICKERS
)

assert (
    ml_panel[
        "Date"
    ].max()
    <= close_prices.index[
        -FORWARD_HORIZON_DAYS - 1
    ]
)


print("ML FEATURE DATASET CHECK")
print("=" * 65)
print(
    "Panel observations:",
    f"{len(ml_panel):,}",
)
print(
    "Stocks:",
    ml_panel[
        "Ticker"
    ].nunique(),
)
print(
    "Features:",
    len(feature_columns),
)
print(
    "Dataset period:",
    ml_panel[
        "Date"
    ].min().date(),
    "to",
    ml_panel[
        "Date"
    ].max().date(),
)
print(
    "Positive-return rate:",
    f"{ml_panel['positive_return_target'].mean():.2%}",
)
print(
    "Nifty outperformance rate:",
    f"{ml_panel['outperform_target'].mean():.2%}",
)
print(
    "Missing feature values:",
    int(
        ml_panel[
            feature_columns
        ]
        .isna()
        .sum()
        .sum()
    ),
)
print(
    "Saved dataset:",
    feature_dataset_path,
)

display(
    ml_panel.head()
)

ML FEATURE DATASET CHECK
Panel observations: 25,760
Stocks: 10
Features: 27
Dataset period: 2016-01-14 to 2026-07-01
Positive-return rate: 58.69%
Nifty outperformance rate: 52.03%
Missing feature values: 0
Saved dataset: /content/bharat-portfolio-lab/data/processed/ml_trading/india10_ml_feature_panel.csv


,Ticker,Date,return_1d,return_5d,momentum_21d,momentum_63d,momentum_126d,momentum_252d,volatility_21d,volatility_63d,...,benchmark_momentum_126d,benchmark_volatility_21d,benchmark_volatility_63d,benchmark_drawdown_252d,benchmark_ma_gap_200d,benchmark_bull_regime,forward_return_21d,forward_excess_return_21d,positive_return_target,outperform_target
0,BEL.NS,2016-01-14,-0.023923,0.010744,0.102567,0.082936,0.151348,0.351668,0.287186,0.259007,...,-0.095065,0.148547,0.121527,-0.162229,-0.075498,0.0,-0.110123,-0.060519,0,0
1,BHARTIARTL.NS,2016-01-14,0.000486,-0.042190,0.008492,-0.111766,-0.284446,-0.149490,0.283601,0.250530,...,-0.095065,0.148547,0.121527,-0.162229,-0.075498,0.0,0.032875,0.082478,1,1
2,HDFCBANK.NS,2016-01-14,-0.009810,-0.006107,-0.005024,-0.033513,-0.022625,0.095659,0.122117,0.128907,...,-0.095065,0.148547,0.121527,-0.162229,-0.075498,0.0,-0.072541,-0.022938,0,0
3,HINDUNILVR.NS,2016-01-14,-0.006604,0.008534,-0.014005,0.019752,-0.094833,0.115090,0.180885,0.181416,...,-0.095065,0.148547,0.121527,-0.162229,-0.075498,0.0,-0.027259,0.022344,0,1
4,LT.NS,2016-01-14,-0.019105,-0.059475,-0.109659,-0.266264,-0.380025,-0.253073,0.167221,0.192795,...,-0.095065,0.148547,0.121527,-0.162229,-0.075498,0.0,0.012999,0.062603,1,1


In [8]:
expected_last_target_date = (
    close_prices.index[
        -FORWARD_HORIZON_DAYS - 1
    ]
)

actual_last_panel_date = (
    ml_panel[
        "Date"
    ].max()
)

calendar_gap_days = (
    expected_last_target_date
    - actual_last_panel_date
).days

assert calendar_gap_days >= 0
assert calendar_gap_days <= 7

print("CORRECTED FEATURE PANEL")
print("=" * 65)
print(
    "Dataset period:",
    ml_panel["Date"].min().date(),
    "to",
    actual_last_panel_date.date(),
)
print(
    "Expected final eligible date:",
    expected_last_target_date.date(),
)
print(
    "Panel observations:",
    f"{len(ml_panel):,}",
)
print(
    "Features:",
    len(feature_columns),
)
print(
    "Missing feature values:",
    int(
        ml_panel[
            feature_columns
        ]
        .isna()
        .sum()
        .sum()
    ),
)
print("Calendar correction:", "PASSED")

CORRECTED FEATURE PANEL
Dataset period: 2016-01-14 to 2026-07-01
Expected final eligible date: 2026-07-01
Panel observations: 25,760
Features: 27
Missing feature values: 0
Calendar correction: PASSED


## Feature Quality and Target Analysis

Before model training, evaluate:

- Missing, infinite and constant feature values
- Duplicate stock-date observations
- Forward-return and classification-target distributions
- Target behaviour across calendar years
- Cross-sectional predictive relationship between each feature and future excess returns

The daily rank Information Coefficient measures whether a feature correctly ranks stocks from weaker to stronger future Nifty 50 outperformance. It is an exploratory statistic, not evidence of a profitable strategy.

In [10]:
# =========================================================
# CORRECT CROSS-SECTIONAL IC VALIDATION
# Separate stock-ranking features from market-wide features
# =========================================================

# Number of valid daily cross-sectional IC observations
daily_ic_valid_counts = (
    daily_feature_rank_ic
    .notna()
    .sum()
)

cross_sectional_features = (
    daily_ic_valid_counts[
        daily_ic_valid_counts > 0
    ]
    .index
    .tolist()
)

non_cross_sectional_features = (
    daily_ic_valid_counts[
        daily_ic_valid_counts == 0
    ]
    .index
    .tolist()
)

expected_market_wide_features = [
    feature
    for feature in feature_columns
    if feature.startswith(
        "benchmark_"
    )
]


# Market-wide features should be the only variables
# without a valid cross-sectional rank IC.
unexpected_non_cross_sectional_features = sorted(
    set(
        non_cross_sectional_features
    )
    - set(
        expected_market_wide_features
    )
)

if unexpected_non_cross_sectional_features:
    raise RuntimeError(
        "Unexpected features have no valid daily rank IC:\n"
        + "\n".join(
            unexpected_non_cross_sectional_features
        )
    )


# Label each feature according to its research role
feature_quality_summary[
    "Feature Role"
] = [
    (
        "Market-wide regime feature"
        if feature in expected_market_wide_features
        else "Cross-sectional stock feature"
    )
    for feature in feature_quality_summary.index
]

feature_quality_summary[
    "Valid Daily IC Observations"
] = (
    daily_ic_valid_counts
    .reindex(
        feature_quality_summary.index
    )
    .fillna(
        0
    )
    .astype(
        int
    )
)


# Sort stock-ranking features by predictive rank strength.
# Market-wide features remain in the table but appear afterward.
feature_quality_summary[
    "IC Available"
] = (
    feature_quality_summary[
        "Valid Daily IC Observations"
    ]
    > 0
)

feature_quality_summary = (
    feature_quality_summary
    .sort_values(
        by=[
            "IC Available",
            "Absolute Mean Daily Rank IC",
        ],
        ascending=[
            False,
            False,
        ],
        na_position="last",
    )
)


# Re-save the corrected feature-quality output
feature_quality_summary.to_csv(
    feature_quality_path
)


# Correct validation
assert duplicate_stock_dates == 0
assert infinite_feature_values == 0
assert missing_feature_values == 0
assert not constant_features
assert not feature_target_overlap

assert len(
    cross_sectional_features
) > 0

assert not (
    unexpected_non_cross_sectional_features
)

assert set(
    non_cross_sectional_features
).issubset(
    set(
        expected_market_wide_features
    )
)


top_rank_features = (
    feature_quality_summary.loc[
        feature_quality_summary[
            "IC Available"
        ],
        [
            "Feature Role",
            "Mean Daily Rank IC",
            "Positive Daily IC Rate",
            "Pooled Spearman IC",
            "Rank IC Information Ratio",
            "Valid Daily IC Observations",
        ],
    ]
    .head(
        10
    )
)


print("CORRECTED FEATURE QUALITY CHECK")
print("=" * 70)
print(
    "Total features:",
    len(
        feature_columns
    ),
)
print(
    "Cross-sectional stock features:",
    len(
        cross_sectional_features
    ),
)
print(
    "Market-wide features:",
    len(
        non_cross_sectional_features
    ),
)
print(
    "Market-wide feature names:",
    ", ".join(
        non_cross_sectional_features
    ),
)
print(
    "Unexpected features without rank IC:",
    len(
        unexpected_non_cross_sectional_features
    ),
)
print(
    "Duplicate stock-date rows:",
    duplicate_stock_dates,
)
print(
    "Missing feature values:",
    missing_feature_values,
)
print(
    "Infinite feature values:",
    infinite_feature_values,
)
print(
    "Feature-quality validation:",
    "PASSED",
)

print("\nTOP CROSS-SECTIONAL FEATURES")
display(
    top_rank_features
)

print("\nTARGET SUMMARY BY YEAR")
display(
    target_summary_by_year
)

CORRECTED FEATURE QUALITY CHECK
Total features: 27
Cross-sectional stock features: 18
Market-wide features: 9
Market-wide feature names: benchmark_return_1d, benchmark_momentum_21d, benchmark_momentum_63d, benchmark_momentum_126d, benchmark_volatility_21d, benchmark_volatility_63d, benchmark_drawdown_252d, benchmark_ma_gap_200d, benchmark_bull_regime
Unexpected features without rank IC: 0
Duplicate stock-date rows: 0
Missing feature values: 0
Infinite feature values: 0
Feature-quality validation: PASSED

TOP CROSS-SECTIONAL FEATURES


,Feature Role,Mean Daily Rank IC,Positive Daily IC Rate,Pooled Spearman IC,Rank IC Information Ratio,Valid Daily IC Observations
momentum_21d,Cross-sectional stock feature,-0.056686,0.427407,-0.063482,-0.155586,2576
ma_gap_21d,Cross-sectional stock feature,-0.048414,0.440217,-0.057060,-0.135389,2576
momentum_252d,Cross-sectional stock feature,0.046217,0.533773,0.057143,0.114503,2576
ma_gap_63d,Cross-sectional stock feature,-0.039596,0.473602,-0.041242,-0.105522,2576
return_5d,Cross-sectional stock feature,-0.038392,0.459627,-0.040758,-0.111295,2576
volatility_63d,Cross-sectional stock feature,0.027692,0.528727,-0.023106,0.070869,2576
drawdown_252d,Cross-sectional stock feature,-0.020416,0.485637,0.009491,-0.052889,2576
volume_ratio_21d,Cross-sectional stock feature,-0.018474,0.490683,-0.011486,-0.054417,2575
return_1d,Cross-sectional stock feature,-0.018182,0.476320,-0.019014,-0.052734,2575
momentum_63d,Cross-sectional stock feature,-0.014130,0.486413,-0.014523,-0.037171,2576



TARGET SUMMARY BY YEAR


,Observations,Trading_Dates,Mean_Forward_Return,Median_Forward_Return,Mean_Forward_Excess_Return,Positive_Return_Rate,Nifty_Outperformance_Rate
Year,,,,,,,
2016,2360,236,0.013168,0.012921,0.000882,0.583051,0.486441
2017,2480,248,0.026994,0.025243,0.005202,0.675000,0.547984
2018,2450,245,-0.002761,0.000579,-0.003702,0.502449,0.503673
2019,2410,241,0.015713,0.010745,0.004464,0.564315,0.523237
2020,2500,250,0.026631,0.024852,0.007840,0.605600,0.462400
2021,2480,248,0.026382,0.020346,0.007128,0.620565,0.493145
2022,2480,248,0.015402,0.010850,0.013211,0.550806,0.560887
2023,2450,245,0.032624,0.027344,0.016027,0.683265,0.604082
2024,2460,246,0.027401,0.019790,0.020307,0.613821,0.578862


## Leakage-Safe Baseline Signals

Before training machine-learning models, evaluate simple investment signals:

1. Expanding historical mean excess return by stock
2. Six-month momentum
3. Twelve-month momentum
4. Blended six- and twelve-month momentum
5. One-month reversal
6. Equal-weight India 10 benchmark

Signals are evaluated on monthly rebalance dates. Historical target information is lagged by 21 trading days so that only outcomes already observable at the signal date are used.

These results measure ranking ability and forward returns. They are not yet a complete transaction-cost-adjusted portfolio backtest.

In [11]:
# =========================================================
# LEAKAGE-SAFE BASELINE SIGNAL EVALUATION
# =========================================================

baseline_panel = ml_panel.copy()

baseline_panel[
    "Date"
] = pd.to_datetime(
    baseline_panel[
        "Date"
    ]
)

market_calendar = pd.DatetimeIndex(
    close_prices.index
).sort_values()

eligible_dates = pd.DatetimeIndex(
    sorted(
        baseline_panel[
            "Date"
        ].unique()
    )
)


# ---------------------------------------------------------
# 1. Select the final eligible observation in each month
# ---------------------------------------------------------

monthly_rebalance_dates = (
    pd.Series(
        eligible_dates,
        index=eligible_dates,
    )
    .groupby(
        eligible_dates.to_period(
            "M"
        )
    )
    .max()
    .tolist()
)


# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------

def cross_sectional_zscore(
    values: pd.Series,
) -> pd.Series:

    standard_deviation = values.std(
        ddof=0
    )

    if (
        pd.isna(
            standard_deviation
        )
        or standard_deviation == 0
    ):
        return pd.Series(
            0.0,
            index=values.index,
        )

    return (
        values
        - values.mean()
    ) / standard_deviation


def calculate_rank_ic(
    frame: pd.DataFrame,
    score_column: str,
) -> float:

    if (
        frame[
            score_column
        ].nunique()
        <= 1
    ):
        return np.nan

    return frame[
        [
            score_column,
            "forward_excess_return_21d",
        ]
    ].corr(
        method="spearman"
    ).iloc[
        0,
        1,
    ]


# ---------------------------------------------------------
# 3. Generate monthly out-of-sample baseline predictions
# ---------------------------------------------------------

baseline_prediction_records = []

for rebalance_date in monthly_rebalance_dates:

    calendar_position = (
        market_calendar.get_indexer(
            [
                rebalance_date
            ]
        )[0]
    )

    if calendar_position == -1:
        continue

    # Require at least three years of prior market history.
    if calendar_position < (
        MINIMUM_TRAINING_DAYS
        + FORWARD_HORIZON_DAYS
    ):
        continue

    target_availability_position = (
        calendar_position
        - FORWARD_HORIZON_DAYS
    )

    target_availability_date = (
        market_calendar[
            target_availability_position
        ]
    )

    training_panel = (
        baseline_panel.loc[
            baseline_panel[
                "Date"
            ]
            <= target_availability_date
        ]
        .copy()
    )

    current_cross_section = (
        baseline_panel.loc[
            baseline_panel[
                "Date"
            ]
            == rebalance_date
        ]
        .copy()
    )

    if (
        len(
            current_cross_section
        )
        != len(
            INDIA_10_TICKERS
        )
    ):
        continue

    if training_panel.empty:
        continue

    historical_stock_means = (
        training_panel
        .groupby(
            "Ticker"
        )[
            "forward_excess_return_21d"
        ]
        .mean()
    )

    global_historical_mean = (
        training_panel[
            "forward_excess_return_21d"
        ]
        .mean()
    )

    current_cross_section[
        "historical_mean_score"
    ] = (
        current_cross_section[
            "Ticker"
        ]
        .map(
            historical_stock_means
        )
        .fillna(
            global_historical_mean
        )
    )

    current_cross_section[
        "momentum_126d_score"
    ] = (
        current_cross_section[
            "momentum_126d"
        ]
    )

    current_cross_section[
        "momentum_252d_score"
    ] = (
        current_cross_section[
            "momentum_252d"
        ]
    )

    current_cross_section[
        "blended_momentum_score"
    ] = (
        0.50
        * cross_sectional_zscore(
            current_cross_section[
                "momentum_126d"
            ]
        )
        + 0.50
        * cross_sectional_zscore(
            current_cross_section[
                "momentum_252d"
            ]
        )
    )

    current_cross_section[
        "short_term_reversal_score"
    ] = (
        -current_cross_section[
            "momentum_21d"
        ]
    )

    current_cross_section[
        "Training Cutoff"
    ] = target_availability_date

    baseline_prediction_records.append(
        current_cross_section
    )


if not baseline_prediction_records:
    raise RuntimeError(
        "No valid monthly baseline predictions were generated."
    )

baseline_predictions = (
    pd.concat(
        baseline_prediction_records,
        ignore_index=True,
    )
    .sort_values(
        [
            "Date",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 4. Evaluate each baseline signal
# ---------------------------------------------------------

baseline_score_columns = {
    "Historical Mean":
        "historical_mean_score",

    "6-Month Momentum":
        "momentum_126d_score",

    "12-Month Momentum":
        "momentum_252d_score",

    "Blended Momentum":
        "blended_momentum_score",

    "1-Month Reversal":
        "short_term_reversal_score",
}

TOP_STOCK_COUNT = 3

baseline_evaluation_records = []

for baseline_name, score_column in (
    baseline_score_columns.items()
):

    daily_rank_ic_values = []
    selected_return_values = []
    selected_excess_values = []
    selected_positive_rates = []
    selected_outperformance_rates = []

    for (
        rebalance_date,
        rebalance_cross_section,
    ) in baseline_predictions.groupby(
        "Date"
    ):

        rank_ic = calculate_rank_ic(
            frame=rebalance_cross_section,
            score_column=score_column,
        )

        daily_rank_ic_values.append(
            rank_ic
        )

        selected_stocks = (
            rebalance_cross_section
            .nlargest(
                TOP_STOCK_COUNT,
                score_column,
            )
        )

        selected_return_values.append(
            selected_stocks[
                "forward_return_21d"
            ].mean()
        )

        selected_excess_values.append(
            selected_stocks[
                "forward_excess_return_21d"
            ].mean()
        )

        selected_positive_rates.append(
            selected_stocks[
                "positive_return_target"
            ].mean()
        )

        selected_outperformance_rates.append(
            selected_stocks[
                "outperform_target"
            ].mean()
        )

    valid_rank_ic = pd.Series(
        daily_rank_ic_values,
        dtype=float,
    ).dropna()

    baseline_evaluation_records.append(
        {
            "Baseline":
                baseline_name,

            "Rebalances":
                baseline_predictions[
                    "Date"
                ].nunique(),

            "Selected Stocks":
                TOP_STOCK_COUNT,

            "Mean Rank IC":
                valid_rank_ic.mean(),

            "Median Rank IC":
                valid_rank_ic.median(),

            "Positive Rank IC Rate":
                valid_rank_ic.gt(
                    0
                ).mean(),

            "Mean Selected Forward Return":
                np.mean(
                    selected_return_values
                ),

            "Mean Selected Excess Return":
                np.mean(
                    selected_excess_values
                ),

            "Selected Positive Return Rate":
                np.mean(
                    selected_positive_rates
                ),

            "Selected Outperformance Rate":
                np.mean(
                    selected_outperformance_rates
                ),
        }
    )


# ---------------------------------------------------------
# 5. Add the equal-weight India 10 benchmark
# ---------------------------------------------------------

equal_weight_monthly_results = (
    baseline_predictions
    .groupby(
        "Date"
    )
    .agg(
        Mean_Forward_Return=(
            "forward_return_21d",
            "mean",
        ),

        Mean_Forward_Excess_Return=(
            "forward_excess_return_21d",
            "mean",
        ),

        Positive_Return_Rate=(
            "positive_return_target",
            "mean",
        ),

        Outperformance_Rate=(
            "outperform_target",
            "mean",
        ),
    )
)

baseline_evaluation_records.append(
    {
        "Baseline":
            "Equal-Weight India 10",

        "Rebalances":
            len(
                equal_weight_monthly_results
            ),

        "Selected Stocks":
            len(
                INDIA_10_TICKERS
            ),

        "Mean Rank IC":
            np.nan,

        "Median Rank IC":
            np.nan,

        "Positive Rank IC Rate":
            np.nan,

        "Mean Selected Forward Return":
            equal_weight_monthly_results[
                "Mean_Forward_Return"
            ].mean(),

        "Mean Selected Excess Return":
            equal_weight_monthly_results[
                "Mean_Forward_Excess_Return"
            ].mean(),

        "Selected Positive Return Rate":
            equal_weight_monthly_results[
                "Positive_Return_Rate"
            ].mean(),

        "Selected Outperformance Rate":
            equal_weight_monthly_results[
                "Outperformance_Rate"
            ].mean(),
    }
)


baseline_evaluation = (
    pd.DataFrame(
        baseline_evaluation_records
    )
    .set_index(
        "Baseline"
    )
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 6. Save baseline research datasets
# ---------------------------------------------------------

baseline_predictions_path = (
    PROCESSED_DATA_DIR
    / "monthly_baseline_predictions.csv"
)

baseline_evaluation_path = (
    PROCESSED_DATA_DIR
    / "baseline_signal_evaluation.csv"
)

baseline_predictions.to_csv(
    baseline_predictions_path,
    index=False,
)

baseline_evaluation.to_csv(
    baseline_evaluation_path
)


# ---------------------------------------------------------
# 7. Leakage and quality validation
# ---------------------------------------------------------

assert (
    baseline_predictions[
        "Training Cutoff"
    ]
    < baseline_predictions[
        "Date"
    ]
).all()

assert (
    baseline_predictions[
        "Date"
    ].nunique()
    >= 60
)

assert (
    baseline_predictions
    .groupby(
        "Date"
    )[
        "Ticker"
    ]
    .nunique()
    .eq(
        len(
            INDIA_10_TICKERS
        )
    )
    .all()
)

assert (
    baseline_predictions[
        list(
            baseline_score_columns.values()
        )
    ]
    .notna()
    .all()
    .all()
)

assert len(
    baseline_evaluation
) == 6


# ---------------------------------------------------------
# 8. Results
# ---------------------------------------------------------

display_evaluation = (
    baseline_evaluation.copy()
)

percentage_columns = [
    "Mean Rank IC",
    "Median Rank IC",
    "Positive Rank IC Rate",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in percentage_columns:

    display_evaluation[
        column
    ] = (
        display_evaluation[
            column
        ]
        * 100
    )


print("BASELINE SIGNAL EVALUATION")
print("=" * 70)
print(
    "Evaluation period:",
    baseline_predictions[
        "Date"
    ].min().date(),
    "to",
    baseline_predictions[
        "Date"
    ].max().date(),
)
print(
    "Monthly rebalances:",
    baseline_predictions[
        "Date"
    ].nunique(),
)
print(
    "Stocks per rebalance:",
    baseline_predictions
    .groupby(
        "Date"
    )[
        "Ticker"
    ]
    .nunique()
    .median(),
)
print(
    "Historical target lag:",
    FORWARD_HORIZON_DAYS,
    "trading days",
)
print(
    "Top stocks selected:",
    TOP_STOCK_COUNT,
)
print(
    "Leakage-safe training cutoffs:",
    "PASSED",
)
print(
    "Baseline evaluation:",
    "PASSED",
)

display(
    display_evaluation.round(
        2
    )
)

BASELINE SIGNAL EVALUATION
Evaluation period: 2018-03-28 to 2026-07-01
Monthly rebalances: 101
Stocks per rebalance: 10.0
Historical target lag: 21 trading days
Top stocks selected: 3
Leakage-safe training cutoffs: PASSED
Baseline evaluation: PASSED


,Rebalances,Selected Stocks,Mean Rank IC,Median Rank IC,Positive Rank IC Rate,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Baseline,,,,,,,,,
1-Month Reversal,101,3,6.89,6.67,59.41,2.80,1.79,59.41,55.12
12-Month Momentum,101,3,3.04,5.45,58.42,2.23,1.21,59.74,51.16
Historical Mean,101,3,3.41,6.67,56.44,2.18,1.17,58.42,53.80
6-Month Momentum,101,3,2.66,5.45,59.41,2.05,1.03,57.43,53.14
Blended Momentum,101,3,3.77,3.03,55.45,1.94,0.92,58.09,52.15
Equal-Weight India 10,101,10,NaN,NaN,NaN,1.80,0.79,58.12,51.58


## Walk-Forward Linear Regression Models

The first machine-learning models predict each stock’s forward 21-trading-day excess return over the Nifty 50.

Models:

- Ordinary Least Squares Linear Regression
- Ridge Regression

Both models use:

- An expanding historical training window
- Only targets observable before the prediction date
- Training-only feature standardisation
- Monthly out-of-sample predictions
- The same evaluation dates as the baseline strategies

These are deliberately simple, untuned models. More complex tree-based models will only be considered after establishing whether linear relationships contain useful predictive information.

In [12]:
# =========================================================
# WALK-FORWARD LINEAR REGRESSION MODELS
# =========================================================

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------
# 1. Prepare the modelling panel and rebalance schedule
# ---------------------------------------------------------

regression_panel = ml_panel.copy()

regression_panel[
    "Date"
] = pd.to_datetime(
    regression_panel[
        "Date"
    ]
)

rebalance_schedule = (
    baseline_predictions
    .groupby(
        "Date"
    )[
        "Training Cutoff"
    ]
    .first()
    .sort_index()
)

rebalance_schedule.index = (
    pd.to_datetime(
        rebalance_schedule.index
    )
)

rebalance_schedule = pd.to_datetime(
    rebalance_schedule
)


# ---------------------------------------------------------
# 2. Define simple regression models
# ---------------------------------------------------------

regression_models = {
    "Linear Regression":
        Pipeline(
            steps=[
                (
                    "standard_scaler",
                    StandardScaler(),
                ),
                (
                    "regression_model",
                    LinearRegression(),
                ),
            ]
        ),

    "Ridge Regression":
        Pipeline(
            steps=[
                (
                    "standard_scaler",
                    StandardScaler(),
                ),
                (
                    "regression_model",
                    Ridge(
                        alpha=10.0
                    ),
                ),
            ]
        ),
}


# ---------------------------------------------------------
# 3. Generate monthly walk-forward predictions
# ---------------------------------------------------------

regression_prediction_records = []

for (
    rebalance_date,
    training_cutoff,
) in rebalance_schedule.items():

    training_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            <= training_cutoff
        ]
        .copy()
    )

    prediction_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            == rebalance_date
        ]
        .copy()
    )

    if (
        len(
            prediction_sample
        )
        != len(
            INDIA_10_TICKERS
        )
    ):
        continue

    if (
        len(
            training_sample
        )
        < (
            len(
                INDIA_10_TICKERS
            )
            * 252
        )
    ):
        continue

    X_train = (
        training_sample[
            feature_columns
        ]
        .astype(
            float
        )
    )

    y_train = (
        training_sample[
            "forward_excess_return_21d"
        ]
        .astype(
            float
        )
    )

    X_predict = (
        prediction_sample[
            feature_columns
        ]
        .astype(
            float
        )
    )

    for (
        model_name,
        model_pipeline,
    ) in regression_models.items():

        model_pipeline.fit(
            X_train,
            y_train,
        )

        predicted_excess_return = (
            model_pipeline.predict(
                X_predict
            )
        )

        model_predictions = (
            prediction_sample[
                [
                    "Date",
                    "Ticker",
                    "forward_return_21d",
                    "forward_excess_return_21d",
                    "positive_return_target",
                    "outperform_target",
                ]
            ]
            .copy()
        )

        model_predictions[
            "Model"
        ] = model_name

        model_predictions[
            "Predicted Excess Return"
        ] = predicted_excess_return

        model_predictions[
            "Training Cutoff"
        ] = training_cutoff

        model_predictions[
            "Training Observations"
        ] = len(
            training_sample
        )

        regression_prediction_records.append(
            model_predictions
        )


if not regression_prediction_records:
    raise RuntimeError(
        "No walk-forward regression predictions "
        "were generated."
    )

regression_predictions = (
    pd.concat(
        regression_prediction_records,
        ignore_index=True,
    )
    .sort_values(
        [
            "Date",
            "Model",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 4. Evaluate each model
# ---------------------------------------------------------

TOP_STOCK_COUNT = 3

regression_evaluation_records = []

for (
    model_name,
    model_predictions,
) in regression_predictions.groupby(
    "Model"
):

    actual_values = (
        model_predictions[
            "forward_excess_return_21d"
        ]
    )

    predicted_values = (
        model_predictions[
            "Predicted Excess Return"
        ]
    )

    monthly_rank_ic_values = []
    selected_forward_returns = []
    selected_excess_returns = []
    selected_positive_rates = []
    selected_outperformance_rates = []

    for (
        prediction_date,
        prediction_cross_section,
    ) in model_predictions.groupby(
        "Date"
    ):

        if (
            prediction_cross_section[
                "Predicted Excess Return"
            ].nunique()
            > 1
        ):

            rank_ic = (
                prediction_cross_section[
                    [
                        "Predicted Excess Return",
                        "forward_excess_return_21d",
                    ]
                ]
                .corr(
                    method="spearman"
                )
                .iloc[
                    0,
                    1,
                ]
            )

        else:

            rank_ic = np.nan

        monthly_rank_ic_values.append(
            rank_ic
        )

        selected_stocks = (
            prediction_cross_section
            .nlargest(
                TOP_STOCK_COUNT,
                "Predicted Excess Return",
            )
        )

        selected_forward_returns.append(
            selected_stocks[
                "forward_return_21d"
            ].mean()
        )

        selected_excess_returns.append(
            selected_stocks[
                "forward_excess_return_21d"
            ].mean()
        )

        selected_positive_rates.append(
            selected_stocks[
                "positive_return_target"
            ].mean()
        )

        selected_outperformance_rates.append(
            selected_stocks[
                "outperform_target"
            ].mean()
        )

    valid_rank_ic = (
        pd.Series(
            monthly_rank_ic_values,
            dtype=float,
        )
        .dropna()
    )

    directional_accuracy = (
        (
            predicted_values
            > 0
        )
        == (
            actual_values
            > 0
        )
    ).mean()

    regression_evaluation_records.append(
        {
            "Model":
                model_name,

            "Rebalances":
                model_predictions[
                    "Date"
                ].nunique(),

            "Predictions":
                len(
                    model_predictions
                ),

            "Mean Absolute Error":
                mean_absolute_error(
                    actual_values,
                    predicted_values,
                ),

            "Root Mean Squared Error":
                np.sqrt(
                    mean_squared_error(
                        actual_values,
                        predicted_values,
                    )
                ),

            "Pooled R-Squared":
                r2_score(
                    actual_values,
                    predicted_values,
                ),

            "Mean Rank IC":
                valid_rank_ic.mean(),

            "Median Rank IC":
                valid_rank_ic.median(),

            "Positive Rank IC Rate":
                valid_rank_ic.gt(
                    0
                ).mean(),

            "Directional Accuracy":
                directional_accuracy,

            "Mean Selected Forward Return":
                np.mean(
                    selected_forward_returns
                ),

            "Mean Selected Excess Return":
                np.mean(
                    selected_excess_returns
                ),

            "Selected Positive Return Rate":
                np.mean(
                    selected_positive_rates
                ),

            "Selected Outperformance Rate":
                np.mean(
                    selected_outperformance_rates
                ),
        }
    )


regression_evaluation = (
    pd.DataFrame(
        regression_evaluation_records
    )
    .set_index(
        "Model"
    )
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 5. Create comparison with the strongest simple baselines
# ---------------------------------------------------------

regression_baseline_comparison = (
    regression_evaluation[
        [
            "Rebalances",
            "Mean Rank IC",
            "Positive Rank IC Rate",
            "Mean Selected Forward Return",
            "Mean Selected Excess Return",
            "Selected Positive Return Rate",
            "Selected Outperformance Rate",
        ]
    ]
    .copy()
)

for baseline_name in [
    "1-Month Reversal",
    "12-Month Momentum",
    "Equal-Weight India 10",
]:

    baseline_row = (
        baseline_evaluation.loc[
            baseline_name
        ]
    )

    regression_baseline_comparison.loc[
        baseline_name
    ] = {
        "Rebalances":
            baseline_row[
                "Rebalances"
            ],

        "Mean Rank IC":
            baseline_row[
                "Mean Rank IC"
            ],

        "Positive Rank IC Rate":
            baseline_row[
                "Positive Rank IC Rate"
            ],

        "Mean Selected Forward Return":
            baseline_row[
                "Mean Selected Forward Return"
            ],

        "Mean Selected Excess Return":
            baseline_row[
                "Mean Selected Excess Return"
            ],

        "Selected Positive Return Rate":
            baseline_row[
                "Selected Positive Return Rate"
            ],

        "Selected Outperformance Rate":
            baseline_row[
                "Selected Outperformance Rate"
            ],
    }


regression_baseline_comparison = (
    regression_baseline_comparison
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 6. Save research outputs
# ---------------------------------------------------------

regression_predictions_path = (
    PROCESSED_DATA_DIR
    / "walk_forward_regression_predictions.csv"
)

regression_evaluation_path = (
    PROCESSED_DATA_DIR
    / "walk_forward_regression_evaluation.csv"
)

regression_comparison_path = (
    PROCESSED_DATA_DIR
    / "regression_baseline_comparison.csv"
)

regression_predictions.to_csv(
    regression_predictions_path,
    index=False,
)

regression_evaluation.to_csv(
    regression_evaluation_path
)

regression_baseline_comparison.to_csv(
    regression_comparison_path
)


# ---------------------------------------------------------
# 7. Leakage and data-quality validation
# ---------------------------------------------------------

assert (
    regression_predictions[
        "Training Cutoff"
    ]
    < regression_predictions[
        "Date"
    ]
).all()

assert (
    regression_predictions[
        "Predicted Excess Return"
    ]
    .notna()
    .all()
)

assert np.isfinite(
    regression_predictions[
        "Predicted Excess Return"
    ]
).all()

assert (
    regression_predictions
    .groupby(
        [
            "Date",
            "Model",
        ]
    )[
        "Ticker"
    ]
    .nunique()
    .eq(
        len(
            INDIA_10_TICKERS
        )
    )
    .all()
)

assert (
    regression_evaluation[
        "Rebalances"
    ]
    .eq(
        regression_predictions[
            "Date"
        ].nunique()
    )
    .all()
)


# ---------------------------------------------------------
# 8. Display results
# ---------------------------------------------------------

display_regression_evaluation = (
    regression_evaluation.copy()
)

percentage_columns = [
    "Mean Absolute Error",
    "Root Mean Squared Error",
    "Pooled R-Squared",
    "Mean Rank IC",
    "Median Rank IC",
    "Positive Rank IC Rate",
    "Directional Accuracy",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in percentage_columns:

    display_regression_evaluation[
        column
    ] = (
        display_regression_evaluation[
            column
        ]
        * 100
    )


display_comparison = (
    regression_baseline_comparison.copy()
)

comparison_percentage_columns = [
    "Mean Rank IC",
    "Positive Rank IC Rate",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in comparison_percentage_columns:

    display_comparison[
        column
    ] = (
        display_comparison[
            column
        ]
        * 100
    )


print("WALK-FORWARD REGRESSION EVALUATION")
print("=" * 70)
print(
    "Evaluation period:",
    regression_predictions[
        "Date"
    ].min().date(),
    "to",
    regression_predictions[
        "Date"
    ].max().date(),
)
print(
    "Monthly rebalances:",
    regression_predictions[
        "Date"
    ].nunique(),
)
print(
    "Models:",
    regression_predictions[
        "Model"
    ].nunique(),
)
print(
    "Features:",
    len(
        feature_columns
    ),
)
print(
    "Top stocks selected:",
    TOP_STOCK_COUNT,
)
print(
    "Training-only standardisation:",
    "PASSED",
)
print(
    "Leakage-safe cutoffs:",
    "PASSED",
)
print(
    "Regression evaluation:",
    "PASSED",
)

print("\nREGRESSION MODEL METRICS")
display(
    display_regression_evaluation.round(
        2
    )
)

print("\nREGRESSION VS SIMPLE BASELINES")
display(
    display_comparison.round(
        2
    )
)

WALK-FORWARD REGRESSION EVALUATION
Evaluation period: 2018-03-28 to 2026-07-01
Monthly rebalances: 101
Models: 2
Features: 27
Top stocks selected: 3
Training-only standardisation: PASSED
Leakage-safe cutoffs: PASSED
Regression evaluation: PASSED

REGRESSION MODEL METRICS


,Rebalances,Predictions,Mean Absolute Error,Root Mean Squared Error,Pooled R-Squared,Mean Rank IC,Median Rank IC,Positive Rank IC Rate,Directional Accuracy,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Model,,,,,,,,,,,,,
Linear Regression,101,1010,5.57,7.46,-5.68,0.09,-3.03,48.51,48.91,2.33,1.31,58.75,51.82
Ridge Regression,101,1010,5.57,7.45,-5.50,0.02,-3.03,48.51,49.21,2.33,1.31,58.75,51.82



REGRESSION VS SIMPLE BASELINES


,Rebalances,Mean Rank IC,Positive Rank IC Rate,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Model,,,,,,,
1-Month Reversal,101.0,6.89,59.41,2.80,1.79,59.41,55.12
Linear Regression,101.0,0.09,48.51,2.33,1.31,58.75,51.82
Ridge Regression,101.0,0.02,48.51,2.33,1.31,58.75,51.82
12-Month Momentum,101.0,3.04,58.42,2.23,1.21,59.74,51.16
Equal-Weight India 10,101.0,NaN,NaN,1.80,0.79,58.12,51.58


## Walk-Forward Logistic Classification Models

Two classification models are evaluated:

1. Positive Return Classifier  
   Estimates the probability that a stock generates a positive return over the next 21 trading days.

2. Nifty Outperformance Classifier  
   Estimates the probability that a stock outperforms the Nifty 50 over the next 21 trading days.

Both models use expanding-window training, training-only feature standardisation and monthly out-of-sample predictions.

The three stocks with the highest predicted probabilities are selected at each rebalance date.

In [13]:
# =========================================================
# WALK-FORWARD LOGISTIC CLASSIFICATION MODELS
# =========================================================

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------
# 1. Define the two classification tasks
# ---------------------------------------------------------

classification_tasks = {
    "Positive Return Logistic": {
        "target_column":
            "positive_return_target",

        "ranking_outcome":
            "forward_return_21d",
    },

    "Nifty Outperformance Logistic": {
        "target_column":
            "outperform_target",

        "ranking_outcome":
            "forward_excess_return_21d",
    },
}


base_logistic_pipeline = Pipeline(
    steps=[
        (
            "standard_scaler",
            StandardScaler(),
        ),
        (
            "classification_model",
            LogisticRegression(
                C=1.0,
                solver="lbfgs",
                max_iter=2_000,
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)


# ---------------------------------------------------------
# 2. Generate walk-forward monthly predictions
# ---------------------------------------------------------

classification_prediction_records = []

for (
    rebalance_date,
    training_cutoff,
) in rebalance_schedule.items():

    training_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            <= training_cutoff
        ]
        .copy()
    )

    prediction_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            == rebalance_date
        ]
        .copy()
    )

    if (
        len(
            prediction_sample
        )
        != len(
            INDIA_10_TICKERS
        )
    ):
        continue

    if (
        len(
            training_sample
        )
        < (
            len(
                INDIA_10_TICKERS
            )
            * 252
        )
    ):
        continue

    X_train = (
        training_sample[
            feature_columns
        ]
        .astype(float)
    )

    X_predict = (
        prediction_sample[
            feature_columns
        ]
        .astype(float)
    )

    for (
        model_name,
        task_details,
    ) in classification_tasks.items():

        target_column = (
            task_details[
                "target_column"
            ]
        )

        ranking_outcome = (
            task_details[
                "ranking_outcome"
            ]
        )

        y_train = (
            training_sample[
                target_column
            ]
            .astype(int)
        )

        if y_train.nunique() != 2:
            raise RuntimeError(
                f"{model_name} training data does not "
                "contain both target classes."
            )

        fitted_model = clone(
            base_logistic_pipeline
        )

        fitted_model.fit(
            X_train,
            y_train,
        )

        predicted_probability = (
            fitted_model.predict_proba(
                X_predict
            )[
                :,
                1,
            ]
        )

        predicted_class = (
            predicted_probability
            >= 0.50
        ).astype(int)

        model_predictions = (
            prediction_sample[
                [
                    "Date",
                    "Ticker",
                    "forward_return_21d",
                    "forward_excess_return_21d",
                    "positive_return_target",
                    "outperform_target",
                ]
            ]
            .copy()
        )

        model_predictions[
            "Model"
        ] = model_name

        model_predictions[
            "Target Column"
        ] = target_column

        model_predictions[
            "Actual Class"
        ] = (
            prediction_sample[
                target_column
            ]
            .astype(int)
            .to_numpy()
        )

        model_predictions[
            "Ranking Outcome"
        ] = (
            prediction_sample[
                ranking_outcome
            ]
            .to_numpy()
        )

        model_predictions[
            "Predicted Probability"
        ] = predicted_probability

        model_predictions[
            "Predicted Class"
        ] = predicted_class

        model_predictions[
            "Training Cutoff"
        ] = training_cutoff

        model_predictions[
            "Training Observations"
        ] = len(
            training_sample
        )

        classification_prediction_records.append(
            model_predictions
        )


if not classification_prediction_records:
    raise RuntimeError(
        "No classification predictions were generated."
    )


classification_predictions = (
    pd.concat(
        classification_prediction_records,
        ignore_index=True,
    )
    .sort_values(
        [
            "Date",
            "Model",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 3. Evaluate classification and stock selection
# ---------------------------------------------------------

TOP_STOCK_COUNT = 3

classification_evaluation_records = []

for (
    model_name,
    model_predictions,
) in classification_predictions.groupby(
    "Model"
):

    actual_class = (
        model_predictions[
            "Actual Class"
        ]
        .astype(int)
    )

    predicted_class = (
        model_predictions[
            "Predicted Class"
        ]
        .astype(int)
    )

    predicted_probability = (
        model_predictions[
            "Predicted Probability"
        ]
    )

    monthly_rank_ic_values = []
    selected_forward_returns = []
    selected_excess_returns = []
    selected_positive_rates = []
    selected_outperformance_rates = []

    for (
        prediction_date,
        prediction_cross_section,
    ) in model_predictions.groupby(
        "Date"
    ):

        if (
            prediction_cross_section[
                "Predicted Probability"
            ].nunique()
            > 1
        ):

            rank_ic = (
                prediction_cross_section[
                    [
                        "Predicted Probability",
                        "Ranking Outcome",
                    ]
                ]
                .corr(
                    method="spearman"
                )
                .iloc[
                    0,
                    1,
                ]
            )

        else:

            rank_ic = np.nan

        monthly_rank_ic_values.append(
            rank_ic
        )

        selected_stocks = (
            prediction_cross_section
            .nlargest(
                TOP_STOCK_COUNT,
                "Predicted Probability",
            )
        )

        selected_forward_returns.append(
            selected_stocks[
                "forward_return_21d"
            ].mean()
        )

        selected_excess_returns.append(
            selected_stocks[
                "forward_excess_return_21d"
            ].mean()
        )

        selected_positive_rates.append(
            selected_stocks[
                "positive_return_target"
            ].mean()
        )

        selected_outperformance_rates.append(
            selected_stocks[
                "outperform_target"
            ].mean()
        )

    valid_rank_ic = (
        pd.Series(
            monthly_rank_ic_values,
            dtype=float,
        )
        .dropna()
    )

    classification_evaluation_records.append(
        {
            "Model":
                model_name,

            "Rebalances":
                model_predictions[
                    "Date"
                ].nunique(),

            "Predictions":
                len(
                    model_predictions
                ),

            "Accuracy":
                accuracy_score(
                    actual_class,
                    predicted_class,
                ),

            "Balanced Accuracy":
                balanced_accuracy_score(
                    actual_class,
                    predicted_class,
                ),

            "ROC AUC":
                roc_auc_score(
                    actual_class,
                    predicted_probability,
                ),

            "Log Loss":
                log_loss(
                    actual_class,
                    predicted_probability,
                    labels=[
                        0,
                        1,
                    ],
                ),

            "Brier Score":
                brier_score_loss(
                    actual_class,
                    predicted_probability,
                ),

            "Mean Rank IC":
                valid_rank_ic.mean(),

            "Median Rank IC":
                valid_rank_ic.median(),

            "Positive Rank IC Rate":
                valid_rank_ic.gt(
                    0
                ).mean(),

            "Mean Selected Forward Return":
                np.mean(
                    selected_forward_returns
                ),

            "Mean Selected Excess Return":
                np.mean(
                    selected_excess_returns
                ),

            "Selected Positive Return Rate":
                np.mean(
                    selected_positive_rates
                ),

            "Selected Outperformance Rate":
                np.mean(
                    selected_outperformance_rates
                ),
        }
    )


classification_evaluation = (
    pd.DataFrame(
        classification_evaluation_records
    )
    .set_index(
        "Model"
    )
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 4. Compare classifiers with existing baselines
# ---------------------------------------------------------

classification_baseline_comparison = (
    classification_evaluation[
        [
            "Rebalances",
            "Mean Rank IC",
            "Positive Rank IC Rate",
            "Mean Selected Forward Return",
            "Mean Selected Excess Return",
            "Selected Positive Return Rate",
            "Selected Outperformance Rate",
        ]
    ]
    .copy()
)

for baseline_name in [
    "1-Month Reversal",
    "12-Month Momentum",
    "Equal-Weight India 10",
]:

    baseline_row = (
        baseline_evaluation.loc[
            baseline_name
        ]
    )

    classification_baseline_comparison.loc[
        baseline_name
    ] = {
        "Rebalances":
            baseline_row[
                "Rebalances"
            ],

        "Mean Rank IC":
            baseline_row[
                "Mean Rank IC"
            ],

        "Positive Rank IC Rate":
            baseline_row[
                "Positive Rank IC Rate"
            ],

        "Mean Selected Forward Return":
            baseline_row[
                "Mean Selected Forward Return"
            ],

        "Mean Selected Excess Return":
            baseline_row[
                "Mean Selected Excess Return"
            ],

        "Selected Positive Return Rate":
            baseline_row[
                "Selected Positive Return Rate"
            ],

        "Selected Outperformance Rate":
            baseline_row[
                "Selected Outperformance Rate"
            ],
    }


classification_baseline_comparison = (
    classification_baseline_comparison
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 5. Save research outputs
# ---------------------------------------------------------

classification_predictions_path = (
    PROCESSED_DATA_DIR
    / "walk_forward_classification_predictions.csv"
)

classification_evaluation_path = (
    PROCESSED_DATA_DIR
    / "walk_forward_classification_evaluation.csv"
)

classification_comparison_path = (
    PROCESSED_DATA_DIR
    / "classification_baseline_comparison.csv"
)

classification_predictions.to_csv(
    classification_predictions_path,
    index=False,
)

classification_evaluation.to_csv(
    classification_evaluation_path
)

classification_baseline_comparison.to_csv(
    classification_comparison_path
)


# ---------------------------------------------------------
# 6. Leakage and quality validation
# ---------------------------------------------------------

assert (
    classification_predictions[
        "Training Cutoff"
    ]
    < classification_predictions[
        "Date"
    ]
).all()

assert (
    classification_predictions[
        "Predicted Probability"
    ]
    .between(
        0,
        1,
    )
    .all()
)

assert (
    classification_predictions[
        "Predicted Probability"
    ]
    .notna()
    .all()
)

assert (
    classification_predictions
    .groupby(
        [
            "Date",
            "Model",
        ]
    )[
        "Ticker"
    ]
    .nunique()
    .eq(
        len(
            INDIA_10_TICKERS
        )
    )
    .all()
)

assert len(
    classification_evaluation
) == 2


# ---------------------------------------------------------
# 7. Display results
# ---------------------------------------------------------

display_classification_evaluation = (
    classification_evaluation.copy()
)

percentage_columns = [
    "Accuracy",
    "Balanced Accuracy",
    "ROC AUC",
    "Brier Score",
    "Mean Rank IC",
    "Median Rank IC",
    "Positive Rank IC Rate",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in percentage_columns:

    display_classification_evaluation[
        column
    ] = (
        display_classification_evaluation[
            column
        ]
        * 100
    )


display_classification_comparison = (
    classification_baseline_comparison.copy()
)

comparison_percentage_columns = [
    "Mean Rank IC",
    "Positive Rank IC Rate",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in comparison_percentage_columns:

    display_classification_comparison[
        column
    ] = (
        display_classification_comparison[
            column
        ]
        * 100
    )


print("WALK-FORWARD CLASSIFICATION EVALUATION")
print("=" * 70)
print(
    "Evaluation period:",
    classification_predictions[
        "Date"
    ].min().date(),
    "to",
    classification_predictions[
        "Date"
    ].max().date(),
)
print(
    "Monthly rebalances:",
    classification_predictions[
        "Date"
    ].nunique(),
)
print(
    "Classification models:",
    classification_predictions[
        "Model"
    ].nunique(),
)
print(
    "Features:",
    len(
        feature_columns
    ),
)
print(
    "Top stocks selected:",
    TOP_STOCK_COUNT,
)
print(
    "Leakage-safe cutoffs:",
    "PASSED",
)
print(
    "Classification evaluation:",
    "PASSED",
)

print("\nCLASSIFICATION MODEL METRICS")
display(
    display_classification_evaluation.round(
        2
    )
)

print("\nCLASSIFICATION VS SIMPLE BASELINES")
display(
    display_classification_comparison.round(
        2
    )
)

WALK-FORWARD CLASSIFICATION EVALUATION
Evaluation period: 2018-03-28 to 2026-07-01
Monthly rebalances: 101
Classification models: 2
Features: 27
Top stocks selected: 3
Leakage-safe cutoffs: PASSED
Classification evaluation: PASSED

CLASSIFICATION MODEL METRICS


,Rebalances,Predictions,Accuracy,Balanced Accuracy,ROC AUC,Log Loss,Brier Score,Mean Rank IC,Median Rank IC,Positive Rank IC Rate,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Model,,,,,,,,,,,,,,
Positive Return Logistic,101,1010,52.28,46.63,45.51,0.73,26.57,-3.15,1.82,51.49,2.19,1.17,58.42,51.49
Nifty Outperformance Logistic,101,1010,48.61,48.33,47.43,0.73,26.37,-3.89,-6.67,45.54,1.76,0.74,57.43,48.84



CLASSIFICATION VS SIMPLE BASELINES


,Rebalances,Mean Rank IC,Positive Rank IC Rate,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Model,,,,,,,
1-Month Reversal,101.0,6.89,59.41,2.80,1.79,59.41,55.12
12-Month Momentum,101.0,3.04,58.42,2.23,1.21,59.74,51.16
Positive Return Logistic,101.0,-3.15,51.49,2.19,1.17,58.42,51.49
Equal-Weight India 10,101.0,NaN,NaN,1.80,0.79,58.12,51.58
Nifty Outperformance Logistic,101.0,-3.89,45.54,1.76,0.74,57.43,48.84


## Walk-Forward Tree-Based Regression Models

Linear and logistic models may miss non-linear relationships and interactions between momentum, volatility, drawdown and market-regime variables.

This section evaluates:

1. Random Forest Regression
2. Histogram Gradient Boosting Regression

Both models predict forward 21-trading-day excess returns and use:

- The same 27 features
- Expanding-window monthly training
- Leakage-safe target cutoffs
- Controlled model complexity
- Top-three stock selection
- The same out-of-sample dates as all previous models

The models are intentionally regularised through depth and minimum-leaf constraints to reduce overfitting.

In [14]:
# =========================================================
# WALK-FORWARD TREE-BASED REGRESSION MODELS
# Random Forest + Histogram Gradient Boosting
# =========================================================

from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


# ---------------------------------------------------------
# 1. Define controlled-complexity tree models
# ---------------------------------------------------------

tree_regression_models = {
    "Random Forest":
        RandomForestRegressor(
            n_estimators=100,
            max_depth=6,
            min_samples_leaf=50,
            max_features="sqrt",
            bootstrap=True,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),

    "Gradient Boosting":
        HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=150,
            max_leaf_nodes=15,
            max_depth=5,
            min_samples_leaf=50,
            l2_regularization=1.0,
            early_stopping=False,
            random_state=RANDOM_SEED,
        ),
}


# ---------------------------------------------------------
# 2. Generate expanding-window monthly predictions
# ---------------------------------------------------------

tree_prediction_records = []

total_rebalances = len(
    rebalance_schedule
)

for rebalance_number, (
    rebalance_date,
    training_cutoff,
) in enumerate(
    rebalance_schedule.items(),
    start=1,
):

    training_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            <= training_cutoff
        ]
        .copy()
    )

    prediction_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            == rebalance_date
        ]
        .copy()
    )

    if (
        len(
            prediction_sample
        )
        != len(
            INDIA_10_TICKERS
        )
    ):
        continue

    if (
        len(
            training_sample
        )
        < (
            len(
                INDIA_10_TICKERS
            )
            * 252
        )
    ):
        continue

    X_train = (
        training_sample[
            feature_columns
        ]
        .astype(float)
    )

    y_train = (
        training_sample[
            "forward_excess_return_21d"
        ]
        .astype(float)
    )

    X_predict = (
        prediction_sample[
            feature_columns
        ]
        .astype(float)
    )

    for model_name, model in (
        tree_regression_models.items()
    ):

        model.fit(
            X_train,
            y_train,
        )

        predicted_excess_return = (
            model.predict(
                X_predict
            )
        )

        model_predictions = (
            prediction_sample[
                [
                    "Date",
                    "Ticker",
                    "forward_return_21d",
                    "forward_excess_return_21d",
                    "positive_return_target",
                    "outperform_target",
                ]
            ]
            .copy()
        )

        model_predictions[
            "Model"
        ] = model_name

        model_predictions[
            "Predicted Excess Return"
        ] = predicted_excess_return

        model_predictions[
            "Training Cutoff"
        ] = training_cutoff

        model_predictions[
            "Training Observations"
        ] = len(
            training_sample
        )

        tree_prediction_records.append(
            model_predictions
        )

    if (
        rebalance_number == 1
        or rebalance_number % 20 == 0
        or rebalance_number == total_rebalances
    ):
        print(
            f"Completed {rebalance_number} "
            f"of {total_rebalances} rebalances."
        )


if not tree_prediction_records:
    raise RuntimeError(
        "No tree-model predictions were generated."
    )


tree_regression_predictions = (
    pd.concat(
        tree_prediction_records,
        ignore_index=True,
    )
    .sort_values(
        [
            "Date",
            "Model",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 3. Evaluate prediction and stock-ranking performance
# ---------------------------------------------------------

TOP_STOCK_COUNT = 3

tree_evaluation_records = []

for model_name, model_predictions in (
    tree_regression_predictions.groupby(
        "Model"
    )
):

    actual_values = (
        model_predictions[
            "forward_excess_return_21d"
        ]
    )

    predicted_values = (
        model_predictions[
            "Predicted Excess Return"
        ]
    )

    monthly_rank_ic_values = []
    selected_forward_returns = []
    selected_excess_returns = []
    selected_positive_rates = []
    selected_outperformance_rates = []

    for prediction_date, prediction_cross_section in (
        model_predictions.groupby(
            "Date"
        )
    ):

        if (
            prediction_cross_section[
                "Predicted Excess Return"
            ].nunique()
            > 1
        ):

            rank_ic = (
                prediction_cross_section[
                    [
                        "Predicted Excess Return",
                        "forward_excess_return_21d",
                    ]
                ]
                .corr(
                    method="spearman"
                )
                .iloc[
                    0,
                    1,
                ]
            )

        else:
            rank_ic = np.nan

        monthly_rank_ic_values.append(
            rank_ic
        )

        selected_stocks = (
            prediction_cross_section
            .nlargest(
                TOP_STOCK_COUNT,
                "Predicted Excess Return",
            )
        )

        selected_forward_returns.append(
            selected_stocks[
                "forward_return_21d"
            ].mean()
        )

        selected_excess_returns.append(
            selected_stocks[
                "forward_excess_return_21d"
            ].mean()
        )

        selected_positive_rates.append(
            selected_stocks[
                "positive_return_target"
            ].mean()
        )

        selected_outperformance_rates.append(
            selected_stocks[
                "outperform_target"
            ].mean()
        )

    valid_rank_ic = (
        pd.Series(
            monthly_rank_ic_values,
            dtype=float,
        )
        .dropna()
    )

    directional_accuracy = (
        (
            predicted_values
            > 0
        )
        == (
            actual_values
            > 0
        )
    ).mean()

    tree_evaluation_records.append(
        {
            "Model":
                model_name,

            "Rebalances":
                model_predictions[
                    "Date"
                ].nunique(),

            "Predictions":
                len(
                    model_predictions
                ),

            "Mean Absolute Error":
                mean_absolute_error(
                    actual_values,
                    predicted_values,
                ),

            "Root Mean Squared Error":
                np.sqrt(
                    mean_squared_error(
                        actual_values,
                        predicted_values,
                    )
                ),

            "Pooled R-Squared":
                r2_score(
                    actual_values,
                    predicted_values,
                ),

            "Mean Rank IC":
                valid_rank_ic.mean(),

            "Median Rank IC":
                valid_rank_ic.median(),

            "Positive Rank IC Rate":
                valid_rank_ic.gt(
                    0
                ).mean(),

            "Directional Accuracy":
                directional_accuracy,

            "Mean Selected Forward Return":
                np.mean(
                    selected_forward_returns
                ),

            "Mean Selected Excess Return":
                np.mean(
                    selected_excess_returns
                ),

            "Selected Positive Return Rate":
                np.mean(
                    selected_positive_rates
                ),

            "Selected Outperformance Rate":
                np.mean(
                    selected_outperformance_rates
                ),
        }
    )


tree_regression_evaluation = (
    pd.DataFrame(
        tree_evaluation_records
    )
    .set_index(
        "Model"
    )
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 4. Compare tree models with previous models and baselines
# ---------------------------------------------------------

tree_model_comparison = (
    tree_regression_evaluation[
        [
            "Rebalances",
            "Mean Rank IC",
            "Positive Rank IC Rate",
            "Mean Selected Forward Return",
            "Mean Selected Excess Return",
            "Selected Positive Return Rate",
            "Selected Outperformance Rate",
        ]
    ]
    .copy()
)


for model_name in [
    "Linear Regression",
    "Ridge Regression",
]:

    model_row = (
        regression_evaluation.loc[
            model_name
        ]
    )

    tree_model_comparison.loc[
        model_name
    ] = {
        "Rebalances":
            model_row[
                "Rebalances"
            ],

        "Mean Rank IC":
            model_row[
                "Mean Rank IC"
            ],

        "Positive Rank IC Rate":
            model_row[
                "Positive Rank IC Rate"
            ],

        "Mean Selected Forward Return":
            model_row[
                "Mean Selected Forward Return"
            ],

        "Mean Selected Excess Return":
            model_row[
                "Mean Selected Excess Return"
            ],

        "Selected Positive Return Rate":
            model_row[
                "Selected Positive Return Rate"
            ],

        "Selected Outperformance Rate":
            model_row[
                "Selected Outperformance Rate"
            ],
    }


for baseline_name in [
    "1-Month Reversal",
    "12-Month Momentum",
    "Equal-Weight India 10",
]:

    baseline_row = (
        baseline_evaluation.loc[
            baseline_name
        ]
    )

    tree_model_comparison.loc[
        baseline_name
    ] = {
        "Rebalances":
            baseline_row[
                "Rebalances"
            ],

        "Mean Rank IC":
            baseline_row[
                "Mean Rank IC"
            ],

        "Positive Rank IC Rate":
            baseline_row[
                "Positive Rank IC Rate"
            ],

        "Mean Selected Forward Return":
            baseline_row[
                "Mean Selected Forward Return"
            ],

        "Mean Selected Excess Return":
            baseline_row[
                "Mean Selected Excess Return"
            ],

        "Selected Positive Return Rate":
            baseline_row[
                "Selected Positive Return Rate"
            ],

        "Selected Outperformance Rate":
            baseline_row[
                "Selected Outperformance Rate"
            ],
    }


tree_model_comparison = (
    tree_model_comparison
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 5. Save the tree-model outputs
# ---------------------------------------------------------

tree_predictions_path = (
    PROCESSED_DATA_DIR
    / "walk_forward_tree_predictions.csv"
)

tree_evaluation_path = (
    PROCESSED_DATA_DIR
    / "walk_forward_tree_evaluation.csv"
)

tree_comparison_path = (
    PROCESSED_DATA_DIR
    / "tree_model_comparison.csv"
)

tree_regression_predictions.to_csv(
    tree_predictions_path,
    index=False,
)

tree_regression_evaluation.to_csv(
    tree_evaluation_path
)

tree_model_comparison.to_csv(
    tree_comparison_path
)


# ---------------------------------------------------------
# 6. Leakage and quality validation
# ---------------------------------------------------------

assert (
    tree_regression_predictions[
        "Training Cutoff"
    ]
    < tree_regression_predictions[
        "Date"
    ]
).all()

assert (
    tree_regression_predictions[
        "Predicted Excess Return"
    ]
    .notna()
    .all()
)

assert np.isfinite(
    tree_regression_predictions[
        "Predicted Excess Return"
    ]
).all()

assert (
    tree_regression_predictions
    .groupby(
        [
            "Date",
            "Model",
        ]
    )[
        "Ticker"
    ]
    .nunique()
    .eq(
        len(
            INDIA_10_TICKERS
        )
    )
    .all()
)

assert len(
    tree_regression_evaluation
) == 2


# ---------------------------------------------------------
# 7. Display results
# ---------------------------------------------------------

display_tree_evaluation = (
    tree_regression_evaluation.copy()
)

tree_percentage_columns = [
    "Mean Absolute Error",
    "Root Mean Squared Error",
    "Pooled R-Squared",
    "Mean Rank IC",
    "Median Rank IC",
    "Positive Rank IC Rate",
    "Directional Accuracy",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in tree_percentage_columns:

    display_tree_evaluation[
        column
    ] = (
        display_tree_evaluation[
            column
        ]
        * 100
    )


display_tree_comparison = (
    tree_model_comparison.copy()
)

comparison_percentage_columns = [
    "Mean Rank IC",
    "Positive Rank IC Rate",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in comparison_percentage_columns:

    display_tree_comparison[
        column
    ] = (
        display_tree_comparison[
            column
        ]
        * 100
    )


print("WALK-FORWARD TREE MODEL EVALUATION")
print("=" * 70)
print(
    "Evaluation period:",
    tree_regression_predictions[
        "Date"
    ].min().date(),
    "to",
    tree_regression_predictions[
        "Date"
    ].max().date(),
)
print(
    "Monthly rebalances:",
    tree_regression_predictions[
        "Date"
    ].nunique(),
)
print(
    "Tree models:",
    tree_regression_predictions[
        "Model"
    ].nunique(),
)
print(
    "Features:",
    len(
        feature_columns
    ),
)
print(
    "Top stocks selected:",
    TOP_STOCK_COUNT,
)
print(
    "Leakage-safe cutoffs:",
    "PASSED",
)
print(
    "Tree-model evaluation:",
    "PASSED",
)

print("\nTREE MODEL METRICS")
display(
    display_tree_evaluation.round(
        2
    )
)

print("\nTREE MODELS VS PREVIOUS MODELS")
display(
    display_tree_comparison.round(
        2
    )
)

Completed 1 of 101 rebalances.
Completed 20 of 101 rebalances.
Completed 40 of 101 rebalances.
Completed 60 of 101 rebalances.
Completed 80 of 101 rebalances.
Completed 100 of 101 rebalances.
Completed 101 of 101 rebalances.
WALK-FORWARD TREE MODEL EVALUATION
Evaluation period: 2018-03-28 to 2026-07-01
Monthly rebalances: 101
Tree models: 2
Features: 27
Top stocks selected: 3
Leakage-safe cutoffs: PASSED
Tree-model evaluation: PASSED

TREE MODEL METRICS


,Rebalances,Predictions,Mean Absolute Error,Root Mean Squared Error,Pooled R-Squared,Mean Rank IC,Median Rank IC,Positive Rank IC Rate,Directional Accuracy,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Model,,,,,,,,,,,,,
Gradient Boosting,101,1010,5.62,7.56,-8.47,1.69,0.61,50.50,50.69,2.24,1.23,59.41,51.49
Random Forest,101,1010,5.44,7.25,0.18,1.27,3.03,52.48,50.00,1.96,0.94,55.78,49.83



TREE MODELS VS PREVIOUS MODELS


,Rebalances,Mean Rank IC,Positive Rank IC Rate,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Model,,,,,,,
1-Month Reversal,101.0,6.89,59.41,2.80,1.79,59.41,55.12
Ridge Regression,101.0,0.02,48.51,2.33,1.31,58.75,51.82
Linear Regression,101.0,0.09,48.51,2.33,1.31,58.75,51.82
Gradient Boosting,101.0,1.69,50.50,2.24,1.23,59.41,51.49
12-Month Momentum,101.0,3.04,58.42,2.23,1.21,59.74,51.16
Random Forest,101.0,1.27,52.48,1.96,0.94,55.78,49.83
Equal-Weight India 10,101.0,NaN,NaN,1.80,0.79,58.12,51.58


## Monthly Signal-to-Portfolio Conversion

Convert every out-of-sample model prediction into an investable long-only portfolio.

Portfolio-construction rules:

- Rebalance monthly
- Execute signals on the next trading day
- Select the three highest-ranked stocks
- Allocate equally across selected stocks
- Maintain zero weight in unselected stocks
- Use the same construction method for all models to ensure a fair comparison
- Include the equal-weight India 10 portfolio as a benchmark

Transaction costs and daily portfolio returns will be applied in the next step.

In [15]:
# =========================================================
# CONVERT MODEL SIGNALS INTO MONTHLY PORTFOLIO WEIGHTS
# =========================================================

required_objects = [
    "baseline_predictions",
    "regression_predictions",
    "classification_predictions",
    "tree_regression_predictions",
    "market_calendar",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required previous-step objects are missing:\n"
        + "\n".join(
            missing_objects
        )
    )


TOP_STOCK_COUNT = 3

signal_records = []


# ---------------------------------------------------------
# 1. Add the strongest simple baseline signals
# ---------------------------------------------------------

baseline_signal_definitions = {
    "1-Month Reversal":
        "short_term_reversal_score",

    "12-Month Momentum":
        "momentum_252d_score",
}

for strategy_name, score_column in (
    baseline_signal_definitions.items()
):

    strategy_signals = (
        baseline_predictions[
            [
                "Date",
                "Ticker",
                score_column,
            ]
        ]
        .rename(
            columns={
                score_column:
                    "Score",
            }
        )
        .copy()
    )

    strategy_signals[
        "Strategy"
    ] = strategy_name

    strategy_signals[
        "Signal Type"
    ] = "Rule-Based Baseline"

    signal_records.append(
        strategy_signals
    )


# ---------------------------------------------------------
# 2. Add linear-regression signals
# ---------------------------------------------------------

for model_name in [
    "Linear Regression",
    "Ridge Regression",
]:

    strategy_signals = (
        regression_predictions.loc[
            regression_predictions[
                "Model"
            ]
            == model_name,
            [
                "Date",
                "Ticker",
                "Predicted Excess Return",
            ],
        ]
        .rename(
            columns={
                "Predicted Excess Return":
                    "Score",
            }
        )
        .copy()
    )

    strategy_signals[
        "Strategy"
    ] = model_name

    strategy_signals[
        "Signal Type"
    ] = "Regression Model"

    signal_records.append(
        strategy_signals
    )


# ---------------------------------------------------------
# 3. Add logistic-classification signals
# ---------------------------------------------------------

for model_name in [
    "Positive Return Logistic",
    "Nifty Outperformance Logistic",
]:

    strategy_signals = (
        classification_predictions.loc[
            classification_predictions[
                "Model"
            ]
            == model_name,
            [
                "Date",
                "Ticker",
                "Predicted Probability",
            ],
        ]
        .rename(
            columns={
                "Predicted Probability":
                    "Score",
            }
        )
        .copy()
    )

    strategy_signals[
        "Strategy"
    ] = model_name

    strategy_signals[
        "Signal Type"
    ] = "Classification Model"

    signal_records.append(
        strategy_signals
    )


# ---------------------------------------------------------
# 4. Add tree-model signals
# ---------------------------------------------------------

for model_name in [
    "Random Forest",
    "Gradient Boosting",
]:

    strategy_signals = (
        tree_regression_predictions.loc[
            tree_regression_predictions[
                "Model"
            ]
            == model_name,
            [
                "Date",
                "Ticker",
                "Predicted Excess Return",
            ],
        ]
        .rename(
            columns={
                "Predicted Excess Return":
                    "Score",
            }
        )
        .copy()
    )

    strategy_signals[
        "Strategy"
    ] = model_name

    strategy_signals[
        "Signal Type"
    ] = "Tree-Based Model"

    signal_records.append(
        strategy_signals
    )


combined_strategy_signals = (
    pd.concat(
        signal_records,
        ignore_index=True,
    )
    .sort_values(
        [
            "Date",
            "Strategy",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)

combined_strategy_signals[
    "Date"
] = pd.to_datetime(
    combined_strategy_signals[
        "Date"
    ]
)


# ---------------------------------------------------------
# 5. Map every signal date to the next trading date
# ---------------------------------------------------------

market_calendar = pd.DatetimeIndex(
    market_calendar
).sort_values()


def get_next_trading_date(
    signal_date,
):

    calendar_position = (
        market_calendar.searchsorted(
            signal_date,
            side="right",
        )
    )

    if calendar_position >= len(
        market_calendar
    ):
        return pd.NaT

    return market_calendar[
        calendar_position
    ]


execution_date_map = {
    signal_date:
        get_next_trading_date(
            signal_date
        )
    for signal_date in sorted(
        combined_strategy_signals[
            "Date"
        ].unique()
    )
}


# ---------------------------------------------------------
# 6. Convert scores into top-three equal-weight portfolios
# ---------------------------------------------------------

portfolio_weight_records = []

for (
    strategy_name,
    signal_date,
), cross_section in (
    combined_strategy_signals.groupby(
        [
            "Strategy",
            "Date",
        ]
    )
):

    if len(
        cross_section
    ) != len(
        INDIA_10_TICKERS
    ):
        raise RuntimeError(
            f"{strategy_name} has an incomplete "
            f"cross-section on {signal_date}."
        )

    execution_date = (
        execution_date_map[
            signal_date
        ]
    )

    if pd.isna(
        execution_date
    ):
        continue

    ranked_cross_section = (
        cross_section
        .sort_values(
            by=[
                "Score",
                "Ticker",
            ],
            ascending=[
                False,
                True,
            ],
        )
    )

    selected_tickers = (
        ranked_cross_section[
            "Ticker"
        ]
        .head(
            TOP_STOCK_COUNT
        )
        .tolist()
    )

    score_map = (
        ranked_cross_section
        .set_index(
            "Ticker"
        )[
            "Score"
        ]
        .to_dict()
    )

    signal_type = (
        cross_section[
            "Signal Type"
        ]
        .iloc[
            0
        ]
    )

    for ticker in INDIA_10_TICKERS:

        portfolio_weight_records.append(
            {
                "Strategy":
                    strategy_name,

                "Signal Type":
                    signal_type,

                "Signal Date":
                    signal_date,

                "Execution Date":
                    execution_date,

                "Ticker":
                    ticker,

                "Score":
                    score_map[
                        ticker
                    ],

                "Selected":
                    ticker
                    in selected_tickers,

                "Target Weight":
                    (
                        1
                        / TOP_STOCK_COUNT
                        if ticker
                        in selected_tickers
                        else 0.0
                    ),
            }
        )


# ---------------------------------------------------------
# 7. Add equal-weight India 10 benchmark weights
# ---------------------------------------------------------

signal_dates = sorted(
    combined_strategy_signals[
        "Date"
    ].unique()
)

for signal_date in signal_dates:

    execution_date = (
        execution_date_map[
            signal_date
        ]
    )

    if pd.isna(
        execution_date
    ):
        continue

    for ticker in INDIA_10_TICKERS:

        portfolio_weight_records.append(
            {
                "Strategy":
                    "Equal-Weight India 10",

                "Signal Type":
                    "Portfolio Benchmark",

                "Signal Date":
                    signal_date,

                "Execution Date":
                    execution_date,

                "Ticker":
                    ticker,

                "Score":
                    np.nan,

                "Selected":
                    True,

                "Target Weight":
                    (
                        1
                        / len(
                            INDIA_10_TICKERS
                        )
                    ),
            }
        )


monthly_strategy_weights = (
    pd.DataFrame(
        portfolio_weight_records
    )
    .sort_values(
        [
            "Execution Date",
            "Strategy",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 8. Create portfolio-construction summary
# ---------------------------------------------------------

rebalance_level_summary = (
    monthly_strategy_weights
    .groupby(
        [
            "Strategy",
            "Signal Date",
            "Execution Date",
        ]
    )
    .agg(
        Weight_Total=(
            "Target Weight",
            "sum",
        ),

        Active_Holdings=(
            "Selected",
            "sum",
        ),

        Maximum_Weight=(
            "Target Weight",
            "max",
        ),
    )
    .reset_index()
)


strategy_weight_summary = (
    rebalance_level_summary
    .groupby(
        "Strategy"
    )
    .agg(
        Rebalances=(
            "Signal Date",
            "nunique",
        ),

        First_Signal_Date=(
            "Signal Date",
            "min",
        ),

        Last_Signal_Date=(
            "Signal Date",
            "max",
        ),

        First_Execution_Date=(
            "Execution Date",
            "min",
        ),

        Last_Execution_Date=(
            "Execution Date",
            "max",
        ),

        Median_Active_Holdings=(
            "Active_Holdings",
            "median",
        ),

        Average_Maximum_Weight=(
            "Maximum_Weight",
            "mean",
        ),
    )
    .sort_index()
)


# ---------------------------------------------------------
# 9. Save portfolio-weight datasets
# ---------------------------------------------------------

strategy_signals_path = (
    PROCESSED_DATA_DIR
    / "combined_strategy_signals.csv"
)

strategy_weights_path = (
    PROCESSED_DATA_DIR
    / "monthly_strategy_weights.csv"
)

strategy_summary_path = (
    PROCESSED_DATA_DIR
    / "strategy_weight_summary.csv"
)

combined_strategy_signals.to_csv(
    strategy_signals_path,
    index=False,
)

monthly_strategy_weights.to_csv(
    strategy_weights_path,
    index=False,
)

strategy_weight_summary.to_csv(
    strategy_summary_path
)


# ---------------------------------------------------------
# 10. Validate portfolio weights
# ---------------------------------------------------------

assert (
    monthly_strategy_weights[
        "Target Weight"
    ]
    .ge(
        0
    )
    .all()
)

assert np.allclose(
    rebalance_level_summary[
        "Weight_Total"
    ],
    1.0,
    atol=1e-10,
)

assert (
    monthly_strategy_weights[
        "Execution Date"
    ]
    > monthly_strategy_weights[
        "Signal Date"
    ]
).all()

assert not monthly_strategy_weights.duplicated(
    subset=[
        "Strategy",
        "Signal Date",
        "Ticker",
    ]
).any()

model_strategy_names = (
    set(
        monthly_strategy_weights[
            "Strategy"
        ].unique()
    )
    - {
        "Equal-Weight India 10",
    }
)

model_rebalance_summary = (
    rebalance_level_summary.loc[
        rebalance_level_summary[
            "Strategy"
        ].isin(
            model_strategy_names
        )
    ]
)

assert (
    model_rebalance_summary[
        "Active_Holdings"
    ]
    .eq(
        TOP_STOCK_COUNT
    )
    .all()
)

assert np.allclose(
    model_rebalance_summary[
        "Maximum_Weight"
    ],
    1
    / TOP_STOCK_COUNT,
)

equal_weight_summary = (
    rebalance_level_summary.loc[
        rebalance_level_summary[
            "Strategy"
        ]
        == "Equal-Weight India 10"
    ]
)

assert (
    equal_weight_summary[
        "Active_Holdings"
    ]
    .eq(
        len(
            INDIA_10_TICKERS
        )
    )
    .all()
)

assert np.allclose(
    equal_weight_summary[
        "Maximum_Weight"
    ],
    1
    / len(
        INDIA_10_TICKERS
    ),
)


# ---------------------------------------------------------
# 11. Display results
# ---------------------------------------------------------

display_summary = (
    strategy_weight_summary.copy()
)

display_summary[
    "Average_Maximum_Weight"
] = (
    display_summary[
        "Average_Maximum_Weight"
    ]
    * 100
)


print("MONTHLY PORTFOLIO WEIGHT CONSTRUCTION")
print("=" * 70)
print(
    "Strategies:",
    monthly_strategy_weights[
        "Strategy"
    ].nunique(),
)
print(
    "Signal dates:",
    monthly_strategy_weights[
        "Signal Date"
    ].nunique(),
)
print(
    "First signal date:",
    monthly_strategy_weights[
        "Signal Date"
    ].min().date(),
)
print(
    "Last signal date:",
    monthly_strategy_weights[
        "Signal Date"
    ].max().date(),
)
print(
    "First execution date:",
    monthly_strategy_weights[
        "Execution Date"
    ].min().date(),
)
print(
    "Top-ranked holdings per model:",
    TOP_STOCK_COUNT,
)
print(
    "One-trading-day execution lag:",
    "PASSED",
)
print(
    "All portfolio weights sum to 100%:",
    "PASSED",
)
print(
    "Signal-to-weight conversion:",
    "PASSED",
)

display(
    display_summary.round(
        2
    )
)

MONTHLY PORTFOLIO WEIGHT CONSTRUCTION
Strategies: 9
Signal dates: 101
First signal date: 2018-03-28
Last signal date: 2026-07-01
First execution date: 2018-04-02
Top-ranked holdings per model: 3
One-trading-day execution lag: PASSED
All portfolio weights sum to 100%: PASSED
Signal-to-weight conversion: PASSED


,Rebalances,First_Signal_Date,Last_Signal_Date,First_Execution_Date,Last_Execution_Date,Median_Active_Holdings,Average_Maximum_Weight
Strategy,,,,,,,
1-Month Reversal,101,2018-03-28,2026-07-01,2018-04-02,2026-07-02,3.0,33.33
12-Month Momentum,101,2018-03-28,2026-07-01,2018-04-02,2026-07-02,3.0,33.33
Equal-Weight India 10,101,2018-03-28,2026-07-01,2018-04-02,2026-07-02,10.0,10.00
Gradient Boosting,101,2018-03-28,2026-07-01,2018-04-02,2026-07-02,3.0,33.33
Linear Regression,101,2018-03-28,2026-07-01,2018-04-02,2026-07-02,3.0,33.33
Nifty Outperformance Logistic,101,2018-03-28,2026-07-01,2018-04-02,2026-07-02,3.0,33.33
Positive Return Logistic,101,2018-03-28,2026-07-01,2018-04-02,2026-07-02,3.0,33.33
Random Forest,101,2018-03-28,2026-07-01,2018-04-02,2026-07-02,3.0,33.33
Ridge Regression,101,2018-03-28,2026-07-01,2018-04-02,2026-07-02,3.0,33.33


## Transaction-Cost-Adjusted Strategy Backtest

The monthly target weights are converted into complete daily portfolio histories.

Methodology:

- Signals are formed at month-end.
- Target weights are executed at the close of the next trading day.
- New weights begin earning returns from the following trading day.
- Holdings drift naturally between rebalances.
- One-way turnover is calculated from required purchases.
- Transaction costs equal 0.15% of one-way turnover.
- The first portfolio allocation also incurs transaction costs.
- Every strategy starts with ₹10,00,000.
- Performance is compared with the Nifty 50 over the same period.

This creates a stricter test than comparing overlapping 21-day forward returns.

In [16]:
# =========================================================
# DAILY TRANSACTION-COST-ADJUSTED STRATEGY BACKTEST
# =========================================================

INITIAL_INVESTMENT_INR = 1_000_000

asset_daily_returns = (
    close_prices[
        INDIA_10_TICKERS
    ]
    .pct_change(
        fill_method=None
    )
)

nifty_daily_returns = (
    close_prices[
        BENCHMARK_TICKER
    ]
    .pct_change(
        fill_method=None
    )
)

monthly_strategy_weights[
    "Signal Date"
] = pd.to_datetime(
    monthly_strategy_weights[
        "Signal Date"
    ]
)

monthly_strategy_weights[
    "Execution Date"
] = pd.to_datetime(
    monthly_strategy_weights[
        "Execution Date"
    ]
)


# ---------------------------------------------------------
# 1. Daily backtest function
# ---------------------------------------------------------

def backtest_monthly_target_weights(
    strategy_weights: pd.DataFrame,
    daily_asset_returns: pd.DataFrame,
    initial_investment_inr: float,
    one_way_transaction_cost: float,
) -> pd.DataFrame:

    target_weight_matrix = (
        strategy_weights
        .pivot(
            index="Execution Date",
            columns="Ticker",
            values="Target Weight",
        )
        .reindex(
            columns=INDIA_10_TICKERS,
            fill_value=0.0,
        )
        .sort_index()
    )

    if target_weight_matrix.empty:
        raise ValueError(
            "No target weights were supplied."
        )

    if not np.allclose(
        target_weight_matrix.sum(
            axis=1
        ),
        1.0,
        atol=1e-10,
    ):
        raise ValueError(
            "Target portfolio weights must sum to 100%."
        )

    missing_execution_dates = (
        target_weight_matrix.index.difference(
            daily_asset_returns.index
        )
    )

    if len(
        missing_execution_dates
    ) > 0:
        raise ValueError(
            "Execution dates are missing from the "
            "market calendar: "
            + ", ".join(
                str(date.date())
                for date in missing_execution_dates
            )
        )

    first_execution_date = (
        target_weight_matrix.index.min()
    )

    backtest_returns = (
        daily_asset_returns
        .loc[
            first_execution_date:
        ]
        .reindex(
            columns=INDIA_10_TICKERS
        )
        .copy()
    )

    if backtest_returns.isna().any().any():
        missing_columns = (
            backtest_returns
            .columns[
                backtest_returns
                .isna()
                .any()
            ]
            .tolist()
        )

        raise ValueError(
            "Missing asset returns in the backtest for: "
            + ", ".join(
                missing_columns
            )
        )

    current_weights = pd.Series(
        0.0,
        index=INDIA_10_TICKERS,
        dtype=float,
    )

    portfolio_value = float(
        initial_investment_inr
    )

    daily_records = []

    for trading_date, daily_returns in (
        backtest_returns.iterrows()
    ):

        previous_value = (
            portfolio_value
        )

        # Existing holdings earn today's close-to-close return.
        gross_portfolio_return = float(
            (
                current_weights
                * daily_returns
            ).sum()
        )

        value_before_rebalance = (
            previous_value
            * (
                1
                + gross_portfolio_return
            )
        )

        # Allow existing holdings to drift before rebalancing.
        if current_weights.sum() > 0:

            gross_growth_factor = (
                1
                + gross_portfolio_return
            )

            if gross_growth_factor <= 0:
                raise RuntimeError(
                    "Portfolio value became non-positive."
                )

            drifted_weights = (
                current_weights
                * (
                    1
                    + daily_returns
                )
                / gross_growth_factor
            )

        else:

            drifted_weights = (
                current_weights.copy()
            )

        one_way_turnover = 0.0
        transaction_cost_rate = 0.0
        transaction_cost_inr = 0.0
        rebalanced = False

        # Rebalance at today's close. The new holdings therefore
        # begin earning returns on the next trading day.
        if trading_date in (
            target_weight_matrix.index
        ):

            target_weights = (
                target_weight_matrix
                .loc[
                    trading_date
                ]
                .astype(float)
            )

            one_way_turnover = float(
                (
                    target_weights
                    - drifted_weights
                )
                .clip(
                    lower=0
                )
                .sum()
            )

            transaction_cost_rate = (
                one_way_turnover
                * one_way_transaction_cost
            )

            transaction_cost_inr = (
                value_before_rebalance
                * transaction_cost_rate
            )

            portfolio_value = (
                value_before_rebalance
                - transaction_cost_inr
            )

            current_weights = (
                target_weights.copy()
            )

            rebalanced = True

        else:

            portfolio_value = (
                value_before_rebalance
            )

            current_weights = (
                drifted_weights
            )

        net_portfolio_return = (
            portfolio_value
            / previous_value
            - 1
        )

        daily_records.append(
            {
                "Date":
                    trading_date,

                "Gross Return":
                    gross_portfolio_return,

                "Net Return":
                    net_portfolio_return,

                "Portfolio Value (₹)":
                    portfolio_value,

                "One-Way Turnover":
                    one_way_turnover,

                "Transaction Cost Rate":
                    transaction_cost_rate,

                "Transaction Cost (₹)":
                    transaction_cost_inr,

                "Rebalanced":
                    rebalanced,

                "Active Holdings":
                    int(
                        current_weights
                        .gt(
                            0
                        )
                        .sum()
                    ),
            }
        )

    return (
        pd.DataFrame(
            daily_records
        )
        .set_index(
            "Date"
        )
    )


# ---------------------------------------------------------
# 2. Backtest every strategy
# ---------------------------------------------------------

strategy_backtest_results = {}

for strategy_name, strategy_weights in (
    monthly_strategy_weights.groupby(
        "Strategy"
    )
):

    strategy_backtest_results[
        strategy_name
    ] = backtest_monthly_target_weights(
        strategy_weights=(
            strategy_weights
        ),
        daily_asset_returns=(
            asset_daily_returns
        ),
        initial_investment_inr=(
            INITIAL_INVESTMENT_INR
        ),
        one_way_transaction_cost=(
            ONE_WAY_TRANSACTION_COST
        ),
    )


# ---------------------------------------------------------
# 3. Combine daily strategy results
# ---------------------------------------------------------

strategy_daily_net_returns = (
    pd.concat(
        {
            strategy_name:
                result[
                    "Net Return"
                ]
            for strategy_name, result in (
                strategy_backtest_results.items()
            )
        },
        axis=1,
    )
)

strategy_daily_gross_returns = (
    pd.concat(
        {
            strategy_name:
                result[
                    "Gross Return"
                ]
            for strategy_name, result in (
                strategy_backtest_results.items()
            )
        },
        axis=1,
    )
)

strategy_daily_values = (
    pd.concat(
        {
            strategy_name:
                result[
                    "Portfolio Value (₹)"
                ]
            for strategy_name, result in (
                strategy_backtest_results.items()
            )
        },
        axis=1,
    )
)

strategy_daily_turnover = (
    pd.concat(
        {
            strategy_name:
                result[
                    "One-Way Turnover"
                ]
            for strategy_name, result in (
                strategy_backtest_results.items()
            )
        },
        axis=1,
    )
)

strategy_daily_costs = (
    pd.concat(
        {
            strategy_name:
                result[
                    "Transaction Cost (₹)"
                ]
            for strategy_name, result in (
                strategy_backtest_results.items()
            )
        },
        axis=1,
    )
)


# ---------------------------------------------------------
# 4. Add the Nifty 50 benchmark
# ---------------------------------------------------------

common_backtest_index = (
    strategy_daily_net_returns.index
)

nifty_backtest_returns = (
    nifty_daily_returns
    .reindex(
        common_backtest_index
    )
    .copy()
)

if nifty_backtest_returns.isna().any():
    raise RuntimeError(
        "The Nifty 50 benchmark contains missing "
        "returns during the backtest."
    )

# Strategies are first invested at the close of the first
# execution date, so the benchmark also begins from that close.
nifty_backtest_returns.iloc[
    0
] = 0.0

nifty_portfolio_value = (
    INITIAL_INVESTMENT_INR
    * (
        1
        + nifty_backtest_returns
    )
    .cumprod()
)

strategy_daily_net_returns[
    "Nifty 50"
] = nifty_backtest_returns

strategy_daily_gross_returns[
    "Nifty 50"
] = nifty_backtest_returns

strategy_daily_values[
    "Nifty 50"
] = nifty_portfolio_value

strategy_daily_turnover[
    "Nifty 50"
] = 0.0

strategy_daily_costs[
    "Nifty 50"
] = 0.0


# ---------------------------------------------------------
# 5. Calculate portfolio performance statistics
# ---------------------------------------------------------

daily_risk_free_rate = (
    (
        1
        + RISK_FREE_RATE
    )
    ** (
        1
        / TRADING_DAYS_PER_YEAR
    )
    - 1
)


def calculate_backtest_statistics(
    strategy_name: str,
    daily_returns: pd.Series,
    portfolio_values: pd.Series,
    benchmark_returns: pd.Series,
    daily_turnover: pd.Series,
    daily_costs: pd.Series,
) -> dict:

    daily_returns = (
        daily_returns
        .dropna()
    )

    portfolio_values = (
        portfolio_values
        .reindex(
            daily_returns.index
        )
    )

    benchmark_returns = (
        benchmark_returns
        .reindex(
            daily_returns.index
        )
    )

    elapsed_years = (
        (
            daily_returns.index.max()
            - daily_returns.index.min()
        ).days
        / 365.25
    )

    if elapsed_years <= 0:
        raise ValueError(
            "The backtest period is too short."
        )

    ending_value = float(
        portfolio_values.iloc[
            -1
        ]
    )

    total_return = (
        ending_value
        / INITIAL_INVESTMENT_INR
        - 1
    )

    cagr = (
        (
            ending_value
            / INITIAL_INVESTMENT_INR
        )
        ** (
            1
            / elapsed_years
        )
        - 1
    )

    annualised_volatility = (
        daily_returns.std(
            ddof=1
        )
        * np.sqrt(
            TRADING_DAYS_PER_YEAR
        )
    )

    annualised_excess_return = (
        (
            daily_returns.mean()
            - daily_risk_free_rate
        )
        * TRADING_DAYS_PER_YEAR
    )

    sharpe_ratio = (
        annualised_excess_return
        / annualised_volatility
        if annualised_volatility > 0
        else np.nan
    )

    drawdown_series = (
        portfolio_values
        / portfolio_values.cummax()
        - 1
    )

    maximum_drawdown = float(
        drawdown_series.min()
    )

    calmar_ratio = (
        cagr
        / abs(
            maximum_drawdown
        )
        if maximum_drawdown < 0
        else np.nan
    )

    benchmark_variance = (
        benchmark_returns.var(
            ddof=1
        )
    )

    beta = (
        daily_returns.cov(
            benchmark_returns
        )
        / benchmark_variance
        if benchmark_variance > 0
        else np.nan
    )

    benchmark_correlation = (
        daily_returns.corr(
            benchmark_returns
        )
    )

    total_turnover = float(
        daily_turnover.sum()
    )

    annualised_turnover = (
        total_turnover
        / elapsed_years
    )

    total_transaction_cost = float(
        daily_costs.sum()
    )

    return {
        "Strategy":
            strategy_name,

        "Start Date":
            daily_returns.index.min(),

        "End Date":
            daily_returns.index.max(),

        "Years":
            elapsed_years,

        "Ending Value (₹)":
            ending_value,

        "Total Return":
            total_return,

        "CAGR":
            cagr,

        "Annualised Volatility":
            annualised_volatility,

        "Sharpe Ratio":
            sharpe_ratio,

        "Maximum Drawdown":
            maximum_drawdown,

        "Calmar Ratio":
            calmar_ratio,

        "Beta vs Nifty":
            beta,

        "Correlation vs Nifty":
            benchmark_correlation,

        "Total One-Way Turnover":
            total_turnover,

        "Annualised Turnover":
            annualised_turnover,

        "Transaction Costs (₹)":
            total_transaction_cost,

        "Rebalances":
            int(
                daily_turnover
                .gt(
                    0
                )
                .sum()
            ),
    }


backtest_summary_records = []

for strategy_name in (
    strategy_daily_net_returns.columns
):

    backtest_summary_records.append(
        calculate_backtest_statistics(
            strategy_name=(
                strategy_name
            ),
            daily_returns=(
                strategy_daily_net_returns[
                    strategy_name
                ]
            ),
            portfolio_values=(
                strategy_daily_values[
                    strategy_name
                ]
            ),
            benchmark_returns=(
                strategy_daily_net_returns[
                    "Nifty 50"
                ]
            ),
            daily_turnover=(
                strategy_daily_turnover[
                    strategy_name
                ]
            ),
            daily_costs=(
                strategy_daily_costs[
                    strategy_name
                ]
            ),
        )
    )


strategy_backtest_summary = (
    pd.DataFrame(
        backtest_summary_records
    )
    .set_index(
        "Strategy"
    )
    .sort_values(
        "Sharpe Ratio",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 6. Save backtest datasets
# ---------------------------------------------------------

strategy_daily_net_returns.to_csv(
    PROCESSED_DATA_DIR
    / "strategy_daily_net_returns.csv",
    index_label="Date",
)

strategy_daily_gross_returns.to_csv(
    PROCESSED_DATA_DIR
    / "strategy_daily_gross_returns.csv",
    index_label="Date",
)

strategy_daily_values.to_csv(
    PROCESSED_DATA_DIR
    / "strategy_daily_portfolio_values.csv",
    index_label="Date",
)

strategy_daily_turnover.to_csv(
    PROCESSED_DATA_DIR
    / "strategy_daily_turnover.csv",
    index_label="Date",
)

strategy_daily_costs.to_csv(
    PROCESSED_DATA_DIR
    / "strategy_daily_transaction_costs.csv",
    index_label="Date",
)

strategy_backtest_summary.to_csv(
    PROCESSED_DATA_DIR
    / "strategy_backtest_summary.csv",
)


# ---------------------------------------------------------
# 7. Validate the complete backtest
# ---------------------------------------------------------

expected_strategy_count = (
    monthly_strategy_weights[
        "Strategy"
    ].nunique()
    + 1
)

assert (
    strategy_daily_net_returns.shape[
        1
    ]
    == expected_strategy_count
)

assert (
    strategy_daily_net_returns
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    strategy_daily_net_returns
    .to_numpy()
).all()

assert (
    strategy_daily_values
    .gt(
        0
    )
    .all()
    .all()
)

assert (
    strategy_daily_net_returns.index
    .equals(
        strategy_daily_values.index
    )
)

assert (
    strategy_daily_turnover[
        "Nifty 50"
    ]
    .eq(
        0
    )
    .all()
)

assert (
    strategy_daily_costs[
        "Nifty 50"
    ]
    .eq(
        0
    )
    .all()
)

assert (
    strategy_backtest_summary[
        "Rebalances"
    ]
    .drop(
        index="Nifty 50"
    )
    .eq(
        101
    )
    .all()
)


# ---------------------------------------------------------
# 8. Display results
# ---------------------------------------------------------

display_backtest_summary = (
    strategy_backtest_summary.copy()
)

percentage_columns = [
    "Total Return",
    "CAGR",
    "Annualised Volatility",
    "Maximum Drawdown",
    "Correlation vs Nifty",
    "Total One-Way Turnover",
    "Annualised Turnover",
]

for column in percentage_columns:

    display_backtest_summary[
        column
    ] = (
        display_backtest_summary[
            column
        ]
        * 100
    )


print("TRANSACTION-COST-ADJUSTED STRATEGY BACKTEST")
print("=" * 72)
print(
    "Backtest period:",
    strategy_daily_net_returns
    .index.min()
    .date(),
    "to",
    strategy_daily_net_returns
    .index.max()
    .date(),
)
print(
    "Trading observations:",
    len(
        strategy_daily_net_returns
    ),
)
print(
    "Portfolio strategies:",
    monthly_strategy_weights[
        "Strategy"
    ].nunique(),
)
print(
    "Benchmark:",
    "Nifty 50",
)
print(
    "Initial investment:",
    f"₹{INITIAL_INVESTMENT_INR:,.0f}",
)
print(
    "One-way transaction cost:",
    f"{ONE_WAY_TRANSACTION_COST:.2%}",
)
print(
    "Rebalance execution:",
    "Next trading-day close",
)
print(
    "Daily portfolio accounting:",
    "PASSED",
)
print(
    "Transaction-cost application:",
    "PASSED",
)
print(
    "Strategy backtest:",
    "PASSED",
)

display(
    display_backtest_summary.round(
        2
    )
)

TRANSACTION-COST-ADJUSTED STRATEGY BACKTEST
Backtest period: 2018-04-02 to 2026-07-30
Trading observations: 2054
Portfolio strategies: 9
Benchmark: Nifty 50
Initial investment: ₹1,000,000
One-way transaction cost: 0.15%
Rebalance execution: Next trading-day close
Daily portfolio accounting: PASSED
Transaction-cost application: PASSED
Strategy backtest: PASSED


,Start Date,End Date,Years,Ending Value (₹),Total Return,CAGR,Annualised Volatility,Sharpe Ratio,Maximum Drawdown,Calmar Ratio,Beta vs Nifty,Correlation vs Nifty,Total One-Way Turnover,Annualised Turnover,Transaction Costs (₹),Rebalances
Strategy,,,,,,,,,,,,,,,,
1-Month Reversal,2018-04-02,2026-07-30,8.33,8460873.01,746.09,29.24,21.54,1.03,-33.56,0.87,0.87,69.34,7205.30,865.42,449638.41,101
Equal-Weight India 10,2018-04-02,2026-07-30,8.33,4816198.61,381.62,20.78,16.70,0.86,-30.47,0.68,0.87,88.76,344.74,41.41,10846.57,101
Ridge Regression,2018-04-02,2026-07-30,8.33,5984914.98,498.49,23.97,21.22,0.84,-29.82,0.80,0.89,72.07,6081.47,730.44,253682.90,101
Linear Regression,2018-04-02,2026-07-30,8.33,5984914.98,498.49,23.97,21.22,0.84,-29.82,0.80,0.89,72.07,6081.47,730.44,253682.90,101
12-Month Momentum,2018-04-02,2026-07-30,8.33,6256936.61,525.69,24.64,22.35,0.84,-30.47,0.81,0.89,68.01,2402.66,288.58,90850.69,101
Gradient Boosting,2018-04-02,2026-07-30,8.33,5868523.31,486.85,23.68,21.14,0.84,-30.39,0.78,0.90,72.66,6152.22,738.93,219676.83,101
Positive Return Logistic,2018-04-02,2026-07-30,8.33,5505116.77,450.51,22.74,20.25,0.82,-31.09,0.73,0.88,74.42,5570.47,669.06,251505.08,101
Random Forest,2018-04-02,2026-07-30,8.33,4420493.52,342.05,19.54,20.88,0.68,-26.33,0.74,0.88,72.25,5723.82,687.48,174612.87,101
Nifty Outperformance Logistic,2018-04-02,2026-07-30,8.33,3671693.38,267.17,16.91,20.57,0.57,-39.67,0.43,0.88,72.95,5904.89,709.23,192415.74,101


## Feature Importance and Model Stability

This section investigates:

- Which features drive Ridge, Logistic, Random Forest and Gradient Boosting models
- Whether feature importance remains stable through time
- How frequently portfolio selections change
- Consecutive-month portfolio overlap
- Target-weight turnover before accounting for natural weight drift

Feature importance is estimated at annual out-of-sample snapshots. Linear and logistic models use standardised coefficients, while tree models use permutation importance on the out-of-sample stock cross-section.

Feature importance should be interpreted as model behaviour, not causal evidence.

In [17]:
# =========================================================
# FEATURE IMPORTANCE AND MODEL-STABILITY ANALYSIS
# =========================================================

from sklearn.base import clone
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import (
    LogisticRegression,
    Ridge,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------
# 1. Create annual leakage-safe model snapshots
# ---------------------------------------------------------

snapshot_schedule = pd.DataFrame(
    {
        "Snapshot Date":
            pd.to_datetime(
                rebalance_schedule.index
            ),

        "Training Cutoff":
            pd.to_datetime(
                rebalance_schedule.to_numpy()
            ),
    }
)

snapshot_schedule[
    "Year"
] = (
    snapshot_schedule[
        "Snapshot Date"
    ].dt.year
)

snapshot_schedule = (
    snapshot_schedule
    .groupby(
        "Year",
        as_index=False,
    )
    .tail(
        1
    )
    .sort_values(
        "Snapshot Date"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 2. Define models for importance analysis
# ---------------------------------------------------------

importance_models = {
    "Ridge Regression":
        {
            "model":
                Pipeline(
                    steps=[
                        (
                            "standard_scaler",
                            StandardScaler(),
                        ),
                        (
                            "model",
                            Ridge(
                                alpha=10.0
                            ),
                        ),
                    ]
                ),

            "target":
                "forward_excess_return_21d",

            "importance_method":
                "Standardised Coefficient",
        },

    "Positive Return Logistic":
        {
            "model":
                Pipeline(
                    steps=[
                        (
                            "standard_scaler",
                            StandardScaler(),
                        ),
                        (
                            "model",
                            LogisticRegression(
                                C=1.0,
                                solver="lbfgs",
                                max_iter=2_000,
                                random_state=RANDOM_SEED,
                            ),
                        ),
                    ]
                ),

            "target":
                "positive_return_target",

            "importance_method":
                "Standardised Coefficient",
        },

    "Nifty Outperformance Logistic":
        {
            "model":
                Pipeline(
                    steps=[
                        (
                            "standard_scaler",
                            StandardScaler(),
                        ),
                        (
                            "model",
                            LogisticRegression(
                                C=1.0,
                                solver="lbfgs",
                                max_iter=2_000,
                                random_state=RANDOM_SEED,
                            ),
                        ),
                    ]
                ),

            "target":
                "outperform_target",

            "importance_method":
                "Standardised Coefficient",
        },

    "Random Forest":
        {
            "model":
                RandomForestRegressor(
                    n_estimators=100,
                    max_depth=6,
                    min_samples_leaf=50,
                    max_features="sqrt",
                    bootstrap=True,
                    random_state=RANDOM_SEED,
                    n_jobs=-1,
                ),

            "target":
                "forward_excess_return_21d",

            "importance_method":
                "Out-of-Sample Permutation Importance",
        },

    "Gradient Boosting":
        {
            "model":
                HistGradientBoostingRegressor(
                    learning_rate=0.05,
                    max_iter=150,
                    max_leaf_nodes=15,
                    max_depth=5,
                    min_samples_leaf=50,
                    l2_regularization=1.0,
                    early_stopping=False,
                    random_state=RANDOM_SEED,
                ),

            "target":
                "forward_excess_return_21d",

            "importance_method":
                "Out-of-Sample Permutation Importance",
        },
}


# ---------------------------------------------------------
# 3. Fit models at each annual snapshot
# ---------------------------------------------------------

importance_records = []

for snapshot_number, snapshot_row in (
    snapshot_schedule.iterrows()
):

    snapshot_date = (
        snapshot_row[
            "Snapshot Date"
        ]
    )

    training_cutoff = (
        snapshot_row[
            "Training Cutoff"
        ]
    )

    training_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            <= training_cutoff
        ]
        .copy()
    )

    prediction_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            == snapshot_date
        ]
        .copy()
    )

    if (
        len(
            prediction_sample
        )
        != len(
            INDIA_10_TICKERS
        )
    ):
        continue

    X_train = (
        training_sample[
            feature_columns
        ]
        .astype(float)
    )

    X_snapshot = (
        prediction_sample[
            feature_columns
        ]
        .astype(float)
    )

    for model_name, model_details in (
        importance_models.items()
    ):

        target_column = (
            model_details[
                "target"
            ]
        )

        fitted_model = clone(
            model_details[
                "model"
            ]
        )

        y_train = (
            training_sample[
                target_column
            ]
        )

        if (
            "Logistic"
            in model_name
        ):

            y_train = (
                y_train.astype(
                    int
                )
            )

            if y_train.nunique() != 2:
                raise RuntimeError(
                    f"{model_name} did not contain "
                    "both classes."
                )

        else:

            y_train = (
                y_train.astype(
                    float
                )
            )

        fitted_model.fit(
            X_train,
            y_train,
        )

        importance_method = (
            model_details[
                "importance_method"
            ]
        )

        if importance_method == (
            "Standardised Coefficient"
        ):

            fitted_estimator = (
                fitted_model.named_steps[
                    "model"
                ]
            )

            raw_importance = np.asarray(
                fitted_estimator.coef_
            ).reshape(
                -1
            )

        else:

            snapshot_target = (
                prediction_sample[
                    "forward_excess_return_21d"
                ]
                .astype(float)
            )

            permutation_result = (
                permutation_importance(
                    estimator=fitted_model,
                    X=X_snapshot,
                    y=snapshot_target,
                    scoring=(
                        "neg_mean_squared_error"
                    ),
                    n_repeats=20,
                    random_state=(
                        RANDOM_SEED
                        + snapshot_number
                    ),
                    n_jobs=-1,
                )
            )

            raw_importance = (
                permutation_result[
                    "importances_mean"
                ]
            )

        for feature, importance_value in zip(
            feature_columns,
            raw_importance,
        ):

            importance_records.append(
                {
                    "Snapshot Date":
                        snapshot_date,

                    "Training Cutoff":
                        training_cutoff,

                    "Model":
                        model_name,

                    "Target":
                        target_column,

                    "Importance Method":
                        importance_method,

                    "Feature":
                        feature,

                    "Raw Importance":
                        float(
                            importance_value
                        ),

                    "Absolute Importance":
                        abs(
                            float(
                                importance_value
                            )
                        ),
                }
            )

    print(
        "Completed feature-importance snapshot:",
        snapshot_date.date(),
    )


feature_importance_snapshots = (
    pd.DataFrame(
        importance_records
    )
)


# ---------------------------------------------------------
# 4. Rank features within each model and snapshot
# ---------------------------------------------------------

feature_importance_snapshots[
    "Importance Rank"
] = (
    feature_importance_snapshots
    .groupby(
        [
            "Snapshot Date",
            "Model",
        ]
    )[
        "Absolute Importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
)

feature_importance_snapshots[
    "Top 5 Feature"
] = (
    feature_importance_snapshots[
        "Importance Rank"
    ]
    <= 5
)


# ---------------------------------------------------------
# 5. Summarise importance across time
# ---------------------------------------------------------

feature_importance_summary = (
    feature_importance_snapshots
    .groupby(
        [
            "Model",
            "Feature",
            "Importance Method",
        ]
    )
    .agg(
        Snapshots=(
            "Snapshot Date",
            "nunique",
        ),

        Mean_Raw_Importance=(
            "Raw Importance",
            "mean",
        ),

        Mean_Absolute_Importance=(
            "Absolute Importance",
            "mean",
        ),

        Median_Absolute_Importance=(
            "Absolute Importance",
            "median",
        ),

        Positive_Importance_Rate=(
            "Raw Importance",
            lambda values: (
                values
                > 0
            ).mean(),
        ),

        Average_Importance_Rank=(
            "Importance Rank",
            "mean",
        ),

        Top_5_Frequency=(
            "Top 5 Feature",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "Model",
            "Mean_Absolute_Importance",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


# ---------------------------------------------------------
# 6. Calculate feature-importance stability through time
# ---------------------------------------------------------

importance_stability_records = []

for model_name, model_importance in (
    feature_importance_snapshots.groupby(
        "Model"
    )
):

    importance_matrix = (
        model_importance
        .pivot(
            index="Snapshot Date",
            columns="Feature",
            values="Raw Importance",
        )
        .sort_index()
    )

    consecutive_rank_correlations = []
    consecutive_top_five_overlap = []

    previous_date = None

    for current_date in (
        importance_matrix.index
    ):

        if previous_date is not None:

            previous_importance = (
                importance_matrix.loc[
                    previous_date
                ]
            )

            current_importance = (
                importance_matrix.loc[
                    current_date
                ]
            )

            rank_correlation = (
                previous_importance.corr(
                    current_importance,
                    method="spearman",
                )
            )

            if pd.notna(
                rank_correlation
            ):

                consecutive_rank_correlations.append(
                    rank_correlation
                )

            previous_top_five = set(
                previous_importance
                .abs()
                .nlargest(
                    5
                )
                .index
            )

            current_top_five = set(
                current_importance
                .abs()
                .nlargest(
                    5
                )
                .index
            )

            top_five_overlap = (
                len(
                    previous_top_five.intersection(
                        current_top_five
                    )
                )
                / 5
            )

            consecutive_top_five_overlap.append(
                top_five_overlap
            )

        previous_date = current_date

    importance_stability_records.append(
        {
            "Model":
                model_name,

            "Snapshots":
                len(
                    importance_matrix
                ),

            "Consecutive Comparisons":
                len(
                    consecutive_top_five_overlap
                ),

            "Mean Consecutive Rank Correlation":
                (
                    np.mean(
                        consecutive_rank_correlations
                    )
                    if consecutive_rank_correlations
                    else np.nan
                ),

            "Median Consecutive Rank Correlation":
                (
                    np.median(
                        consecutive_rank_correlations
                    )
                    if consecutive_rank_correlations
                    else np.nan
                ),

            "Mean Top-5 Feature Overlap":
                (
                    np.mean(
                        consecutive_top_five_overlap
                    )
                    if consecutive_top_five_overlap
                    else np.nan
                ),
        }
    )


model_importance_stability = (
    pd.DataFrame(
        importance_stability_records
    )
    .set_index(
        "Model"
    )
    .sort_values(
        "Mean Consecutive Rank Correlation",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 7. Analyse monthly portfolio-selection stability
# ---------------------------------------------------------

selection_stability_records = []

for strategy_name, strategy_weights in (
    monthly_strategy_weights.groupby(
        "Strategy"
    )
):

    target_weight_matrix = (
        strategy_weights
        .pivot(
            index="Signal Date",
            columns="Ticker",
            values="Target Weight",
        )
        .reindex(
            columns=INDIA_10_TICKERS,
            fill_value=0.0,
        )
        .sort_index()
    )

    selected_baskets = [
        frozenset(
            target_weight_matrix.columns[
                target_weight_matrix
                .loc[
                    signal_date
                ]
                .gt(
                    0
                )
            ]
        )
        for signal_date in (
            target_weight_matrix.index
        )
    ]

    consecutive_jaccard = []
    consecutive_overlap = []

    for previous_basket, current_basket in zip(
        selected_baskets[
            :-1
        ],
        selected_baskets[
            1:
        ],
    ):

        union_size = len(
            previous_basket.union(
                current_basket
            )
        )

        intersection_size = len(
            previous_basket.intersection(
                current_basket
            )
        )

        consecutive_jaccard.append(
            (
                intersection_size
                / union_size
                if union_size > 0
                else np.nan
            )
        )

        consecutive_overlap.append(
            intersection_size
        )

    target_turnover = (
        target_weight_matrix
        .diff()
        .abs()
        .sum(
            axis=1
        )
        / 2
    )

    # The first allocation is from cash and equals 100%.
    target_turnover.iloc[
        0
    ] = 1.0

    stock_selection_rates = (
        target_weight_matrix
        .gt(
            0
        )
        .mean()
        .sort_values(
            ascending=False
        )
    )

    selection_stability_records.append(
        {
            "Strategy":
                strategy_name,

            "Rebalances":
                len(
                    target_weight_matrix
                ),

            "Median Holdings":
                target_weight_matrix
                .gt(
                    0
                )
                .sum(
                    axis=1
                )
                .median(),

            "Average Consecutive Jaccard":
                np.mean(
                    consecutive_jaccard
                ),

            "Median Consecutive Stock Overlap":
                np.median(
                    consecutive_overlap
                ),

            "Unique Selected Baskets":
                len(
                    set(
                        selected_baskets
                    )
                ),

            "Average Monthly Target Turnover":
                target_turnover.iloc[
                    1:
                ].mean(),

            "Annualised Target Turnover":
                (
                    target_turnover.iloc[
                        1:
                    ].mean()
                    * 12
                ),

            "Most Frequently Selected Stock":
                stock_selection_rates.index[
                    0
                ],

            "Top Stock Selection Rate":
                stock_selection_rates.iloc[
                    0
                ],
        }
    )


portfolio_selection_stability = (
    pd.DataFrame(
        selection_stability_records
    )
    .set_index(
        "Strategy"
    )
    .sort_values(
        "Average Consecutive Jaccard",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 8. Save research outputs
# ---------------------------------------------------------

feature_importance_snapshots.to_csv(
    PROCESSED_DATA_DIR
    / "feature_importance_snapshots.csv",
    index=False,
)

feature_importance_summary.to_csv(
    PROCESSED_DATA_DIR
    / "feature_importance_summary.csv",
    index=False,
)

model_importance_stability.to_csv(
    PROCESSED_DATA_DIR
    / "model_importance_stability.csv",
)

portfolio_selection_stability.to_csv(
    PROCESSED_DATA_DIR
    / "portfolio_selection_stability.csv",
)


# ---------------------------------------------------------
# 9. Validate results
# ---------------------------------------------------------

assert (
    snapshot_schedule[
        "Snapshot Date"
    ].nunique()
    >= 8
)

assert (
    feature_importance_snapshots[
        "Model"
    ].nunique()
    == len(
        importance_models
    )
)

assert (
    feature_importance_snapshots[
        "Raw Importance"
    ]
    .notna()
    .all()
)

assert np.isfinite(
    feature_importance_snapshots[
        "Raw Importance"
    ]
).all()

assert (
    portfolio_selection_stability[
        "Average Consecutive Jaccard"
    ]
    .between(
        0,
        1,
    )
    .all()
)

assert (
    portfolio_selection_stability[
        "Top Stock Selection Rate"
    ]
    .between(
        0,
        1,
    )
    .all()
)


# ---------------------------------------------------------
# 10. Display results
# ---------------------------------------------------------

top_features_by_model = (
    feature_importance_summary
    .groupby(
        "Model",
        group_keys=False,
    )
    .head(
        5
    )
    .set_index(
        [
            "Model",
            "Feature",
        ]
    )
)


display_importance_stability = (
    model_importance_stability.copy()
)

for column in [
    "Mean Consecutive Rank Correlation",
    "Median Consecutive Rank Correlation",
    "Mean Top-5 Feature Overlap",
]:

    display_importance_stability[
        column
    ] = (
        display_importance_stability[
            column
        ]
        * 100
    )


display_selection_stability = (
    portfolio_selection_stability.copy()
)

for column in [
    "Average Consecutive Jaccard",
    "Average Monthly Target Turnover",
    "Annualised Target Turnover",
    "Top Stock Selection Rate",
]:

    display_selection_stability[
        column
    ] = (
        display_selection_stability[
            column
        ]
        * 100
    )


print("FEATURE IMPORTANCE AND MODEL STABILITY")
print("=" * 72)
print(
    "Annual snapshots:",
    snapshot_schedule[
        "Snapshot Date"
    ].nunique(),
)
print(
    "Snapshot period:",
    snapshot_schedule[
        "Snapshot Date"
    ].min().date(),
    "to",
    snapshot_schedule[
        "Snapshot Date"
    ].max().date(),
)
print(
    "Models analysed:",
    feature_importance_snapshots[
        "Model"
    ].nunique(),
)
print(
    "Features per model:",
    len(
        feature_columns
    ),
)
print(
    "Portfolio strategies analysed:",
    len(
        portfolio_selection_stability
    ),
)
print(
    "Feature-importance validation:",
    "PASSED",
)
print(
    "Portfolio-stability validation:",
    "PASSED",
)

print("\nTOP FIVE FEATURES BY MODEL")
display(
    top_features_by_model.round(
        4
    )
)

print("\nMODEL IMPORTANCE STABILITY")
display(
    display_importance_stability.round(
        2
    )
)

print("\nPORTFOLIO SELECTION STABILITY")
display(
    display_selection_stability.round(
        2
    )
)

Completed feature-importance snapshot: 2018-12-31
Completed feature-importance snapshot: 2019-12-31
Completed feature-importance snapshot: 2020-12-31
Completed feature-importance snapshot: 2021-12-31
Completed feature-importance snapshot: 2022-12-30
Completed feature-importance snapshot: 2023-12-29
Completed feature-importance snapshot: 2024-12-31
Completed feature-importance snapshot: 2025-12-31
Completed feature-importance snapshot: 2026-07-01
FEATURE IMPORTANCE AND MODEL STABILITY
Annual snapshots: 9
Snapshot period: 2018-12-31 to 2026-07-01
Models analysed: 5
Features per model: 27
Portfolio strategies analysed: 9
Feature-importance validation: PASSED
Portfolio-stability validation: PASSED

TOP FIVE FEATURES BY MODEL


Importance Method  \
Model                         Feature                                                           
Gradient Boosting             momentum_252d              Out-of-Sample Permutation Importance   
                              relative_strength_126d     Out-of-Sample Permutation Importance   
                              benchmark_correlation_63d  Out-of-Sample Permutation Importance   
                              volatility_63d             Out-of-Sample Permutation Importance   
                              drawdown_252d              Out-of-Sample Permutation Importance   
Nifty Outperformance Logistic benchmark_volatility_63d               Standardised Coefficient   
                              volatility_63d                         Standardised Coefficient   
                              momentum_21d                           Standardised Coefficient   
                              benchmark_drawdown_252d                Standardised Coefficient   
                              volatility_21d                         Standardised Coefficient   
Positive Return Logistic      benchmark_ma_gap_200d                  Standardised Coefficient   
                              benchmark_drawdown_252d                Standardised Coefficient   
                              ma_gap_63d                             Standardised Coefficient   
                              benchmark_momentum_21d                 Standardised Coefficient   
                              ma_gap_200d                            Standardised Coefficient   
Random Forest                 momentum_126d              Out-of-Sample Permutation Importance   
                              momentum_252d              Out-of-Sample Permutation Importance   
                              relative_strength_126d     Out-of-Sample Permutation Importance   
                              volatility_63d             Out-of-Sample Permutation Importance   
                              ma_gap_21d                 Out-of-Sample Permutation Importance   
Ridge Regression              volatility_63d                         Standardised Coefficient   
                              benchmark_drawdown_252d                Standardised Coefficient   
                              ma_gap_200d                            Standardised Coefficient   
                              volatility_21d                         Standardised Coefficient   
                              momentum_21d                           Standardised Coefficient   

                                                         Snapshots  \
Model                         Feature                                
Gradient Boosting             momentum_252d                      9   
                              relative_strength_126d             9   
                              benchmark_correlation_63d          9   
                              volatility_63d                     9   
                              drawdown_252d                      9   
Nifty Outperformance Logistic benchmark_volatility_63d           9   
                              volatility_63d                     9   
                              momentum_21d                       9   
                              benchmark_drawdown_252d            9   
                              volatility_21d                     9   
Positive Return Logistic      benchmark_ma_gap_200d              9   
                              benchmark_drawdown_252d            9   
                              ma_gap_63d                         9   
                              benchmark_momentum_21d             9   
                              ma_gap_200d                        9   
Random Forest                 momentum_126d                      9   
                              momentum_252d                      9   
                              relative_strength_126d             9   
                              volatility_63d            


MODEL IMPORTANCE STABILITY


,Snapshots,Consecutive Comparisons,Mean Consecutive Rank Correlation,Median Consecutive Rank Correlation,Mean Top-5 Feature Overlap
Model,,,,,
Ridge Regression,9,8,83.75,89.77,62.5
Nifty Outperformance Logistic,9,8,80.21,88.86,65.0
Positive Return Logistic,9,8,79.87,88.10,70.0
Gradient Boosting,9,8,7.06,9.10,42.5
Random Forest,9,8,-4.27,-13.49,37.5



PORTFOLIO SELECTION STABILITY


,Rebalances,Median Holdings,Average Consecutive Jaccard,Median Consecutive Stock Overlap,Unique Selected Baskets,Average Monthly Target Turnover,Annualised Target Turnover,Most Frequently Selected Stock,Top Stock Selection Rate
Strategy,,,,,,,,,
Equal-Weight India 10,101,10.0,100.0,10.0,1,0.00,0.0,HDFCBANK.NS,100.00
12-Month Momentum,101,3.0,68.7,2.0,37,21.67,260.0,BEL.NS,64.36
Positive Return Logistic,101,3.0,34.1,1.0,61,53.33,640.0,HINDUNILVR.NS,48.51
Random Forest,101,3.0,34.0,1.0,58,55.00,660.0,HINDUNILVR.NS,40.59
Nifty Outperformance Logistic,101,3.0,32.2,1.0,57,56.67,680.0,HINDUNILVR.NS,49.50
Ridge Regression,101,3.0,29.5,1.0,61,58.33,700.0,TRENT.NS,39.60
Linear Regression,101,3.0,29.5,1.0,61,58.33,700.0,TRENT.NS,39.60
Gradient Boosting,101,3.0,29.1,1.0,66,59.33,712.0,HINDUNILVR.NS,38.61
1-Month Reversal,101,3.0,21.1,1.0,67,69.33,832.0,HINDUNILVR.NS,37.62


## Market-Regime Performance Analysis

Evaluate every strategy across four Nifty 50 market environments:

1. Bull market with lower volatility
2. Bull market with higher volatility
3. Bear market with lower volatility
4. Bear market with higher volatility

The market trend is determined using the Nifty 50 versus its 200-day moving average.

The volatility threshold uses an expanding historical median of 63-day annualised volatility, shifted by one trading day. This prevents future volatility information from influencing the regime classification.

Regime results are conditional historical analysis and do not imply that future regimes can be predicted accurately.

In [18]:
# =========================================================
# MARKET-REGIME PERFORMANCE ANALYSIS
# =========================================================

# ---------------------------------------------------------
# 1. Build leakage-safe Nifty 50 regime indicators
# ---------------------------------------------------------

full_benchmark_price = (
    close_prices[
        BENCHMARK_TICKER
    ]
    .dropna()
    .copy()
)

full_benchmark_return = (
    full_benchmark_price
    .pct_change(
        fill_method=None
    )
)

benchmark_ma_200d = (
    full_benchmark_price
    .rolling(
        window=200,
        min_periods=200,
    )
    .mean()
)

benchmark_volatility_63d = (
    full_benchmark_return
    .rolling(
        window=63,
        min_periods=63,
    )
    .std()
    * np.sqrt(
        TRADING_DAYS_PER_YEAR
    )
)

# The volatility threshold uses only information available
# before the current trading date.
historical_volatility_threshold = (
    benchmark_volatility_63d
    .expanding(
        min_periods=252
    )
    .median()
    .shift(
        1
    )
)

market_regime_frame = pd.DataFrame(
    {
        "Nifty Close":
            full_benchmark_price,

        "Nifty 200D MA":
            benchmark_ma_200d,

        "Nifty 63D Volatility":
            benchmark_volatility_63d,

        "Historical Volatility Threshold":
            historical_volatility_threshold,
    }
)

market_regime_frame[
    "Bull Market"
] = (
    market_regime_frame[
        "Nifty Close"
    ]
    >= market_regime_frame[
        "Nifty 200D MA"
    ]
)

market_regime_frame[
    "High Volatility"
] = (
    market_regime_frame[
        "Nifty 63D Volatility"
    ]
    > market_regime_frame[
        "Historical Volatility Threshold"
    ]
)

valid_regime_data = (
    market_regime_frame[
        [
            "Nifty 200D MA",
            "Nifty 63D Volatility",
            "Historical Volatility Threshold",
        ]
    ]
    .notna()
    .all(
        axis=1
    )
)

market_regime_frame[
    "Regime"
] = np.select(
    condlist=[
        (
            valid_regime_data
            & market_regime_frame[
                "Bull Market"
            ]
            & ~market_regime_frame[
                "High Volatility"
            ]
        ),

        (
            valid_regime_data
            & market_regime_frame[
                "Bull Market"
            ]
            & market_regime_frame[
                "High Volatility"
            ]
        ),

        (
            valid_regime_data
            & ~market_regime_frame[
                "Bull Market"
            ]
            & ~market_regime_frame[
                "High Volatility"
            ]
        ),

        (
            valid_regime_data
            & ~market_regime_frame[
                "Bull Market"
            ]
            & market_regime_frame[
                "High Volatility"
            ]
        ),
    ],

    choicelist=[
        "Bull / Lower Volatility",
        "Bull / Higher Volatility",
        "Bear / Lower Volatility",
        "Bear / Higher Volatility",
    ],

    default=None,
)


# ---------------------------------------------------------
# 2. Align regimes with the strategy backtest
# ---------------------------------------------------------

backtest_regime_frame = (
    market_regime_frame
    .reindex(
        strategy_daily_net_returns.index
    )
    .dropna(
        subset=[
            "Regime"
        ]
    )
    .copy()
)

regime_order = [
    "Bull / Lower Volatility",
    "Bull / Higher Volatility",
    "Bear / Lower Volatility",
    "Bear / Higher Volatility",
]

backtest_regime_frame[
    "Regime"
] = pd.Categorical(
    backtest_regime_frame[
        "Regime"
    ],
    categories=regime_order,
    ordered=True,
)

regime_day_counts = (
    backtest_regime_frame[
        "Regime"
    ]
    .value_counts(
        sort=False
    )
    .rename(
        "Trading Days"
    )
    .to_frame()
)

regime_day_counts[
    "Percentage of Classified Days"
] = (
    regime_day_counts[
        "Trading Days"
    ]
    / regime_day_counts[
        "Trading Days"
    ].sum()
)


# ---------------------------------------------------------
# 3. Calculate strategy statistics within each regime
# ---------------------------------------------------------

daily_risk_free_rate = (
    (
        1
        + RISK_FREE_RATE
    )
    ** (
        1
        / TRADING_DAYS_PER_YEAR
    )
    - 1
)

regime_performance_records = []

for regime_name in regime_order:

    regime_dates = (
        backtest_regime_frame.index[
            backtest_regime_frame[
                "Regime"
            ]
            == regime_name
        ]
    )

    for strategy_name in (
        strategy_daily_net_returns.columns
    ):

        regime_returns = (
            strategy_daily_net_returns
            .loc[
                regime_dates,
                strategy_name,
            ]
            .dropna()
        )

        observations = len(
            regime_returns
        )

        if observations == 0:
            continue

        compounded_return = (
            (
                1
                + regime_returns
            )
            .prod()
            - 1
        )

        annualised_return = (
            (
                1
                + compounded_return
            )
            ** (
                TRADING_DAYS_PER_YEAR
                / observations
            )
            - 1
        )

        annualised_volatility = (
            regime_returns.std(
                ddof=1
            )
            * np.sqrt(
                TRADING_DAYS_PER_YEAR
            )
        )

        annualised_excess_return = (
            (
                regime_returns.mean()
                - daily_risk_free_rate
            )
            * TRADING_DAYS_PER_YEAR
        )

        sharpe_ratio = (
            annualised_excess_return
            / annualised_volatility
            if annualised_volatility > 0
            else np.nan
        )

        downside_returns = np.minimum(
            regime_returns
            - daily_risk_free_rate,
            0,
        )

        downside_deviation = (
            pd.Series(
                downside_returns,
                index=regime_returns.index,
            )
            .std(
                ddof=1
            )
            * np.sqrt(
                TRADING_DAYS_PER_YEAR
            )
        )

        sortino_ratio = (
            annualised_excess_return
            / downside_deviation
            if downside_deviation > 0
            else np.nan
        )

        regime_performance_records.append(
            {
                "Regime":
                    regime_name,

                "Strategy":
                    strategy_name,

                "Trading Days":
                    observations,

                "Compounded Return":
                    compounded_return,

                "Annualised Return":
                    annualised_return,

                "Annualised Volatility":
                    annualised_volatility,

                "Sharpe Ratio":
                    sharpe_ratio,

                "Sortino Ratio":
                    sortino_ratio,

                "Positive Day Rate":
                    regime_returns
                    .gt(
                        0
                    )
                    .mean(),

                "Worst Daily Return":
                    regime_returns.min(),

                "Best Daily Return":
                    regime_returns.max(),
            }
        )


strategy_regime_performance = (
    pd.DataFrame(
        regime_performance_records
    )
    .sort_values(
        [
            "Regime",
            "Sharpe Ratio",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 4. Identify regime leaders
# ---------------------------------------------------------

regime_leader_records = []

for regime_name, regime_results in (
    strategy_regime_performance.groupby(
        "Regime",
        observed=True,
    )
):

    best_sharpe_row = (
        regime_results.loc[
            regime_results[
                "Sharpe Ratio"
            ].idxmax()
        ]
    )

    best_return_row = (
        regime_results.loc[
            regime_results[
                "Annualised Return"
            ].idxmax()
        ]
    )

    lowest_volatility_row = (
        regime_results.loc[
            regime_results[
                "Annualised Volatility"
            ].idxmin()
        ]
    )

    regime_leader_records.append(
        {
            "Regime":
                regime_name,

            "Best Sharpe Strategy":
                best_sharpe_row[
                    "Strategy"
                ],

            "Best Sharpe Ratio":
                best_sharpe_row[
                    "Sharpe Ratio"
                ],

            "Best Return Strategy":
                best_return_row[
                    "Strategy"
                ],

            "Best Annualised Return":
                best_return_row[
                    "Annualised Return"
                ],

            "Lowest Volatility Strategy":
                lowest_volatility_row[
                    "Strategy"
                ],

            "Lowest Annualised Volatility":
                lowest_volatility_row[
                    "Annualised Volatility"
                ],
        }
    )


regime_leaders = (
    pd.DataFrame(
        regime_leader_records
    )
    .set_index(
        "Regime"
    )
    .reindex(
        regime_order
    )
)


# ---------------------------------------------------------
# 5. Measure strategy stability across regimes
# ---------------------------------------------------------

strategy_regime_stability = (
    strategy_regime_performance
    .groupby(
        "Strategy"
    )
    .agg(
        Regimes=(
            "Regime",
            "nunique",
        ),

        Average_Regime_Return=(
            "Annualised Return",
            "mean",
        ),

        Worst_Regime_Return=(
            "Annualised Return",
            "min",
        ),

        Best_Regime_Return=(
            "Annualised Return",
            "max",
        ),

        Average_Regime_Sharpe=(
            "Sharpe Ratio",
            "mean",
        ),

        Worst_Regime_Sharpe=(
            "Sharpe Ratio",
            "min",
        ),

        Best_Regime_Sharpe=(
            "Sharpe Ratio",
            "max",
        ),

        Positive_Sharpe_Regimes=(
            "Sharpe Ratio",
            lambda values: (
                values
                > 0
            ).sum(),
        ),

        Average_Regime_Volatility=(
            "Annualised Volatility",
            "mean",
        ),
    )
)

strategy_regime_stability[
    "Sharpe Range"
] = (
    strategy_regime_stability[
        "Best_Regime_Sharpe"
    ]
    - strategy_regime_stability[
        "Worst_Regime_Sharpe"
    ]
)

strategy_regime_stability[
    "Return Range"
] = (
    strategy_regime_stability[
        "Best_Regime_Return"
    ]
    - strategy_regime_stability[
        "Worst_Regime_Return"
    ]
)

strategy_regime_stability = (
    strategy_regime_stability
    .sort_values(
        [
            "Worst_Regime_Sharpe",
            "Average_Regime_Sharpe",
        ],
        ascending=False,
    )
)


# ---------------------------------------------------------
# 6. Create Sharpe and return comparison tables
# ---------------------------------------------------------

regime_sharpe_pivot = (
    strategy_regime_performance
    .pivot(
        index="Strategy",
        columns="Regime",
        values="Sharpe Ratio",
    )
    .reindex(
        columns=regime_order
    )
)

regime_return_pivot = (
    strategy_regime_performance
    .pivot(
        index="Strategy",
        columns="Regime",
        values="Annualised Return",
    )
    .reindex(
        columns=regime_order
    )
)


# ---------------------------------------------------------
# 7. Save regime-analysis outputs
# ---------------------------------------------------------

backtest_regime_frame.to_csv(
    PROCESSED_DATA_DIR
    / "market_regime_daily_labels.csv",
    index_label="Date",
)

regime_day_counts.to_csv(
    PROCESSED_DATA_DIR
    / "market_regime_day_counts.csv",
)

strategy_regime_performance.to_csv(
    PROCESSED_DATA_DIR
    / "strategy_regime_performance.csv",
    index=False,
)

regime_leaders.to_csv(
    PROCESSED_DATA_DIR
    / "regime_leaders.csv",
)

strategy_regime_stability.to_csv(
    PROCESSED_DATA_DIR
    / "strategy_regime_stability.csv",
)

regime_sharpe_pivot.to_csv(
    PROCESSED_DATA_DIR
    / "strategy_regime_sharpe_pivot.csv",
)

regime_return_pivot.to_csv(
    PROCESSED_DATA_DIR
    / "strategy_regime_return_pivot.csv",
)


# ---------------------------------------------------------
# 8. Validate regime analysis
# ---------------------------------------------------------

assert (
    backtest_regime_frame[
        "Regime"
    ]
    .notna()
    .all()
)

assert set(
    backtest_regime_frame[
        "Regime"
    ].astype(str).unique()
) == set(
    regime_order
)

assert (
    regime_day_counts[
        "Trading Days"
    ]
    .gt(
        20
    )
    .all()
)

assert (
    strategy_regime_performance[
        "Strategy"
    ].nunique()
    == strategy_daily_net_returns.shape[
        1
    ]
)

assert (
    strategy_regime_performance[
        "Regime"
    ].nunique()
    == 4
)

assert (
    strategy_regime_stability[
        "Regimes"
    ]
    .eq(
        4
    )
    .all()
)

assert np.isfinite(
    strategy_regime_performance[
        [
            "Annualised Return",
            "Annualised Volatility",
            "Sharpe Ratio",
        ]
    ]
    .to_numpy()
).all()


# ---------------------------------------------------------
# 9. Display results
# ---------------------------------------------------------

display_regime_counts = (
    regime_day_counts.copy()
)

display_regime_counts[
    "Percentage of Classified Days"
] = (
    display_regime_counts[
        "Percentage of Classified Days"
    ]
    * 100
)


display_regime_leaders = (
    regime_leaders.copy()
)

for column in [
    "Best Annualised Return",
    "Lowest Annualised Volatility",
]:

    display_regime_leaders[
        column
    ] = (
        display_regime_leaders[
            column
        ]
        * 100
    )


display_regime_stability = (
    strategy_regime_stability.copy()
)

for column in [
    "Average_Regime_Return",
    "Worst_Regime_Return",
    "Best_Regime_Return",
    "Average_Regime_Volatility",
    "Return Range",
]:

    display_regime_stability[
        column
    ] = (
        display_regime_stability[
            column
        ]
        * 100
    )


print("MARKET-REGIME PERFORMANCE ANALYSIS")
print("=" * 72)
print(
    "Classified period:",
    backtest_regime_frame
    .index.min()
    .date(),
    "to",
    backtest_regime_frame
    .index.max()
    .date(),
)
print(
    "Classified trading days:",
    len(
        backtest_regime_frame
    ),
)
print(
    "Regimes:",
    backtest_regime_frame[
        "Regime"
    ].nunique(),
)
print(
    "Strategies analysed:",
    strategy_regime_performance[
        "Strategy"
    ].nunique(),
)
print(
    "Trend indicator:",
    "Nifty 50 200-day moving average",
)
print(
    "Volatility threshold:",
    "Shifted expanding historical median",
)
print(
    "Leakage-safe regime construction:",
    "PASSED",
)
print(
    "Regime analysis:",
    "PASSED",
)

print("\nREGIME DAY COUNTS")
display(
    display_regime_counts.round(
        2
    )
)

print("\nREGIME LEADERS")
display(
    display_regime_leaders.round(
        2
    )
)

print("\nSTRATEGY STABILITY ACROSS REGIMES")
display(
    display_regime_stability.round(
        2
    )
)

MARKET-REGIME PERFORMANCE ANALYSIS
Classified period: 2018-04-02 to 2026-07-30
Classified trading days: 2054
Regimes: 4
Strategies analysed: 10
Trend indicator: Nifty 50 200-day moving average
Volatility threshold: Shifted expanding historical median
Leakage-safe regime construction: PASSED
Regime analysis: PASSED

REGIME DAY COUNTS


,Trading Days,Percentage of Classified Days
Regime,,
Bull / Lower Volatility,908,44.21
Bull / Higher Volatility,654,31.84
Bear / Lower Volatility,120,5.84
Bear / Higher Volatility,372,18.11



REGIME LEADERS


,Best Sharpe Strategy,Best Sharpe Ratio,Best Return Strategy,Best Annualised Return,Lowest Volatility Strategy,Lowest Annualised Volatility
Regime,,,,,,
Bull / Lower Volatility,12-Month Momentum,1.58,12-Month Momentum,39.09,Nifty 50,11.21
Bull / Higher Volatility,1-Month Reversal,2.95,1-Month Reversal,86.79,Nifty 50,15.52
Bear / Lower Volatility,Gradient Boosting,-1.18,Gradient Boosting,-16.16,Nifty 50,12.10
Bear / Higher Volatility,Random Forest,0.15,Random Forest,6.27,Equal-Weight India 10,25.70



STRATEGY STABILITY ACROSS REGIMES


,Regimes,Average_Regime_Return,Worst_Regime_Return,Best_Regime_Return,Average_Regime_Sharpe,Worst_Regime_Sharpe,Best_Regime_Sharpe,Positive_Sharpe_Regimes,Average_Regime_Volatility,Sharpe Range,Return Range
Strategy,,,,,,,,,,,
Gradient Boosting,4,16.24,-16.16,57.11,0.40,-1.18,1.97,3,21.61,3.15,73.27
1-Month Reversal,4,22.37,-18.72,86.79,0.56,-1.40,2.95,3,21.80,4.35,105.51
Positive Return Logistic,4,14.31,-17.32,53.36,0.37,-1.44,2.06,2,20.50,3.49,70.68
Linear Regression,4,12.49,-30.44,60.19,0.16,-2.38,2.17,2,21.38,4.55,90.63
Ridge Regression,4,12.49,-30.44,60.19,0.16,-2.38,2.17,2,21.38,4.55,90.63
Random Forest,4,9.09,-33.58,44.14,0.03,-2.44,1.61,3,21.41,4.05,77.72
Equal-Weight India 10,4,11.00,-26.31,51.54,0.10,-2.76,2.30,2,16.75,5.06,77.85
12-Month Momentum,4,8.71,-48.83,47.04,-0.00,-3.02,1.58,2,23.46,4.59,95.87
Nifty Outperformance Logistic,4,5.53,-39.64,52.16,-0.21,-3.23,2.00,2,20.83,5.23,91.79


## Transaction-Cost Sensitivity Analysis

High-turnover strategies may appear attractive before realistic implementation costs.

This section repeats the complete daily backtest under one-way transaction costs of:

- 0 basis points
- 5 basis points
- 15 basis points — base assumption
- 30 basis points
- 50 basis points
- 100 basis points — severe-cost scenario

The analysis evaluates the effect on ending value, CAGR, Sharpe ratio, maximum drawdown and total transaction costs.

The Nifty 50 benchmark remains cost-free. Results are modelling estimates and do not represent an exact brokerage, tax or market-impact calculation for every investor.

In [19]:
# =========================================================
# TRANSACTION-COST SENSITIVITY ANALYSIS
# =========================================================

required_objects = [
    "monthly_strategy_weights",
    "asset_daily_returns",
    "nifty_daily_returns",
    "backtest_monthly_target_weights",
    "calculate_backtest_statistics",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required previous-step objects are missing:\n"
        + "\n".join(
            missing_objects
        )
    )


# ---------------------------------------------------------
# 1. Define one-way transaction-cost scenarios
# ---------------------------------------------------------

transaction_cost_scenarios = [
    {
        "Cost Scenario":
            "0 bps",

        "One-Way Cost":
            0.0000,
    },

    {
        "Cost Scenario":
            "5 bps",

        "One-Way Cost":
            0.0005,
    },

    {
        "Cost Scenario":
            "15 bps — Base",

        "One-Way Cost":
            0.0015,
    },

    {
        "Cost Scenario":
            "30 bps",

        "One-Way Cost":
            0.0030,
    },

    {
        "Cost Scenario":
            "50 bps",

        "One-Way Cost":
            0.0050,
    },

    {
        "Cost Scenario":
            "100 bps",

        "One-Way Cost":
            0.0100,
    },
]


# ---------------------------------------------------------
# 2. Repeat every strategy backtest under every cost
# ---------------------------------------------------------

cost_sensitivity_records = []

for scenario in transaction_cost_scenarios:

    cost_scenario = (
        scenario[
            "Cost Scenario"
        ]
    )

    one_way_cost = (
        scenario[
            "One-Way Cost"
        ]
    )

    for strategy_name, strategy_weights in (
        monthly_strategy_weights.groupby(
            "Strategy"
        )
    ):

        strategy_result = (
            backtest_monthly_target_weights(
                strategy_weights=(
                    strategy_weights
                ),

                daily_asset_returns=(
                    asset_daily_returns
                ),

                initial_investment_inr=(
                    INITIAL_INVESTMENT_INR
                ),

                one_way_transaction_cost=(
                    one_way_cost
                ),
            )
        )

        aligned_nifty_returns = (
            nifty_daily_returns
            .reindex(
                strategy_result.index
            )
            .copy()
        )

        if aligned_nifty_returns.isna().any():
            raise RuntimeError(
                "Missing Nifty 50 returns in the "
                "transaction-cost test."
            )

        aligned_nifty_returns.iloc[
            0
        ] = 0.0

        statistics = (
            calculate_backtest_statistics(
                strategy_name=(
                    strategy_name
                ),

                daily_returns=(
                    strategy_result[
                        "Net Return"
                    ]
                ),

                portfolio_values=(
                    strategy_result[
                        "Portfolio Value (₹)"
                    ]
                ),

                benchmark_returns=(
                    aligned_nifty_returns
                ),

                daily_turnover=(
                    strategy_result[
                        "One-Way Turnover"
                    ]
                ),

                daily_costs=(
                    strategy_result[
                        "Transaction Cost (₹)"
                    ]
                ),
            )
        )

        statistics[
            "Cost Scenario"
        ] = cost_scenario

        statistics[
            "One-Way Cost"
        ] = one_way_cost

        statistics[
            "Cost (bps)"
        ] = (
            one_way_cost
            * 10_000
        )

        cost_sensitivity_records.append(
            statistics
        )

    # Add the cost-free Nifty 50 comparison.
    first_strategy_result = next(
        iter(
            strategy_result
            for strategy_result in [
                strategy_result
            ]
        )
    )

    benchmark_index = (
        first_strategy_result.index
    )

    benchmark_returns = (
        nifty_daily_returns
        .reindex(
            benchmark_index
        )
        .copy()
    )

    benchmark_returns.iloc[
        0
    ] = 0.0

    benchmark_values = (
        INITIAL_INVESTMENT_INR
        * (
            1
            + benchmark_returns
        )
        .cumprod()
    )

    zero_turnover = pd.Series(
        0.0,
        index=benchmark_index,
    )

    zero_costs = pd.Series(
        0.0,
        index=benchmark_index,
    )

    benchmark_statistics = (
        calculate_backtest_statistics(
            strategy_name="Nifty 50",

            daily_returns=(
                benchmark_returns
            ),

            portfolio_values=(
                benchmark_values
            ),

            benchmark_returns=(
                benchmark_returns
            ),

            daily_turnover=(
                zero_turnover
            ),

            daily_costs=(
                zero_costs
            ),
        )
    )

    benchmark_statistics[
        "Cost Scenario"
    ] = cost_scenario

    benchmark_statistics[
        "One-Way Cost"
    ] = one_way_cost

    benchmark_statistics[
        "Cost (bps)"
    ] = (
        one_way_cost
        * 10_000
    )

    cost_sensitivity_records.append(
        benchmark_statistics
    )


transaction_cost_sensitivity = (
    pd.DataFrame(
        cost_sensitivity_records
    )
    .sort_values(
        [
            "One-Way Cost",
            "Sharpe Ratio",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 3. Add performance advantage versus benchmarks
# ---------------------------------------------------------

equal_weight_cagr_map = (
    transaction_cost_sensitivity.loc[
        transaction_cost_sensitivity[
            "Strategy"
        ]
        == "Equal-Weight India 10"
    ]
    .set_index(
        "One-Way Cost"
    )[
        "CAGR"
    ]
    .to_dict()
)

nifty_cagr_map = (
    transaction_cost_sensitivity.loc[
        transaction_cost_sensitivity[
            "Strategy"
        ]
        == "Nifty 50"
    ]
    .set_index(
        "One-Way Cost"
    )[
        "CAGR"
    ]
    .to_dict()
)

transaction_cost_sensitivity[
    "CAGR Advantage vs Equal Weight"
] = (
    transaction_cost_sensitivity[
        "CAGR"
    ]
    - transaction_cost_sensitivity[
        "One-Way Cost"
    ]
    .map(
        equal_weight_cagr_map
    )
)

transaction_cost_sensitivity[
    "CAGR Advantage vs Nifty"
] = (
    transaction_cost_sensitivity[
        "CAGR"
    ]
    - transaction_cost_sensitivity[
        "One-Way Cost"
    ]
    .map(
        nifty_cagr_map
    )
)


# ---------------------------------------------------------
# 4. Create cost-sensitivity comparison tables
# ---------------------------------------------------------

cost_scenario_order = [
    scenario[
        "Cost Scenario"
    ]
    for scenario in transaction_cost_scenarios
]

cost_cagr_pivot = (
    transaction_cost_sensitivity
    .pivot(
        index="Strategy",
        columns="Cost Scenario",
        values="CAGR",
    )
    .reindex(
        columns=cost_scenario_order
    )
)

cost_sharpe_pivot = (
    transaction_cost_sensitivity
    .pivot(
        index="Strategy",
        columns="Cost Scenario",
        values="Sharpe Ratio",
    )
    .reindex(
        columns=cost_scenario_order
    )
)

cost_ending_value_pivot = (
    transaction_cost_sensitivity
    .pivot(
        index="Strategy",
        columns="Cost Scenario",
        values="Ending Value (₹)",
    )
    .reindex(
        columns=cost_scenario_order
    )
)


# ---------------------------------------------------------
# 5. Summarise base and severe-cost robustness
# ---------------------------------------------------------

base_cost_results = (
    transaction_cost_sensitivity.loc[
        np.isclose(
            transaction_cost_sensitivity[
                "One-Way Cost"
            ],
            0.0015,
        )
    ]
    .set_index(
        "Strategy"
    )
)

severe_cost_results = (
    transaction_cost_sensitivity.loc[
        np.isclose(
            transaction_cost_sensitivity[
                "One-Way Cost"
            ],
            0.0100,
        )
    ]
    .set_index(
        "Strategy"
    )
)

zero_cost_results = (
    transaction_cost_sensitivity.loc[
        np.isclose(
            transaction_cost_sensitivity[
                "One-Way Cost"
            ],
            0.0000,
        )
    ]
    .set_index(
        "Strategy"
    )
)

cost_robustness_summary = pd.DataFrame(
    index=base_cost_results.index
)

cost_robustness_summary[
    "Zero-Cost CAGR"
] = (
    zero_cost_results[
        "CAGR"
    ]
)

cost_robustness_summary[
    "Base-Cost CAGR"
] = (
    base_cost_results[
        "CAGR"
    ]
)

cost_robustness_summary[
    "100bps CAGR"
] = (
    severe_cost_results[
        "CAGR"
    ]
)

cost_robustness_summary[
    "CAGR Drag: 0 to 100bps"
] = (
    zero_cost_results[
        "CAGR"
    ]
    - severe_cost_results[
        "CAGR"
    ]
)

cost_robustness_summary[
    "Base-Cost Sharpe"
] = (
    base_cost_results[
        "Sharpe Ratio"
    ]
)

cost_robustness_summary[
    "100bps Sharpe"
] = (
    severe_cost_results[
        "Sharpe Ratio"
    ]
)

cost_robustness_summary[
    "Annualised Turnover"
] = (
    base_cost_results[
        "Annualised Turnover"
    ]
)

cost_robustness_summary[
    "Base Transaction Costs (₹)"
] = (
    base_cost_results[
        "Transaction Costs (₹)"
    ]
)

cost_robustness_summary[
    "100bps Transaction Costs (₹)"
] = (
    severe_cost_results[
        "Transaction Costs (₹)"
    ]
)

cost_robustness_summary[
    "Beats Equal Weight at 100bps"
] = (
    severe_cost_results[
        "CAGR"
    ]
    > severe_cost_results.loc[
        "Equal-Weight India 10",
        "CAGR",
    ]
)

cost_robustness_summary[
    "Beats Nifty at 100bps"
] = (
    severe_cost_results[
        "CAGR"
    ]
    > severe_cost_results.loc[
        "Nifty 50",
        "CAGR",
    ]
)

cost_robustness_summary = (
    cost_robustness_summary
    .sort_values(
        "100bps CAGR",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 6. Identify the best strategy under each cost scenario
# ---------------------------------------------------------

cost_leader_records = []

for cost_scenario, scenario_results in (
    transaction_cost_sensitivity.groupby(
        "Cost Scenario",
        sort=False,
    )
):

    best_cagr_row = (
        scenario_results.loc[
            scenario_results[
                "CAGR"
            ].idxmax()
        ]
    )

    best_sharpe_row = (
        scenario_results.loc[
            scenario_results[
                "Sharpe Ratio"
            ].idxmax()
        ]
    )

    cost_leader_records.append(
        {
            "Cost Scenario":
                cost_scenario,

            "Best CAGR Strategy":
                best_cagr_row[
                    "Strategy"
                ],

            "Best CAGR":
                best_cagr_row[
                    "CAGR"
                ],

            "Best Sharpe Strategy":
                best_sharpe_row[
                    "Strategy"
                ],

            "Best Sharpe Ratio":
                best_sharpe_row[
                    "Sharpe Ratio"
                ],
        }
    )


transaction_cost_leaders = (
    pd.DataFrame(
        cost_leader_records
    )
    .set_index(
        "Cost Scenario"
    )
    .reindex(
        cost_scenario_order
    )
)


# ---------------------------------------------------------
# 7. Validate monotonic cost effects
# ---------------------------------------------------------

non_benchmark_ending_values = (
    cost_ending_value_pivot.drop(
        index="Nifty 50"
    )
)

ending_value_differences = (
    non_benchmark_ending_values.diff(
        axis=1
    )
    .iloc[
        :,
        1:
    ]
)

assert (
    ending_value_differences
    <= 1e-6
).all().all()

assert (
    transaction_cost_sensitivity[
        "Ending Value (₹)"
    ]
    .gt(
        0
    )
    .all()
)

assert (
    transaction_cost_sensitivity[
        "CAGR"
    ]
    .notna()
    .all()
)

assert (
    transaction_cost_sensitivity[
        "Sharpe Ratio"
    ]
    .notna()
    .all()
)

assert (
    len(
        transaction_cost_sensitivity
    )
    == (
        len(
            transaction_cost_scenarios
        )
        * (
            monthly_strategy_weights[
                "Strategy"
            ].nunique()
            + 1
        )
    )
)


# ---------------------------------------------------------
# 8. Save robustness outputs
# ---------------------------------------------------------

transaction_cost_sensitivity.to_csv(
    PROCESSED_DATA_DIR
    / "transaction_cost_sensitivity.csv",
    index=False,
)

cost_cagr_pivot.to_csv(
    PROCESSED_DATA_DIR
    / "transaction_cost_cagr_pivot.csv",
)

cost_sharpe_pivot.to_csv(
    PROCESSED_DATA_DIR
    / "transaction_cost_sharpe_pivot.csv",
)

cost_robustness_summary.to_csv(
    PROCESSED_DATA_DIR
    / "transaction_cost_robustness_summary.csv",
)

transaction_cost_leaders.to_csv(
    PROCESSED_DATA_DIR
    / "transaction_cost_leaders.csv",
)


# ---------------------------------------------------------
# 9. Display results
# ---------------------------------------------------------

display_cost_cagr = (
    cost_cagr_pivot
    * 100
)

display_cost_robustness = (
    cost_robustness_summary.copy()
)

for column in [
    "Zero-Cost CAGR",
    "Base-Cost CAGR",
    "100bps CAGR",
    "CAGR Drag: 0 to 100bps",
    "Annualised Turnover",
]:

    display_cost_robustness[
        column
    ] = (
        display_cost_robustness[
            column
        ]
        * 100
    )


display_cost_leaders = (
    transaction_cost_leaders.copy()
)

display_cost_leaders[
    "Best CAGR"
] = (
    display_cost_leaders[
        "Best CAGR"
    ]
    * 100
)


print("TRANSACTION-COST SENSITIVITY ANALYSIS")
print("=" * 72)
print(
    "Cost scenarios:",
    len(
        transaction_cost_scenarios
    ),
)
print(
    "One-way cost range:",
    "0 to 100 basis points",
)
print(
    "Portfolio strategies:",
    monthly_strategy_weights[
        "Strategy"
    ].nunique(),
)
print(
    "Benchmark:",
    "Nifty 50",
)
print(
    "Base cost:",
    "15 basis points",
)
print(
    "Monotonic ending-value validation:",
    "PASSED",
)
print(
    "Transaction-cost robustness:",
    "PASSED",
)

print("\nCAGR BY TRANSACTION-COST SCENARIO (%)")
display(
    display_cost_cagr.round(
        2
    )
)

print("\nSTRATEGY COST ROBUSTNESS")
display(
    display_cost_robustness.round(
        2
    )
)

print("\nTRANSACTION-COST LEADERS")
display(
    display_cost_leaders.round(
        2
    )
)

TRANSACTION-COST SENSITIVITY ANALYSIS
Cost scenarios: 6
One-way cost range: 0 to 100 basis points
Portfolio strategies: 9
Benchmark: Nifty 50
Base cost: 15 basis points
Monotonic ending-value validation: PASSED
Transaction-cost robustness: PASSED

CAGR BY TRANSACTION-COST SCENARIO (%)


Cost Scenario,0 bps,5 bps,15 bps — Base,30 bps,50 bps,100 bps
Strategy,,,,,,
1-Month Reversal,30.93,30.36,29.24,27.57,25.37,20.03
12-Month Momentum,25.18,25.00,24.64,24.10,23.38,21.61
Equal-Weight India 10,20.86,20.83,20.78,20.71,20.61,20.36
Gradient Boosting,25.06,24.60,23.68,22.32,20.52,16.12
Linear Regression,25.34,24.88,23.97,22.62,20.84,16.48
Nifty 50,10.98,10.98,10.98,10.98,10.98,10.98
Nifty Outperformance Logistic,18.16,17.74,16.91,15.67,14.04,10.04
Positive Return Logistic,23.97,23.56,22.74,21.51,19.89,15.93
Random Forest,20.78,20.37,19.54,18.32,16.70,12.73



STRATEGY COST ROBUSTNESS


,Zero-Cost CAGR,Base-Cost CAGR,100bps CAGR,CAGR Drag: 0 to 100bps,Base-Cost Sharpe,100bps Sharpe,Annualised Turnover,Base Transaction Costs (₹),100bps Transaction Costs (₹),Beats Equal Weight at 100bps,Beats Nifty at 100bps
Strategy,,,,,,,,,,,
12-Month Momentum,25.18,24.64,21.61,3.57,0.84,0.73,288.58,90850.69,529388.89,True,True
Equal-Weight India 10,20.86,20.78,20.36,0.50,0.86,0.84,41.41,10846.57,70940.25,False,True
1-Month Reversal,30.93,29.24,20.03,10.90,1.03,0.68,865.42,449638.41,1986657.04,False,True
Linear Regression,25.34,23.97,16.48,8.86,0.84,0.54,730.44,253682.90,1205504.83,False,True
Ridge Regression,25.34,23.97,16.48,8.86,0.84,0.54,730.44,253682.90,1205504.83,False,True
Gradient Boosting,25.06,23.68,16.12,8.94,0.84,0.53,738.93,219676.83,1043822.82,False,True
Positive Return Logistic,23.97,22.74,15.93,8.05,0.82,0.53,669.06,251505.08,1241228.38,False,True
Random Forest,20.78,19.54,12.73,8.05,0.68,0.39,687.48,174612.87,855665.38,False,True
Nifty 50,10.98,10.98,10.98,0.00,0.34,0.34,0.00,0.00,0.00,False,False



TRANSACTION-COST LEADERS


,Best CAGR Strategy,Best CAGR,Best Sharpe Strategy,Best Sharpe Ratio
Cost Scenario,,,,
0 bps,1-Month Reversal,30.93,1-Month Reversal,1.09
5 bps,1-Month Reversal,30.36,1-Month Reversal,1.07
15 bps — Base,1-Month Reversal,29.24,1-Month Reversal,1.03
30 bps,1-Month Reversal,27.57,1-Month Reversal,0.97
50 bps,1-Month Reversal,25.37,1-Month Reversal,0.89
100 bps,12-Month Momentum,21.61,Equal-Weight India 10,0.84


## Advanced ML Portfolio Construction

The earlier strategies selected the top three stocks and assigned equal weights.

This section creates three additional portfolios:

1. Positive Return Logistic — Probability Weighted  
2. Nifty Outperformance Logistic — Probability Weighted  
3. Gradient Boosting — Inverse Volatility  

Probability-weighted portfolios allocate more capital to stocks with higher predicted probabilities.

The inverse-volatility portfolio selects the three stocks with the strongest Gradient Boosting signals and assigns larger weights to stocks with lower recent volatility.

All weights use only information available on the signal date and retain the one-trading-day execution lag.

In [20]:
# =========================================================
# ADVANCED ML PORTFOLIO CONSTRUCTION
# Probability weighting + inverse-volatility sizing
# =========================================================

required_objects = [
    "classification_predictions",
    "tree_regression_predictions",
    "ml_panel",
    "market_calendar",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required previous-step objects are missing:\n"
        + "\n".join(
            missing_objects
        )
    )


TOP_STOCK_COUNT = 3

market_calendar = pd.DatetimeIndex(
    market_calendar
).sort_values()


# ---------------------------------------------------------
# 1. Helper for one-trading-day execution lag
# ---------------------------------------------------------

def get_advanced_execution_date(
    signal_date,
):

    calendar_position = (
        market_calendar.searchsorted(
            pd.Timestamp(
                signal_date
            ),
            side="right",
        )
    )

    if calendar_position >= len(
        market_calendar
    ):
        return pd.NaT

    return market_calendar[
        calendar_position
    ]


# ---------------------------------------------------------
# 2. Build probability-weighted classification portfolios
# ---------------------------------------------------------

advanced_weight_records = []

classification_weight_models = [
    "Positive Return Logistic",
    "Nifty Outperformance Logistic",
]

for model_name in classification_weight_models:

    model_predictions = (
        classification_predictions.loc[
            classification_predictions[
                "Model"
            ]
            == model_name
        ]
        .copy()
    )

    strategy_name = (
        model_name
        + " — Probability Weighted"
    )

    for signal_date, cross_section in (
        model_predictions.groupby(
            "Date"
        )
    ):

        if len(
            cross_section
        ) != len(
            INDIA_10_TICKERS
        ):
            raise RuntimeError(
                f"Incomplete classification cross-section "
                f"for {model_name} on {signal_date}."
            )

        execution_date = (
            get_advanced_execution_date(
                signal_date
            )
        )

        if pd.isna(
            execution_date
        ):
            continue

        selected_stocks = (
            cross_section
            .sort_values(
                by=[
                    "Predicted Probability",
                    "Ticker",
                ],
                ascending=[
                    False,
                    True,
                ],
            )
            .head(
                TOP_STOCK_COUNT
            )
            .copy()
        )

        selected_stocks[
            "Raw Weight"
        ] = (
            selected_stocks[
                "Predicted Probability"
            ]
            .clip(
                lower=1e-6
            )
        )

        selected_stocks[
            "Target Weight"
        ] = (
            selected_stocks[
                "Raw Weight"
            ]
            / selected_stocks[
                "Raw Weight"
            ].sum()
        )

        target_weight_map = (
            selected_stocks
            .set_index(
                "Ticker"
            )[
                "Target Weight"
            ]
            .to_dict()
        )

        score_map = (
            cross_section
            .set_index(
                "Ticker"
            )[
                "Predicted Probability"
            ]
            .to_dict()
        )

        for ticker in INDIA_10_TICKERS:

            advanced_weight_records.append(
                {
                    "Strategy":
                        strategy_name,

                    "Construction Method":
                        "Probability Weighted",

                    "Signal Date":
                        pd.Timestamp(
                            signal_date
                        ),

                    "Execution Date":
                        execution_date,

                    "Ticker":
                        ticker,

                    "Score":
                        score_map[
                            ticker
                        ],

                    "Volatility 63D":
                        np.nan,

                    "Selected":
                        ticker
                        in target_weight_map,

                    "Target Weight":
                        target_weight_map.get(
                            ticker,
                            0.0,
                        ),
                }
            )


# ---------------------------------------------------------
# 3. Prepare volatility data for Gradient Boosting
# ---------------------------------------------------------

volatility_signal_data = (
    ml_panel[
        [
            "Date",
            "Ticker",
            "volatility_63d",
        ]
    ]
    .copy()
)

volatility_signal_data[
    "Date"
] = pd.to_datetime(
    volatility_signal_data[
        "Date"
    ]
)

gradient_predictions = (
    tree_regression_predictions.loc[
        tree_regression_predictions[
            "Model"
        ]
        == "Gradient Boosting"
    ]
    .copy()
)

gradient_predictions[
    "Date"
] = pd.to_datetime(
    gradient_predictions[
        "Date"
    ]
)

gradient_predictions = (
    gradient_predictions
    .merge(
        volatility_signal_data,
        on=[
            "Date",
            "Ticker",
        ],
        how="left",
        validate="one_to_one",
    )
)

if gradient_predictions[
    "volatility_63d"
].isna().any():
    raise RuntimeError(
        "Missing volatility observations for "
        "Gradient Boosting portfolio sizing."
    )


# ---------------------------------------------------------
# 4. Build Gradient Boosting inverse-volatility portfolio
# ---------------------------------------------------------

gradient_strategy_name = (
    "Gradient Boosting — Inverse Volatility"
)

for signal_date, cross_section in (
    gradient_predictions.groupby(
        "Date"
    )
):

    if len(
        cross_section
    ) != len(
        INDIA_10_TICKERS
    ):
        raise RuntimeError(
            "Incomplete Gradient Boosting cross-section "
            f"on {signal_date}."
        )

    execution_date = (
        get_advanced_execution_date(
            signal_date
        )
    )

    if pd.isna(
        execution_date
    ):
        continue

    selected_stocks = (
        cross_section
        .sort_values(
            by=[
                "Predicted Excess Return",
                "Ticker",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .head(
            TOP_STOCK_COUNT
        )
        .copy()
    )

    # A small floor prevents an unusually low volatility
    # estimate from creating an extreme portfolio weight.
    selected_stocks[
        "Adjusted Volatility"
    ] = (
        selected_stocks[
            "volatility_63d"
        ]
        .clip(
            lower=0.05
        )
    )

    selected_stocks[
        "Inverse Volatility"
    ] = (
        1
        / selected_stocks[
            "Adjusted Volatility"
        ]
    )

    selected_stocks[
        "Target Weight"
    ] = (
        selected_stocks[
            "Inverse Volatility"
        ]
        / selected_stocks[
            "Inverse Volatility"
        ].sum()
    )

    target_weight_map = (
        selected_stocks
        .set_index(
            "Ticker"
        )[
            "Target Weight"
        ]
        .to_dict()
    )

    score_map = (
        cross_section
        .set_index(
            "Ticker"
        )[
            "Predicted Excess Return"
        ]
        .to_dict()
    )

    volatility_map = (
        cross_section
        .set_index(
            "Ticker"
        )[
            "volatility_63d"
        ]
        .to_dict()
    )

    for ticker in INDIA_10_TICKERS:

        advanced_weight_records.append(
            {
                "Strategy":
                    gradient_strategy_name,

                "Construction Method":
                    "Top-3 Inverse Volatility",

                "Signal Date":
                    pd.Timestamp(
                        signal_date
                    ),

                "Execution Date":
                    execution_date,

                "Ticker":
                    ticker,

                "Score":
                    score_map[
                        ticker
                    ],

                "Volatility 63D":
                    volatility_map[
                        ticker
                    ],

                "Selected":
                    ticker
                    in target_weight_map,

                "Target Weight":
                    target_weight_map.get(
                        ticker,
                        0.0,
                    ),
            }
        )


# ---------------------------------------------------------
# 5. Create the complete advanced-weights table
# ---------------------------------------------------------

advanced_strategy_weights = (
    pd.DataFrame(
        advanced_weight_records
    )
    .sort_values(
        [
            "Execution Date",
            "Strategy",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


advanced_rebalance_summary = (
    advanced_strategy_weights
    .groupby(
        [
            "Strategy",
            "Signal Date",
            "Execution Date",
        ]
    )
    .agg(
        Weight_Total=(
            "Target Weight",
            "sum",
        ),

        Active_Holdings=(
            "Selected",
            "sum",
        ),

        Minimum_Active_Weight=(
            "Target Weight",
            lambda values: (
                values[
                    values > 0
                ].min()
            ),
        ),

        Maximum_Weight=(
            "Target Weight",
            "max",
        ),

        Weight_Concentration=(
            "Target Weight",
            lambda values: (
                values.pow(
                    2
                ).sum()
            ),
        ),
    )
    .reset_index()
)


advanced_strategy_summary = (
    advanced_rebalance_summary
    .groupby(
        "Strategy"
    )
    .agg(
        Rebalances=(
            "Signal Date",
            "nunique",
        ),

        First_Signal_Date=(
            "Signal Date",
            "min",
        ),

        Last_Signal_Date=(
            "Signal Date",
            "max",
        ),

        Median_Active_Holdings=(
            "Active_Holdings",
            "median",
        ),

        Average_Minimum_Active_Weight=(
            "Minimum_Active_Weight",
            "mean",
        ),

        Average_Maximum_Weight=(
            "Maximum_Weight",
            "mean",
        ),

        Average_Weight_Concentration=(
            "Weight_Concentration",
            "mean",
        ),
    )
    .sort_index()
)


# ---------------------------------------------------------
# 6. Validate advanced portfolio construction
# ---------------------------------------------------------

assert (
    advanced_strategy_weights[
        "Execution Date"
    ]
    > advanced_strategy_weights[
        "Signal Date"
    ]
).all()

assert (
    advanced_strategy_weights[
        "Target Weight"
    ]
    .ge(
        0
    )
    .all()
)

assert np.allclose(
    advanced_rebalance_summary[
        "Weight_Total"
    ],
    1.0,
    atol=1e-10,
)

assert (
    advanced_rebalance_summary[
        "Active_Holdings"
    ]
    .eq(
        TOP_STOCK_COUNT
    )
    .all()
)

assert (
    advanced_strategy_weights[
        "Strategy"
    ].nunique()
    == 3
)

assert (
    advanced_strategy_summary[
        "Rebalances"
    ]
    .eq(
        101
    )
    .all()
)

assert not advanced_strategy_weights.duplicated(
    subset=[
        "Strategy",
        "Signal Date",
        "Ticker",
    ]
).any()


# ---------------------------------------------------------
# 7. Save datasets
# ---------------------------------------------------------

advanced_strategy_weights.to_csv(
    PROCESSED_DATA_DIR
    / "advanced_ml_strategy_weights.csv",
    index=False,
)

advanced_rebalance_summary.to_csv(
    PROCESSED_DATA_DIR
    / "advanced_ml_rebalance_summary.csv",
    index=False,
)

advanced_strategy_summary.to_csv(
    PROCESSED_DATA_DIR
    / "advanced_ml_strategy_summary.csv",
)


# ---------------------------------------------------------
# 8. Display results
# ---------------------------------------------------------

display_advanced_summary = (
    advanced_strategy_summary.copy()
)

for column in [
    "Average_Minimum_Active_Weight",
    "Average_Maximum_Weight",
    "Average_Weight_Concentration",
]:

    display_advanced_summary[
        column
    ] = (
        display_advanced_summary[
            column
        ]
        * 100
    )


print("ADVANCED ML PORTFOLIO CONSTRUCTION")
print("=" * 72)
print(
    "Advanced strategies:",
    advanced_strategy_weights[
        "Strategy"
    ].nunique(),
)
print(
    "Monthly rebalances:",
    advanced_strategy_weights[
        "Signal Date"
    ].nunique(),
)
print(
    "Stocks selected per strategy:",
    TOP_STOCK_COUNT,
)
print(
    "Probability-weighted strategies:",
    2,
)
print(
    "Inverse-volatility strategies:",
    1,
)
print(
    "One-trading-day execution lag:",
    "PASSED",
)
print(
    "All target weights sum to 100%:",
    "PASSED",
)
print(
    "Advanced portfolio construction:",
    "PASSED",
)

display(
    display_advanced_summary.round(
        2
    )
)

ADVANCED ML PORTFOLIO CONSTRUCTION
Advanced strategies: 3
Monthly rebalances: 101
Stocks selected per strategy: 3
Probability-weighted strategies: 2
Inverse-volatility strategies: 1
One-trading-day execution lag: PASSED
All target weights sum to 100%: PASSED
Advanced portfolio construction: PASSED


,Rebalances,First_Signal_Date,Last_Signal_Date,Median_Active_Holdings,Average_Minimum_Active_Weight,Average_Maximum_Weight,Average_Weight_Concentration
Strategy,,,,,,,
Gradient Boosting — Inverse Volatility,101,2018-03-28,2026-07-01,3.0,26.29,40.76,34.77
Nifty Outperformance Logistic — Probability Weighted,101,2018-03-28,2026-07-01,3.0,31.99,34.92,33.44
Positive Return Logistic — Probability Weighted,101,2018-03-28,2026-07-01,3.0,32.25,34.53,33.38


## Advanced ML Portfolio Backtest

The probability-weighted and inverse-volatility portfolios are evaluated using the same daily accounting framework as the original strategies.

The comparison isolates the effect of portfolio construction:

- Positive Return Logistic: equal weight versus probability weight
- Nifty Outperformance Logistic: equal weight versus probability weight
- Gradient Boosting: equal weight versus inverse-volatility weight

All strategies use identical stock rankings, monthly rebalance dates, next-trading-day execution and 0.15% one-way transaction costs.

In [21]:
# =========================================================
# BACKTEST ADVANCED ML PORTFOLIO ALLOCATIONS
# =========================================================

required_objects = [
    "advanced_strategy_weights",
    "asset_daily_returns",
    "nifty_daily_returns",
    "backtest_monthly_target_weights",
    "calculate_backtest_statistics",
    "strategy_backtest_summary",
    "strategy_daily_net_returns",
    "strategy_daily_gross_returns",
    "strategy_daily_values",
    "strategy_daily_turnover",
    "strategy_daily_costs",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required previous-step objects are missing:\n"
        + "\n".join(
            missing_objects
        )
    )


# ---------------------------------------------------------
# 1. Backtest each advanced portfolio
# ---------------------------------------------------------

advanced_strategy_backtest_results = {}

for strategy_name, strategy_weights in (
    advanced_strategy_weights.groupby(
        "Strategy"
    )
):

    advanced_strategy_backtest_results[
        strategy_name
    ] = backtest_monthly_target_weights(
        strategy_weights=(
            strategy_weights
        ),

        daily_asset_returns=(
            asset_daily_returns
        ),

        initial_investment_inr=(
            INITIAL_INVESTMENT_INR
        ),

        one_way_transaction_cost=(
            ONE_WAY_TRANSACTION_COST
        ),
    )


# ---------------------------------------------------------
# 2. Combine advanced daily results
# ---------------------------------------------------------

advanced_daily_net_returns = pd.concat(
    {
        strategy_name:
            result[
                "Net Return"
            ]
        for strategy_name, result in (
            advanced_strategy_backtest_results.items()
        )
    },
    axis=1,
)

advanced_daily_gross_returns = pd.concat(
    {
        strategy_name:
            result[
                "Gross Return"
            ]
        for strategy_name, result in (
            advanced_strategy_backtest_results.items()
        )
    },
    axis=1,
)

advanced_daily_values = pd.concat(
    {
        strategy_name:
            result[
                "Portfolio Value (₹)"
            ]
        for strategy_name, result in (
            advanced_strategy_backtest_results.items()
        )
    },
    axis=1,
)

advanced_daily_turnover = pd.concat(
    {
        strategy_name:
            result[
                "One-Way Turnover"
            ]
        for strategy_name, result in (
            advanced_strategy_backtest_results.items()
        )
    },
    axis=1,
)

advanced_daily_costs = pd.concat(
    {
        strategy_name:
            result[
                "Transaction Cost (₹)"
            ]
        for strategy_name, result in (
            advanced_strategy_backtest_results.items()
        )
    },
    axis=1,
)


# ---------------------------------------------------------
# 3. Calculate advanced-strategy performance statistics
# ---------------------------------------------------------

advanced_summary_records = []

for strategy_name in (
    advanced_daily_net_returns.columns
):

    strategy_index = (
        advanced_daily_net_returns.index
    )

    aligned_nifty_returns = (
        nifty_daily_returns
        .reindex(
            strategy_index
        )
        .copy()
    )

    if aligned_nifty_returns.isna().any():
        raise RuntimeError(
            "Missing Nifty 50 observations during "
            "the advanced strategy backtest."
        )

    aligned_nifty_returns.iloc[
        0
    ] = 0.0

    statistics = calculate_backtest_statistics(
        strategy_name=(
            strategy_name
        ),

        daily_returns=(
            advanced_daily_net_returns[
                strategy_name
            ]
        ),

        portfolio_values=(
            advanced_daily_values[
                strategy_name
            ]
        ),

        benchmark_returns=(
            aligned_nifty_returns
        ),

        daily_turnover=(
            advanced_daily_turnover[
                strategy_name
            ]
        ),

        daily_costs=(
            advanced_daily_costs[
                strategy_name
            ]
        ),
    )

    advanced_summary_records.append(
        statistics
    )


advanced_strategy_backtest_summary = (
    pd.DataFrame(
        advanced_summary_records
    )
    .set_index(
        "Strategy"
    )
    .sort_values(
        "Sharpe Ratio",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 4. Compare alternative weighting with base strategies
# ---------------------------------------------------------

advanced_to_base_strategy = {
    "Positive Return Logistic — Probability Weighted":
        "Positive Return Logistic",

    "Nifty Outperformance Logistic — Probability Weighted":
        "Nifty Outperformance Logistic",

    "Gradient Boosting — Inverse Volatility":
        "Gradient Boosting",
}

construction_comparison_records = []

for advanced_strategy, base_strategy in (
    advanced_to_base_strategy.items()
):

    advanced_row = (
        advanced_strategy_backtest_summary.loc[
            advanced_strategy
        ]
    )

    base_row = (
        strategy_backtest_summary.loc[
            base_strategy
        ]
    )

    construction_comparison_records.append(
        {
            "Advanced Strategy":
                advanced_strategy,

            "Base Strategy":
                base_strategy,

            "Advanced Ending Value (₹)":
                advanced_row[
                    "Ending Value (₹)"
                ],

            "Base Ending Value (₹)":
                base_row[
                    "Ending Value (₹)"
                ],

            "Advanced CAGR":
                advanced_row[
                    "CAGR"
                ],

            "Base CAGR":
                base_row[
                    "CAGR"
                ],

            "CAGR Change":
                (
                    advanced_row[
                        "CAGR"
                    ]
                    - base_row[
                        "CAGR"
                    ]
                ),

            "Advanced Volatility":
                advanced_row[
                    "Annualised Volatility"
                ],

            "Base Volatility":
                base_row[
                    "Annualised Volatility"
                ],

            "Volatility Change":
                (
                    advanced_row[
                        "Annualised Volatility"
                    ]
                    - base_row[
                        "Annualised Volatility"
                    ]
                ),

            "Advanced Sharpe":
                advanced_row[
                    "Sharpe Ratio"
                ],

            "Base Sharpe":
                base_row[
                    "Sharpe Ratio"
                ],

            "Sharpe Change":
                (
                    advanced_row[
                        "Sharpe Ratio"
                    ]
                    - base_row[
                        "Sharpe Ratio"
                    ]
                ),

            "Advanced Maximum Drawdown":
                advanced_row[
                    "Maximum Drawdown"
                ],

            "Base Maximum Drawdown":
                base_row[
                    "Maximum Drawdown"
                ],

            "Advanced Annualised Turnover":
                advanced_row[
                    "Annualised Turnover"
                ],

            "Base Annualised Turnover":
                base_row[
                    "Annualised Turnover"
                ],

            "Turnover Change":
                (
                    advanced_row[
                        "Annualised Turnover"
                    ]
                    - base_row[
                        "Annualised Turnover"
                    ]
                ),

            "Advanced Transaction Costs (₹)":
                advanced_row[
                    "Transaction Costs (₹)"
                ],

            "Base Transaction Costs (₹)":
                base_row[
                    "Transaction Costs (₹)"
                ],
        }
    )


portfolio_construction_comparison = (
    pd.DataFrame(
        construction_comparison_records
    )
    .set_index(
        "Advanced Strategy"
    )
    .sort_values(
        "Sharpe Change",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 5. Add advanced portfolios to the complete strategy set
# ---------------------------------------------------------

all_strategy_daily_net_returns = pd.concat(
    [
        strategy_daily_net_returns,
        advanced_daily_net_returns,
    ],
    axis=1,
)

all_strategy_daily_gross_returns = pd.concat(
    [
        strategy_daily_gross_returns,
        advanced_daily_gross_returns,
    ],
    axis=1,
)

all_strategy_daily_values = pd.concat(
    [
        strategy_daily_values,
        advanced_daily_values,
    ],
    axis=1,
)

all_strategy_daily_turnover = pd.concat(
    [
        strategy_daily_turnover,
        advanced_daily_turnover,
    ],
    axis=1,
)

all_strategy_daily_costs = pd.concat(
    [
        strategy_daily_costs,
        advanced_daily_costs,
    ],
    axis=1,
)

all_strategy_backtest_summary = pd.concat(
    [
        strategy_backtest_summary,
        advanced_strategy_backtest_summary,
    ],
    axis=0,
).sort_values(
    "Sharpe Ratio",
    ascending=False,
)


# ---------------------------------------------------------
# 6. Save advanced and combined backtest outputs
# ---------------------------------------------------------

advanced_daily_net_returns.to_csv(
    PROCESSED_DATA_DIR
    / "advanced_strategy_daily_net_returns.csv",
    index_label="Date",
)

advanced_daily_values.to_csv(
    PROCESSED_DATA_DIR
    / "advanced_strategy_daily_portfolio_values.csv",
    index_label="Date",
)

advanced_daily_turnover.to_csv(
    PROCESSED_DATA_DIR
    / "advanced_strategy_daily_turnover.csv",
    index_label="Date",
)

advanced_daily_costs.to_csv(
    PROCESSED_DATA_DIR
    / "advanced_strategy_daily_transaction_costs.csv",
    index_label="Date",
)

advanced_strategy_backtest_summary.to_csv(
    PROCESSED_DATA_DIR
    / "advanced_strategy_backtest_summary.csv",
)

portfolio_construction_comparison.to_csv(
    PROCESSED_DATA_DIR
    / "portfolio_construction_comparison.csv",
)

all_strategy_backtest_summary.to_csv(
    PROCESSED_DATA_DIR
    / "all_strategy_backtest_summary.csv",
)


# ---------------------------------------------------------
# 7. Validate advanced backtests
# ---------------------------------------------------------

assert (
    advanced_daily_net_returns.shape[
        1
    ]
    == 3
)

assert (
    advanced_daily_net_returns
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    advanced_daily_net_returns.to_numpy()
).all()

assert (
    advanced_daily_values
    .gt(
        0
    )
    .all()
    .all()
)

assert (
    advanced_daily_net_returns.index
    .equals(
        strategy_daily_net_returns.index
    )
)

assert (
    advanced_strategy_backtest_summary[
        "Ending Value (₹)"
    ]
    .gt(
        0
    )
    .all()
)

assert (
    portfolio_construction_comparison[
        [
            "CAGR Change",
            "Sharpe Change",
            "Turnover Change",
        ]
    ]
    .notna()
    .all()
    .all()
)

assert (
    all_strategy_daily_net_returns.columns
    .nunique()
    == 13
)


# ---------------------------------------------------------
# 8. Display results
# ---------------------------------------------------------

display_advanced_backtest = (
    advanced_strategy_backtest_summary.copy()
)

advanced_percentage_columns = [
    "Total Return",
    "CAGR",
    "Annualised Volatility",
    "Maximum Drawdown",
    "Correlation vs Nifty",
    "Total One-Way Turnover",
    "Annualised Turnover",
]

for column in advanced_percentage_columns:

    display_advanced_backtest[
        column
    ] = (
        display_advanced_backtest[
            column
        ]
        * 100
    )


display_construction_comparison = (
    portfolio_construction_comparison.copy()
)

comparison_percentage_columns = [
    "Advanced CAGR",
    "Base CAGR",
    "CAGR Change",
    "Advanced Volatility",
    "Base Volatility",
    "Volatility Change",
    "Advanced Maximum Drawdown",
    "Base Maximum Drawdown",
    "Advanced Annualised Turnover",
    "Base Annualised Turnover",
    "Turnover Change",
]

for column in comparison_percentage_columns:

    display_construction_comparison[
        column
    ] = (
        display_construction_comparison[
            column
        ]
        * 100
    )


print("ADVANCED ML PORTFOLIO BACKTEST")
print("=" * 72)
print(
    "Backtest period:",
    advanced_daily_net_returns
    .index.min()
    .date(),
    "to",
    advanced_daily_net_returns
    .index.max()
    .date(),
)
print(
    "Advanced strategies:",
    advanced_daily_net_returns.shape[
        1
    ],
)
print(
    "Trading observations:",
    len(
        advanced_daily_net_returns
    ),
)
print(
    "Initial investment:",
    f"₹{INITIAL_INVESTMENT_INR:,.0f}",
)
print(
    "One-way transaction cost:",
    f"{ONE_WAY_TRANSACTION_COST:.2%}",
)
print(
    "Common backtest calendar:",
    "PASSED",
)
print(
    "Advanced strategy accounting:",
    "PASSED",
)
print(
    "Portfolio-construction comparison:",
    "PASSED",
)

print("\nADVANCED STRATEGY PERFORMANCE")
display(
    display_advanced_backtest.round(
        2
    )
)

print("\nADVANCED VERSUS BASE CONSTRUCTION")
display(
    display_construction_comparison.round(
        2
    )
)

ADVANCED ML PORTFOLIO BACKTEST
Backtest period: 2018-04-02 to 2026-07-30
Advanced strategies: 3
Trading observations: 2054
Initial investment: ₹1,000,000
One-way transaction cost: 0.15%
Common backtest calendar: PASSED
Advanced strategy accounting: PASSED
Portfolio-construction comparison: PASSED

ADVANCED STRATEGY PERFORMANCE


,Start Date,End Date,Years,Ending Value (₹),Total Return,CAGR,Annualised Volatility,Sharpe Ratio,Maximum Drawdown,Calmar Ratio,Beta vs Nifty,Correlation vs Nifty,Total One-Way Turnover,Annualised Turnover,Transaction Costs (₹),Rebalances
Strategy,,,,,,,,,,,,,,,,
Positive Return Logistic — Probability Weighted,2018-04-02,2026-07-30,8.33,5464918.21,446.49,22.63,20.23,0.82,-30.55,0.74,0.88,74.33,5590.27,671.44,249403.01,101
Gradient Boosting — Inverse Volatility,2018-04-02,2026-07-30,8.33,4829730.76,382.97,20.82,20.47,0.74,-32.24,0.65,0.88,73.20,6392.47,767.79,212168.08,101
Nifty Outperformance Logistic — Probability Weighted,2018-04-02,2026-07-30,8.33,3665276.10,266.53,16.88,20.59,0.57,-39.19,0.43,0.88,72.91,5933.21,712.63,190455.30,101



ADVANCED VERSUS BASE CONSTRUCTION


,Base Strategy,Advanced Ending Value (₹),Base Ending Value (₹),Advanced CAGR,Base CAGR,CAGR Change,Advanced Volatility,Base Volatility,Volatility Change,Advanced Sharpe,Base Sharpe,Sharpe Change,Advanced Maximum Drawdown,Base Maximum Drawdown,Advanced Annualised Turnover,Base Annualised Turnover,Turnover Change,Advanced Transaction Costs (₹),Base Transaction Costs (₹)
Advanced Strategy,,,,,,,,,,,,,,,,,,,
Nifty Outperformance Logistic — Probability Weighted,Nifty Outperformance Logistic,3665276.10,3671693.38,16.88,16.91,-0.02,20.59,20.57,0.02,0.57,0.57,-0.0,-39.19,-39.67,712.63,709.23,3.40,190455.30,192415.74
Positive Return Logistic — Probability Weighted,Positive Return Logistic,5464918.21,5505116.77,22.63,22.74,-0.11,20.23,20.25,-0.02,0.82,0.82,-0.0,-30.55,-31.09,671.44,669.06,2.38,249403.01,251505.08
Gradient Boosting — Inverse Volatility,Gradient Boosting,4829730.76,5868523.31,20.82,23.68,-2.86,20.47,21.14,-0.67,0.74,0.84,-0.1,-32.24,-30.39,767.79,738.93,28.86,212168.08,219676.83


## Walk-Forward Regime-Aware Portfolio

The regime-aware portfolio dynamically chooses among:

- 1-Month Reversal
- 12-Month Momentum
- Gradient Boosting
- Equal-Weight India 10

At each signal date:

1. Identify the current Nifty 50 regime.
2. Examine candidate-strategy returns from prior trading dates only.
3. Retain historical observations belonging to the same regime.
4. Calculate each candidate’s historical same-regime Sharpe ratio.
5. Select the candidate with the highest Sharpe ratio.
6. Use equal weight when fewer than 63 same-regime observations are available.

The selected portfolio is executed on the next trading day. No future strategy returns or future regime labels are used.

In [22]:
# =========================================================
# WALK-FORWARD REGIME-AWARE PORTFOLIO CONSTRUCTION
# =========================================================

required_objects = [
    "monthly_strategy_weights",
    "strategy_daily_net_returns",
    "market_regime_frame",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required previous-step objects are missing:\n"
        + "\n".join(
            missing_objects
        )
    )


REGIME_AWARE_STRATEGY_NAME = (
    "Regime-Aware Walk-Forward"
)

MINIMUM_REGIME_OBSERVATIONS = 63

regime_candidate_strategies = [
    "1-Month Reversal",
    "12-Month Momentum",
    "Gradient Boosting",
    "Equal-Weight India 10",
]

fallback_strategy = (
    "Equal-Weight India 10"
)


# ---------------------------------------------------------
# 1. Validate candidate data
# ---------------------------------------------------------

missing_candidate_returns = sorted(
    set(
        regime_candidate_strategies
    )
    - set(
        strategy_daily_net_returns.columns
    )
)

if missing_candidate_returns:
    raise RuntimeError(
        "Candidate return series are missing:\n"
        + "\n".join(
            missing_candidate_returns
        )
    )


missing_candidate_weights = sorted(
    set(
        regime_candidate_strategies
    )
    - set(
        monthly_strategy_weights[
            "Strategy"
        ].unique()
    )
)

if missing_candidate_weights:
    raise RuntimeError(
        "Candidate weight histories are missing:\n"
        + "\n".join(
            missing_candidate_weights
        )
    )


daily_risk_free_rate_regime_selector = (
    (
        1
        + RISK_FREE_RATE
    )
    ** (
        1
        / TRADING_DAYS_PER_YEAR
    )
    - 1
)


# ---------------------------------------------------------
# 2. Prepare the common monthly signal schedule
# ---------------------------------------------------------

regime_signal_dates = pd.DatetimeIndex(
    sorted(
        monthly_strategy_weights[
            "Signal Date"
        ].unique()
    )
)

candidate_return_history = (
    strategy_daily_net_returns[
        regime_candidate_strategies
    ]
    .copy()
)

candidate_return_history.index = (
    pd.to_datetime(
        candidate_return_history.index
    )
)

daily_regime_labels = (
    market_regime_frame[
        "Regime"
    ]
    .reindex(
        candidate_return_history.index
    )
)


# ---------------------------------------------------------
# 3. Select the best candidate using prior same-regime data
# ---------------------------------------------------------

regime_selection_records = []
candidate_score_records = []
regime_aware_weight_records = []

for signal_date in regime_signal_dates:

    if signal_date not in (
        market_regime_frame.index
    ):
        raise RuntimeError(
            f"Missing market regime on {signal_date}."
        )

    current_regime = (
        market_regime_frame.loc[
            signal_date,
            "Regime",
        ]
    )

    if pd.isna(
        current_regime
    ):
        raise RuntimeError(
            f"Undefined market regime on {signal_date}."
        )

    # Strictly exclude the current signal date and all
    # future observations from strategy selection.
    historical_date_mask = (
        candidate_return_history.index
        < signal_date
    )

    same_regime_mask = (
        daily_regime_labels
        == current_regime
    )

    eligible_history_mask = (
        historical_date_mask
        & same_regime_mask
    )

    strategy_scores = {}

    for candidate_strategy in (
        regime_candidate_strategies
    ):

        candidate_returns = (
            candidate_return_history.loc[
                eligible_history_mask,
                candidate_strategy,
            ]
            .dropna()
        )

        observations = len(
            candidate_returns
        )

        annualised_volatility = (
            candidate_returns.std(
                ddof=1
            )
            * np.sqrt(
                TRADING_DAYS_PER_YEAR
            )
            if observations > 1
            else np.nan
        )

        annualised_excess_return = (
            (
                candidate_returns.mean()
                - daily_risk_free_rate_regime_selector
            )
            * TRADING_DAYS_PER_YEAR
            if observations > 0
            else np.nan
        )

        eligible_for_selection = (
            observations
            >= MINIMUM_REGIME_OBSERVATIONS
            and pd.notna(
                annualised_volatility
            )
            and annualised_volatility > 0
        )

        historical_sharpe = (
            annualised_excess_return
            / annualised_volatility
            if eligible_for_selection
            else np.nan
        )

        strategy_scores[
            candidate_strategy
        ] = historical_sharpe

        candidate_score_records.append(
            {
                "Signal Date":
                    signal_date,

                "Regime":
                    current_regime,

                "Candidate Strategy":
                    candidate_strategy,

                "Same-Regime Observations":
                    observations,

                "Annualised Historical Return":
                    (
                        candidate_returns.mean()
                        * TRADING_DAYS_PER_YEAR
                        if observations > 0
                        else np.nan
                    ),

                "Annualised Historical Volatility":
                    annualised_volatility,

                "Historical Same-Regime Sharpe":
                    historical_sharpe,

                "Eligible":
                    eligible_for_selection,
            }
        )

    valid_strategy_scores = {
        strategy_name:
            score
        for strategy_name, score in (
            strategy_scores.items()
        )
        if pd.notna(
            score
        )
    }

    if valid_strategy_scores:

        selected_strategy = max(
            valid_strategy_scores,
            key=valid_strategy_scores.get,
        )

        selected_score = (
            valid_strategy_scores[
                selected_strategy
            ]
        )

        selection_method = (
            "Highest Prior Same-Regime Sharpe"
        )

        fallback_used = False

    else:

        selected_strategy = (
            fallback_strategy
        )

        selected_score = np.nan

        selection_method = (
            "Insufficient History — Equal-Weight Fallback"
        )

        fallback_used = True

    selected_source_weights = (
        monthly_strategy_weights.loc[
            (
                monthly_strategy_weights[
                    "Strategy"
                ]
                == selected_strategy
            )
            & (
                monthly_strategy_weights[
                    "Signal Date"
                ]
                == signal_date
            )
        ]
        .copy()
    )

    if len(
        selected_source_weights
    ) != len(
        INDIA_10_TICKERS
    ):
        raise RuntimeError(
            "Incomplete source portfolio for "
            f"{selected_strategy} on {signal_date}."
        )

    execution_date = (
        selected_source_weights[
            "Execution Date"
        ]
        .iloc[
            0
        ]
    )

    regime_selection_records.append(
        {
            "Signal Date":
                signal_date,

            "Execution Date":
                execution_date,

            "Regime":
                current_regime,

            "Selected Strategy":
                selected_strategy,

            "Selection Score":
                selected_score,

            "Selection Method":
                selection_method,

            "Fallback Used":
                fallback_used,

            "Available Eligible Candidates":
                len(
                    valid_strategy_scores
                ),
        }
    )

    for _, source_row in (
        selected_source_weights.iterrows()
    ):

        regime_aware_weight_records.append(
            {
                "Strategy":
                    REGIME_AWARE_STRATEGY_NAME,

                "Signal Type":
                    "Walk-Forward Regime Selector",

                "Signal Date":
                    signal_date,

                "Execution Date":
                    execution_date,

                "Regime":
                    current_regime,

                "Source Strategy":
                    selected_strategy,

                "Selection Score":
                    selected_score,

                "Ticker":
                    source_row[
                        "Ticker"
                    ],

                "Selected":
                    source_row[
                        "Selected"
                    ],

                "Target Weight":
                    source_row[
                        "Target Weight"
                    ],
            }
        )


# ---------------------------------------------------------
# 4. Create selection and weight datasets
# ---------------------------------------------------------

regime_selector_history = (
    pd.DataFrame(
        regime_selection_records
    )
    .sort_values(
        "Signal Date"
    )
    .reset_index(
        drop=True
    )
)

regime_candidate_scores = (
    pd.DataFrame(
        candidate_score_records
    )
    .sort_values(
        [
            "Signal Date",
            "Candidate Strategy",
        ]
    )
    .reset_index(
        drop=True
    )
)

regime_aware_strategy_weights = (
    pd.DataFrame(
        regime_aware_weight_records
    )
    .sort_values(
        [
            "Execution Date",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 5. Summarise strategy selection by regime
# ---------------------------------------------------------

regime_selection_summary = (
    regime_selector_history
    .groupby(
        [
            "Regime",
            "Selected Strategy",
        ]
    )
    .agg(
        Selections=(
            "Signal Date",
            "count",
        ),

        First_Selection=(
            "Signal Date",
            "min",
        ),

        Last_Selection=(
            "Signal Date",
            "max",
        ),

        Average_Selection_Score=(
            "Selection Score",
            "mean",
        ),

        Fallback_Selections=(
            "Fallback Used",
            "sum",
        ),
    )
    .reset_index()
)

regime_selection_pivot = (
    regime_selector_history
    .pivot_table(
        index="Regime",
        columns="Selected Strategy",
        values="Signal Date",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(
        index=[
            "Bull / Lower Volatility",
            "Bull / Higher Volatility",
            "Bear / Lower Volatility",
            "Bear / Higher Volatility",
        ],
        fill_value=0,
    )
)

overall_regime_selector_summary = (
    regime_selector_history
    .groupby(
        "Selected Strategy"
    )
    .agg(
        Selections=(
            "Signal Date",
            "count",
        ),

        Selection_Rate=(
            "Signal Date",
            lambda values: (
                len(
                    values
                )
                / len(
                    regime_selector_history
                )
            ),
        ),

        Regimes_Selected_In=(
            "Regime",
            "nunique",
        ),
    )
    .sort_values(
        "Selections",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 6. Validate leakage safety and portfolio weights
# ---------------------------------------------------------

regime_weight_totals = (
    regime_aware_strategy_weights
    .groupby(
        [
            "Signal Date",
            "Execution Date",
        ]
    )[
        "Target Weight"
    ]
    .sum()
)

regime_active_holdings = (
    regime_aware_strategy_weights
    .groupby(
        [
            "Signal Date",
            "Execution Date",
        ]
    )[
        "Selected"
    ]
    .sum()
)

assert (
    regime_selector_history[
        "Execution Date"
    ]
    > regime_selector_history[
        "Signal Date"
    ]
).all()

assert len(
    regime_selector_history
) == 101

assert (
    regime_aware_strategy_weights[
        "Strategy"
    ]
    .eq(
        REGIME_AWARE_STRATEGY_NAME
    )
    .all()
)

assert np.allclose(
    regime_weight_totals,
    1.0,
    atol=1e-10,
)

assert (
    regime_aware_strategy_weights[
        "Target Weight"
    ]
    .ge(
        0
    )
    .all()
)

assert (
    regime_active_holdings
    .isin(
        [
            TOP_STOCK_COUNT,
            len(
                INDIA_10_TICKERS
            ),
        ]
    )
    .all()
)

assert not regime_aware_strategy_weights.duplicated(
    subset=[
        "Signal Date",
        "Ticker",
    ]
).any()

assert (
    regime_candidate_scores.loc[
        regime_candidate_scores[
            "Eligible"
        ],
        "Same-Regime Observations",
    ]
    .ge(
        MINIMUM_REGIME_OBSERVATIONS
    )
    .all()
)


# ---------------------------------------------------------
# 7. Save regime-selector outputs
# ---------------------------------------------------------

regime_selector_history.to_csv(
    PROCESSED_DATA_DIR
    / "regime_selector_history.csv",
    index=False,
)

regime_candidate_scores.to_csv(
    PROCESSED_DATA_DIR
    / "regime_candidate_scores.csv",
    index=False,
)

regime_aware_strategy_weights.to_csv(
    PROCESSED_DATA_DIR
    / "regime_aware_strategy_weights.csv",
    index=False,
)

regime_selection_summary.to_csv(
    PROCESSED_DATA_DIR
    / "regime_selection_summary.csv",
    index=False,
)

regime_selection_pivot.to_csv(
    PROCESSED_DATA_DIR
    / "regime_selection_pivot.csv",
)

overall_regime_selector_summary.to_csv(
    PROCESSED_DATA_DIR
    / "overall_regime_selector_summary.csv",
)


# ---------------------------------------------------------
# 8. Display results
# ---------------------------------------------------------

display_overall_selector_summary = (
    overall_regime_selector_summary.copy()
)

display_overall_selector_summary[
    "Selection_Rate"
] = (
    display_overall_selector_summary[
        "Selection_Rate"
    ]
    * 100
)


print("WALK-FORWARD REGIME-AWARE PORTFOLIO")
print("=" * 72)
print(
    "Signal dates:",
    len(
        regime_selector_history
    ),
)
print(
    "Candidate strategies:",
    len(
        regime_candidate_strategies
    ),
)
print(
    "Minimum same-regime observations:",
    MINIMUM_REGIME_OBSERVATIONS,
)
print(
    "Fallback strategy:",
    fallback_strategy,
)
print(
    "Fallback selections:",
    int(
        regime_selector_history[
            "Fallback Used"
        ].sum()
    ),
)
print(
    "Observed regimes:",
    regime_selector_history[
        "Regime"
    ].nunique(),
)
print(
    "Prior-date performance only:",
    "PASSED",
)
print(
    "One-trading-day execution lag:",
    "PASSED",
)
print(
    "All target weights sum to 100%:",
    "PASSED",
)
print(
    "Regime-aware construction:",
    "PASSED",
)

print("\nSTRATEGY SELECTIONS BY REGIME")
display(
    regime_selection_pivot
)

print("\nOVERALL STRATEGY SELECTION SUMMARY")
display(
    display_overall_selector_summary.round(
        2
    )
)

WALK-FORWARD REGIME-AWARE PORTFOLIO
Signal dates: 101
Candidate strategies: 4
Minimum same-regime observations: 63
Fallback strategy: Equal-Weight India 10
Fallback selections: 14
Observed regimes: 4
Prior-date performance only: PASSED
One-trading-day execution lag: PASSED
All target weights sum to 100%: PASSED
Regime-aware construction: PASSED

STRATEGY SELECTIONS BY REGIME


Selected Strategy,1-Month Reversal,12-Month Momentum,Equal-Weight India 10,Gradient Boosting
Regime,,,,
Bull / Lower Volatility,0,40,2,0
Bull / Higher Volatility,27,1,4,0
Bear / Lower Volatility,1,0,5,2
Bear / Higher Volatility,13,0,3,3



OVERALL STRATEGY SELECTION SUMMARY


,Selections,Selection_Rate,Regimes_Selected_In
Selected Strategy,,,
1-Month Reversal,41,40.59,3
12-Month Momentum,41,40.59,2
Equal-Weight India 10,14,13.86,4
Gradient Boosting,5,4.95,2


## Regime-Aware Strategy Backtest

The walk-forward regime selector is now converted into a complete daily portfolio history.

The backtest uses:

- Market-regime information available on the signal date
- Candidate performance from prior dates only
- Next-trading-day execution
- Monthly rebalancing
- Natural portfolio-weight drift
- 0.15% one-way transaction costs
- An initial investment of ₹10,00,000

Performance is compared with its four candidate strategies and the Nifty 50.

In [23]:
# =========================================================
# BACKTEST WALK-FORWARD REGIME-AWARE STRATEGY
# =========================================================

required_objects = [
    "regime_aware_strategy_weights",
    "regime_selector_history",
    "asset_daily_returns",
    "nifty_daily_returns",
    "backtest_monthly_target_weights",
    "calculate_backtest_statistics",
    "all_strategy_daily_net_returns",
    "all_strategy_daily_gross_returns",
    "all_strategy_daily_values",
    "all_strategy_daily_turnover",
    "all_strategy_daily_costs",
    "all_strategy_backtest_summary",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required previous-step objects are missing:\n"
        + "\n".join(missing_objects)
    )


# ---------------------------------------------------------
# 1. Run the regime-aware daily backtest
# ---------------------------------------------------------

regime_aware_backtest_result = (
    backtest_monthly_target_weights(
        strategy_weights=(
            regime_aware_strategy_weights
        ),
        daily_asset_returns=(
            asset_daily_returns
        ),
        initial_investment_inr=(
            INITIAL_INVESTMENT_INR
        ),
        one_way_transaction_cost=(
            ONE_WAY_TRANSACTION_COST
        ),
    )
)


# ---------------------------------------------------------
# 2. Prepare the aligned Nifty 50 benchmark
# ---------------------------------------------------------

regime_aware_nifty_returns = (
    nifty_daily_returns
    .reindex(
        regime_aware_backtest_result.index
    )
    .copy()
)

if regime_aware_nifty_returns.isna().any():
    raise RuntimeError(
        "Missing Nifty 50 observations during "
        "the regime-aware backtest."
    )

# The portfolio is invested at the close of the first
# execution date, so benchmark performance also starts there.
regime_aware_nifty_returns.iloc[0] = 0.0


# ---------------------------------------------------------
# 3. Calculate performance statistics
# ---------------------------------------------------------

regime_aware_statistics = (
    calculate_backtest_statistics(
        strategy_name=(
            REGIME_AWARE_STRATEGY_NAME
        ),
        daily_returns=(
            regime_aware_backtest_result[
                "Net Return"
            ]
        ),
        portfolio_values=(
            regime_aware_backtest_result[
                "Portfolio Value (₹)"
            ]
        ),
        benchmark_returns=(
            regime_aware_nifty_returns
        ),
        daily_turnover=(
            regime_aware_backtest_result[
                "One-Way Turnover"
            ]
        ),
        daily_costs=(
            regime_aware_backtest_result[
                "Transaction Cost (₹)"
            ]
        ),
    )
)

regime_aware_backtest_summary = (
    pd.DataFrame(
        [
            regime_aware_statistics
        ]
    )
    .set_index(
        "Strategy"
    )
)


# ---------------------------------------------------------
# 4. Compare against candidates and Nifty 50
# ---------------------------------------------------------

comparison_strategies = [
    REGIME_AWARE_STRATEGY_NAME,
    "1-Month Reversal",
    "12-Month Momentum",
    "Gradient Boosting",
    "Equal-Weight India 10",
    "Nifty 50",
]

candidate_comparison_base = (
    all_strategy_backtest_summary
    .drop(
        index=REGIME_AWARE_STRATEGY_NAME,
        errors="ignore",
    )
)

regime_aware_candidate_comparison = (
    pd.concat(
        [
            candidate_comparison_base,
            regime_aware_backtest_summary,
        ],
        axis=0,
    )
    .loc[
        comparison_strategies
    ]
    .copy()
)

regime_aware_candidate_comparison[
    "CAGR Rank"
] = (
    regime_aware_candidate_comparison[
        "CAGR"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

regime_aware_candidate_comparison[
    "Sharpe Rank"
] = (
    regime_aware_candidate_comparison[
        "Sharpe Ratio"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

regime_aware_candidate_comparison[
    "Drawdown Rank"
] = (
    regime_aware_candidate_comparison[
        "Maximum Drawdown"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

regime_aware_candidate_comparison = (
    regime_aware_candidate_comparison
    .sort_values(
        "Sharpe Ratio",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 5. Summarise the selector's decisions
# ---------------------------------------------------------

regime_aware_selection_outcomes = (
    regime_selector_history
    .groupby(
        "Selected Strategy"
    )
    .agg(
        Selections=(
            "Signal Date",
            "count",
        ),
        Selection_Rate=(
            "Signal Date",
            lambda values: (
                len(values)
                / len(
                    regime_selector_history
                )
            ),
        ),
        Regimes_Selected_In=(
            "Regime",
            "nunique",
        ),
        First_Selection=(
            "Signal Date",
            "min",
        ),
        Last_Selection=(
            "Signal Date",
            "max",
        ),
        Fallback_Selections=(
            "Fallback Used",
            "sum",
        ),
    )
    .sort_values(
        "Selections",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 6. Add regime-aware results to the full strategy dataset
# ---------------------------------------------------------

final_strategy_daily_net_returns = (
    all_strategy_daily_net_returns.copy()
)

final_strategy_daily_gross_returns = (
    all_strategy_daily_gross_returns.copy()
)

final_strategy_daily_values = (
    all_strategy_daily_values.copy()
)

final_strategy_daily_turnover = (
    all_strategy_daily_turnover.copy()
)

final_strategy_daily_costs = (
    all_strategy_daily_costs.copy()
)

final_strategy_daily_net_returns[
    REGIME_AWARE_STRATEGY_NAME
] = (
    regime_aware_backtest_result[
        "Net Return"
    ]
)

final_strategy_daily_gross_returns[
    REGIME_AWARE_STRATEGY_NAME
] = (
    regime_aware_backtest_result[
        "Gross Return"
    ]
)

final_strategy_daily_values[
    REGIME_AWARE_STRATEGY_NAME
] = (
    regime_aware_backtest_result[
        "Portfolio Value (₹)"
    ]
)

final_strategy_daily_turnover[
    REGIME_AWARE_STRATEGY_NAME
] = (
    regime_aware_backtest_result[
        "One-Way Turnover"
    ]
)

final_strategy_daily_costs[
    REGIME_AWARE_STRATEGY_NAME
] = (
    regime_aware_backtest_result[
        "Transaction Cost (₹)"
    ]
)

final_all_strategy_backtest_summary = (
    pd.concat(
        [
            all_strategy_backtest_summary.drop(
                index=REGIME_AWARE_STRATEGY_NAME,
                errors="ignore",
            ),
            regime_aware_backtest_summary,
        ],
        axis=0,
    )
    .sort_values(
        "Sharpe Ratio",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 7. Calculate performance difference versus candidates
# ---------------------------------------------------------

regime_aware_cagr = (
    regime_aware_backtest_summary.loc[
        REGIME_AWARE_STRATEGY_NAME,
        "CAGR",
    ]
)

regime_aware_sharpe = (
    regime_aware_backtest_summary.loc[
        REGIME_AWARE_STRATEGY_NAME,
        "Sharpe Ratio",
    ]
)

regime_aware_candidate_comparison[
    "CAGR Difference vs Regime-Aware"
] = (
    regime_aware_candidate_comparison[
        "CAGR"
    ]
    - regime_aware_cagr
)

regime_aware_candidate_comparison[
    "Sharpe Difference vs Regime-Aware"
] = (
    regime_aware_candidate_comparison[
        "Sharpe Ratio"
    ]
    - regime_aware_sharpe
)


# ---------------------------------------------------------
# 8. Save all outputs
# ---------------------------------------------------------

regime_aware_backtest_result.to_csv(
    PROCESSED_DATA_DIR
    / "regime_aware_daily_backtest.csv",
    index_label="Date",
)

regime_aware_backtest_summary.to_csv(
    PROCESSED_DATA_DIR
    / "regime_aware_backtest_summary.csv",
)

regime_aware_candidate_comparison.to_csv(
    PROCESSED_DATA_DIR
    / "regime_aware_candidate_comparison.csv",
)

regime_aware_selection_outcomes.to_csv(
    PROCESSED_DATA_DIR
    / "regime_aware_selection_outcomes.csv",
)

final_strategy_daily_net_returns.to_csv(
    PROCESSED_DATA_DIR
    / "final_strategy_daily_net_returns.csv",
    index_label="Date",
)

final_strategy_daily_values.to_csv(
    PROCESSED_DATA_DIR
    / "final_strategy_daily_portfolio_values.csv",
    index_label="Date",
)

final_strategy_daily_turnover.to_csv(
    PROCESSED_DATA_DIR
    / "final_strategy_daily_turnover.csv",
    index_label="Date",
)

final_strategy_daily_costs.to_csv(
    PROCESSED_DATA_DIR
    / "final_strategy_daily_transaction_costs.csv",
    index_label="Date",
)

final_all_strategy_backtest_summary.to_csv(
    PROCESSED_DATA_DIR
    / "final_all_strategy_backtest_summary.csv",
)


# ---------------------------------------------------------
# 9. Validate the regime-aware backtest
# ---------------------------------------------------------

assert (
    regime_aware_backtest_result[
        "Portfolio Value (₹)"
    ]
    .gt(0)
    .all()
)

assert (
    regime_aware_backtest_result[
        "Net Return"
    ]
    .notna()
    .all()
)

assert np.isfinite(
    regime_aware_backtest_result[
        "Net Return"
    ]
).all()

assert (
    regime_aware_backtest_result.index
    .equals(
        all_strategy_daily_net_returns.index
    )
)

assert (
    regime_aware_backtest_summary.loc[
        REGIME_AWARE_STRATEGY_NAME,
        "Rebalances",
    ]
    == 101
)

assert (
    regime_aware_selection_outcomes[
        "Selections"
    ].sum()
    == 101
)

assert (
    final_strategy_daily_net_returns.columns
    .nunique()
    == (
        all_strategy_daily_net_returns.columns
        .nunique()
        + 1
    )
)

assert (
    final_all_strategy_backtest_summary.index
    .nunique()
    == 14
)


# ---------------------------------------------------------
# 10. Display results
# ---------------------------------------------------------

display_regime_aware_summary = (
    regime_aware_backtest_summary.copy()
)

summary_percentage_columns = [
    "Total Return",
    "CAGR",
    "Annualised Volatility",
    "Maximum Drawdown",
    "Correlation vs Nifty",
    "Total One-Way Turnover",
    "Annualised Turnover",
]

for column in summary_percentage_columns:

    display_regime_aware_summary[
        column
    ] = (
        display_regime_aware_summary[
            column
        ]
        * 100
    )


display_candidate_comparison = (
    regime_aware_candidate_comparison.copy()
)

comparison_percentage_columns = [
    "Total Return",
    "CAGR",
    "Annualised Volatility",
    "Maximum Drawdown",
    "Correlation vs Nifty",
    "Annualised Turnover",
    "CAGR Difference vs Regime-Aware",
]

for column in comparison_percentage_columns:

    display_candidate_comparison[
        column
    ] = (
        display_candidate_comparison[
            column
        ]
        * 100
    )


display_selection_outcomes = (
    regime_aware_selection_outcomes.copy()
)

display_selection_outcomes[
    "Selection_Rate"
] = (
    display_selection_outcomes[
        "Selection_Rate"
    ]
    * 100
)


print("REGIME-AWARE STRATEGY BACKTEST")
print("=" * 72)
print(
    "Backtest period:",
    regime_aware_backtest_result
    .index.min()
    .date(),
    "to",
    regime_aware_backtest_result
    .index.max()
    .date(),
)
print(
    "Trading observations:",
    len(
        regime_aware_backtest_result
    ),
)
print(
    "Monthly rebalances:",
    int(
        regime_aware_backtest_summary.loc[
            REGIME_AWARE_STRATEGY_NAME,
            "Rebalances",
        ]
    ),
)
print(
    "Initial investment:",
    f"₹{INITIAL_INVESTMENT_INR:,.0f}",
)
print(
    "One-way transaction cost:",
    f"{ONE_WAY_TRANSACTION_COST:.2%}",
)
print(
    "Final strategy count:",
    len(
        final_all_strategy_backtest_summary
    ),
)
print(
    "Daily portfolio accounting:",
    "PASSED",
)
print(
    "Candidate comparison:",
    "PASSED",
)
print(
    "Regime-aware backtest:",
    "PASSED",
)

print("\nREGIME-AWARE PERFORMANCE")
display(
    display_regime_aware_summary.round(2)
)

print("\nREGIME-AWARE VERSUS CANDIDATES")
display(
    display_candidate_comparison.round(2)
)

print("\nREGIME SELECTOR OUTCOMES")
display(
    display_selection_outcomes.round(2)
)

REGIME-AWARE STRATEGY BACKTEST
Backtest period: 2018-04-02 to 2026-07-30
Trading observations: 2054
Monthly rebalances: 101
Initial investment: ₹1,000,000
One-way transaction cost: 0.15%
Final strategy count: 14
Daily portfolio accounting: PASSED
Candidate comparison: PASSED
Regime-aware backtest: PASSED

REGIME-AWARE PERFORMANCE


,Start Date,End Date,Years,Ending Value (₹),Total Return,CAGR,Annualised Volatility,Sharpe Ratio,Maximum Drawdown,Calmar Ratio,Beta vs Nifty,Correlation vs Nifty,Total One-Way Turnover,Annualised Turnover,Transaction Costs (₹),Rebalances
Strategy,,,,,,,,,,,,,,,,
Regime-Aware Walk-Forward,2018-04-02,2026-07-30,8.33,11752022.41,1075.2,34.44,21.15,1.24,-27.69,1.24,0.88,71.09,5431.35,652.35,354506.83,101



REGIME-AWARE VERSUS CANDIDATES


,Start Date,End Date,Years,Ending Value (₹),Total Return,CAGR,Annualised Volatility,Sharpe Ratio,Maximum Drawdown,Calmar Ratio,...,Correlation vs Nifty,Total One-Way Turnover,Annualised Turnover,Transaction Costs (₹),Rebalances,CAGR Rank,Sharpe Rank,Drawdown Rank,CAGR Difference vs Regime-Aware,Sharpe Difference vs Regime-Aware
Strategy,,,,,,,,,,,,,,,,,,,,,
Regime-Aware Walk-Forward,2018-04-02,2026-07-30,8.33,11752022.41,1075.20,34.44,21.15,1.24,-27.69,1.24,...,71.09,54.31,652.35,354506.83,101,1,1,1,0.00,0.00
1-Month Reversal,2018-04-02,2026-07-30,8.33,8460873.01,746.09,29.24,21.54,1.03,-33.56,0.87,...,69.34,72.05,865.42,449638.41,101,2,2,5,-5.20,-0.21
Equal-Weight India 10,2018-04-02,2026-07-30,8.33,4816198.61,381.62,20.78,16.70,0.86,-30.47,0.68,...,88.76,3.45,41.41,10846.57,101,5,3,4,-13.66,-0.38
12-Month Momentum,2018-04-02,2026-07-30,8.33,6256936.61,525.69,24.64,22.35,0.84,-30.47,0.81,...,68.01,24.03,288.58,90850.69,101,3,4,3,-9.80,-0.40
Gradient Boosting,2018-04-02,2026-07-30,8.33,5868523.31,486.85,23.68,21.14,0.84,-30.39,0.78,...,72.66,61.52,738.93,219676.83,101,4,5,2,-10.76,-0.40
Nifty 50,2018-04-02,2026-07-30,8.33,2381279.58,138.13,10.98,17.09,0.34,-38.44,0.29,...,100.00,0.00,0.00,0.00,0,6,6,6,-23.46,-0.90



REGIME SELECTOR OUTCOMES


,Selections,Selection_Rate,Regimes_Selected_In,First_Selection,Last_Selection,Fallback_Selections
Selected Strategy,,,,,,
1-Month Reversal,41,40.59,3,2019-09-30,2026-07-01,0
12-Month Momentum,41,40.59,2,2018-07-31,2026-01-30,0
Equal-Weight India 10,14,13.86,4,2018-03-28,2024-12-31,14
Gradient Boosting,5,4.95,2,2019-08-30,2026-02-27,0


## Regime-Selector Robustness Audit

The regime-aware strategy materially outperformed every individual candidate. This section checks whether the result depends excessively on one specific parameter choice.

Six leakage-safe selector variants are tested:

- Expanding history with 42 same-regime observations
- Expanding history with 63 observations — base case
- Expanding history with 126 observations
- Trailing 504 trading days with 42 observations
- Trailing 756 trading days with 63 observations
- Trailing 1,008 trading days with 63 observations

Every variant:

- Uses only dates before the signal date
- Uses the regime observed on the signal date
- Executes on the next trading day
- Applies 0.15% one-way transaction costs
- Falls back to the equal-weight India 10 portfolio when history is insufficient

A robust result should remain economically competitive across several reasonable variants rather than relying entirely on the base specification.

In [25]:
# =========================================================
# REGIME-SELECTOR ROBUSTNESS AUDIT
# =========================================================

required_objects = [
    "monthly_strategy_weights",
    "candidate_return_history",
    "daily_regime_labels",
    "regime_signal_dates",
    "market_regime_frame",
    "regime_selector_history",
    "regime_aware_strategy_weights",
    "regime_aware_backtest_summary",
    "backtest_monthly_target_weights",
    "calculate_backtest_statistics",
    "asset_daily_returns",
    "nifty_daily_returns",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required previous-step objects are missing:\n"
        + "\n".join(
            missing_objects
        )
    )


# ---------------------------------------------------------
# 1. Define economically reasonable selector variants
# ---------------------------------------------------------

selector_variant_specs = [
    {
        "Variant":
            "Expanding / 42 observations",

        "Lookback Trading Days":
            None,

        "Minimum Same-Regime Observations":
            42,
    },

    {
        "Variant":
            "Expanding / 63 observations — Base",

        "Lookback Trading Days":
            None,

        "Minimum Same-Regime Observations":
            63,
    },

    {
        "Variant":
            "Expanding / 126 observations",

        "Lookback Trading Days":
            None,

        "Minimum Same-Regime Observations":
            126,
    },

    {
        "Variant":
            "Trailing 504 days / 42 observations",

        "Lookback Trading Days":
            504,

        "Minimum Same-Regime Observations":
            42,
    },

    {
        "Variant":
            "Trailing 756 days / 63 observations",

        "Lookback Trading Days":
            756,

        "Minimum Same-Regime Observations":
            63,
    },

    {
        "Variant":
            "Trailing 1,008 days / 63 observations",

        "Lookback Trading Days":
            1_008,

        "Minimum Same-Regime Observations":
            63,
    },
]

variant_order = [
    specification[
        "Variant"
    ]
    for specification in selector_variant_specs
]

robustness_candidate_strategies = [
    "1-Month Reversal",
    "12-Month Momentum",
    "Gradient Boosting",
    "Equal-Weight India 10",
]

robustness_fallback_strategy = (
    "Equal-Weight India 10"
)

daily_selector_risk_free_rate = (
    (
        1
        + RISK_FREE_RATE
    )
    ** (
        1
        / TRADING_DAYS_PER_YEAR
    )
    - 1
)


# ---------------------------------------------------------
# 2. Create one leakage-safe selector variant
# ---------------------------------------------------------

def construct_selector_variant(
    variant_name,
    lookback_trading_days,
    minimum_same_regime_observations,
):

    selection_records = []
    weight_records = []

    for signal_date in regime_signal_dates:

        signal_date = pd.Timestamp(
            signal_date
        )

        current_regime = (
            market_regime_frame.loc[
                signal_date,
                "Regime",
            ]
        )

        if pd.isna(
            current_regime
        ):
            raise RuntimeError(
                f"Undefined regime on {signal_date}."
            )

        # Only trading dates strictly before the signal date
        # are available for strategy evaluation.
        historical_dates = (
            candidate_return_history.index[
                candidate_return_history.index
                < signal_date
            ]
        )

        if lookback_trading_days is not None:

            historical_dates = (
                historical_dates[
                    -lookback_trading_days:
                ]
            )

        historical_regimes = (
            daily_regime_labels
            .reindex(
                historical_dates
            )
            .astype(
                object
            )
        )

        same_regime_dates = (
            historical_dates[
                historical_regimes.to_numpy()
                == current_regime
            ]
        )

        candidate_scores = {}

        for candidate_strategy in (
            robustness_candidate_strategies
        ):

            candidate_returns = (
                candidate_return_history.loc[
                    same_regime_dates,
                    candidate_strategy,
                ]
                .dropna()
            )

            observations = len(
                candidate_returns
            )

            annualised_volatility = (
                candidate_returns.std(
                    ddof=1
                )
                * np.sqrt(
                    TRADING_DAYS_PER_YEAR
                )
                if observations > 1
                else np.nan
            )

            annualised_excess_return = (
                (
                    candidate_returns.mean()
                    - daily_selector_risk_free_rate
                )
                * TRADING_DAYS_PER_YEAR
                if observations > 0
                else np.nan
            )

            eligible = (
                observations
                >= minimum_same_regime_observations
                and pd.notna(
                    annualised_volatility
                )
                and annualised_volatility > 0
            )

            candidate_scores[
                candidate_strategy
            ] = (
                annualised_excess_return
                / annualised_volatility
                if eligible
                else np.nan
            )

        valid_scores = {
            strategy_name:
                score
            for strategy_name, score in (
                candidate_scores.items()
            )
            if pd.notna(
                score
            )
        }

        if valid_scores:

            selected_strategy = max(
                valid_scores,
                key=valid_scores.get,
            )

            selected_score = (
                valid_scores[
                    selected_strategy
                ]
            )

            fallback_used = False

        else:

            selected_strategy = (
                robustness_fallback_strategy
            )

            selected_score = np.nan
            fallback_used = True

        source_weights = (
            monthly_strategy_weights.loc[
                (
                    monthly_strategy_weights[
                        "Strategy"
                    ]
                    == selected_strategy
                )
                & (
                    monthly_strategy_weights[
                        "Signal Date"
                    ]
                    == signal_date
                )
            ]
            .copy()
        )

        if len(
            source_weights
        ) != len(
            INDIA_10_TICKERS
        ):
            raise RuntimeError(
                f"Incomplete source weights for "
                f"{selected_strategy} on {signal_date}."
            )

        execution_date = (
            source_weights[
                "Execution Date"
            ]
            .iloc[
                0
            ]
        )

        selection_records.append(
            {
                "Variant":
                    variant_name,

                "Signal Date":
                    signal_date,

                "Execution Date":
                    execution_date,

                "Regime":
                    current_regime,

                "Selected Strategy":
                    selected_strategy,

                "Selection Score":
                    selected_score,

                "Fallback Used":
                    fallback_used,

                "Eligible Candidates":
                    len(
                        valid_scores
                    ),

                "Lookback Trading Days":
                    (
                        lookback_trading_days
                        if lookback_trading_days
                        is not None
                        else np.nan
                    ),

                "Minimum Same-Regime Observations":
                    minimum_same_regime_observations,
            }
        )

        for _, source_row in (
            source_weights.iterrows()
        ):

            weight_records.append(
                {
                    # Corrected from "Strategy" to "Variant".
                    "Variant":
                        variant_name,

                    "Signal Date":
                        signal_date,

                    "Execution Date":
                        execution_date,

                    "Regime":
                        current_regime,

                    "Source Strategy":
                        selected_strategy,

                    "Ticker":
                        source_row[
                            "Ticker"
                        ],

                    "Selected":
                        source_row[
                            "Selected"
                        ],

                    "Target Weight":
                        source_row[
                            "Target Weight"
                        ],
                }
            )

    return (
        pd.DataFrame(
            selection_records
        ),
        pd.DataFrame(
            weight_records
        ),
    )


# ---------------------------------------------------------
# 3. Construct and backtest every selector variant
# ---------------------------------------------------------

variant_selection_frames = []
variant_weight_frames = []
variant_backtest_results = {}
robustness_summary_records = []

base_selection_map = (
    regime_selector_history
    .set_index(
        "Signal Date"
    )[
        "Selected Strategy"
    ]
)

for specification in selector_variant_specs:

    variant_name = (
        specification[
            "Variant"
        ]
    )

    lookback_days = (
        specification[
            "Lookback Trading Days"
        ]
    )

    minimum_observations = (
        specification[
            "Minimum Same-Regime Observations"
        ]
    )

    variant_history, variant_weights = (
        construct_selector_variant(
            variant_name=(
                variant_name
            ),
            lookback_trading_days=(
                lookback_days
            ),
            minimum_same_regime_observations=(
                minimum_observations
            ),
        )
    )

    variant_selection_frames.append(
        variant_history
    )

    variant_weight_frames.append(
        variant_weights
    )

    variant_result = (
        backtest_monthly_target_weights(
            strategy_weights=(
                variant_weights
            ),
            daily_asset_returns=(
                asset_daily_returns
            ),
            initial_investment_inr=(
                INITIAL_INVESTMENT_INR
            ),
            one_way_transaction_cost=(
                ONE_WAY_TRANSACTION_COST
            ),
        )
    )

    variant_backtest_results[
        variant_name
    ] = variant_result

    aligned_nifty_returns = (
        nifty_daily_returns
        .reindex(
            variant_result.index
        )
        .copy()
    )

    if aligned_nifty_returns.isna().any():
        raise RuntimeError(
            "Missing Nifty 50 returns during "
            f"{variant_name} backtest."
        )

    aligned_nifty_returns.iloc[
        0
    ] = 0.0

    statistics = (
        calculate_backtest_statistics(
            strategy_name=(
                variant_name
            ),
            daily_returns=(
                variant_result[
                    "Net Return"
                ]
            ),
            portfolio_values=(
                variant_result[
                    "Portfolio Value (₹)"
                ]
            ),
            benchmark_returns=(
                aligned_nifty_returns
            ),
            daily_turnover=(
                variant_result[
                    "One-Way Turnover"
                ]
            ),
            daily_costs=(
                variant_result[
                    "Transaction Cost (₹)"
                ]
            ),
        )
    )

    variant_choice_map = (
        variant_history
        .set_index(
            "Signal Date"
        )[
            "Selected Strategy"
        ]
    )

    base_agreement = (
        variant_choice_map
        .reindex(
            base_selection_map.index
        )
        .eq(
            base_selection_map
        )
        .mean()
    )

    selection_counts = (
        variant_history[
            "Selected Strategy"
        ]
        .value_counts()
    )

    robustness_summary_records.append(
        {
            "Variant":
                variant_name,

            "Lookback Trading Days":
                (
                    lookback_days
                    if lookback_days
                    is not None
                    else np.nan
                ),

            "Minimum Same-Regime Observations":
                minimum_observations,

            "Fallback Selections":
                int(
                    variant_history[
                        "Fallback Used"
                    ].sum()
                ),

            "Fallback Rate":
                variant_history[
                    "Fallback Used"
                ].mean(),

            "Agreement with Base Selector":
                base_agreement,

            "1-Month Reversal Selections":
                int(
                    selection_counts.get(
                        "1-Month Reversal",
                        0,
                    )
                ),

            "12-Month Momentum Selections":
                int(
                    selection_counts.get(
                        "12-Month Momentum",
                        0,
                    )
                ),

            "Gradient Boosting Selections":
                int(
                    selection_counts.get(
                        "Gradient Boosting",
                        0,
                    )
                ),

            "Equal-Weight Selections":
                int(
                    selection_counts.get(
                        "Equal-Weight India 10",
                        0,
                    )
                ),

            "Ending Value (₹)":
                statistics[
                    "Ending Value (₹)"
                ],

            "CAGR":
                statistics[
                    "CAGR"
                ],

            "Annualised Volatility":
                statistics[
                    "Annualised Volatility"
                ],

            "Sharpe Ratio":
                statistics[
                    "Sharpe Ratio"
                ],

            "Maximum Drawdown":
                statistics[
                    "Maximum Drawdown"
                ],

            "Annualised Turnover":
                statistics[
                    "Annualised Turnover"
                ],

            "Transaction Costs (₹)":
                statistics[
                    "Transaction Costs (₹)"
                ],
        }
    )

    print(
        "Completed selector variant:",
        variant_name,
    )


selector_variant_history = (
    pd.concat(
        variant_selection_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "Variant",
            "Signal Date",
        ]
    )
    .reset_index(
        drop=True
    )
)

selector_variant_weights = (
    pd.concat(
        variant_weight_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "Variant",
            "Execution Date",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)

selector_robustness_summary = (
    pd.DataFrame(
        robustness_summary_records
    )
    .set_index(
        "Variant"
    )
    .reindex(
        variant_order
    )
)


# ---------------------------------------------------------
# 4. Build selection-count and performance-range summaries
# ---------------------------------------------------------

selector_selection_counts = (
    selector_variant_history
    .pivot_table(
        index="Variant",
        columns="Selected Strategy",
        values="Signal Date",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(
        index=variant_order,
        columns=robustness_candidate_strategies,
        fill_value=0,
    )
)

selector_performance_range = pd.DataFrame(
    {
        "Minimum":
            selector_robustness_summary[
                [
                    "CAGR",
                    "Sharpe Ratio",
                    "Maximum Drawdown",
                    "Annualised Turnover",
                ]
            ].min(),

        "Median":
            selector_robustness_summary[
                [
                    "CAGR",
                    "Sharpe Ratio",
                    "Maximum Drawdown",
                    "Annualised Turnover",
                ]
            ].median(),

        "Maximum":
            selector_robustness_summary[
                [
                    "CAGR",
                    "Sharpe Ratio",
                    "Maximum Drawdown",
                    "Annualised Turnover",
                ]
            ].max(),
    }
)


# ---------------------------------------------------------
# 5. Strictly verify that the base variant is reproduced
# ---------------------------------------------------------

base_variant_name = (
    "Expanding / 63 observations — Base"
)

recreated_base_history = (
    selector_variant_history.loc[
        selector_variant_history[
            "Variant"
        ]
        == base_variant_name
    ]
    .sort_values(
        "Signal Date"
    )
    .reset_index(
        drop=True
    )
)

original_base_history = (
    regime_selector_history
    .sort_values(
        "Signal Date"
    )
    .reset_index(
        drop=True
    )
)

assert (
    recreated_base_history[
        "Selected Strategy"
    ]
    .equals(
        original_base_history[
            "Selected Strategy"
        ]
    )
)

recreated_base_weights = (
    selector_variant_weights.loc[
        selector_variant_weights[
            "Variant"
        ]
        == base_variant_name
    ]
    .pivot(
        index="Signal Date",
        columns="Ticker",
        values="Target Weight",
    )
    .reindex(
        columns=INDIA_10_TICKERS
    )
    .sort_index()
)

original_base_weights = (
    regime_aware_strategy_weights
    .pivot(
        index="Signal Date",
        columns="Ticker",
        values="Target Weight",
    )
    .reindex(
        columns=INDIA_10_TICKERS
    )
    .sort_index()
)

assert recreated_base_weights.index.equals(
    original_base_weights.index
)

assert np.allclose(
    recreated_base_weights,
    original_base_weights,
    atol=1e-12,
)

assert np.isclose(
    selector_robustness_summary.loc[
        base_variant_name,
        "Ending Value (₹)",
    ],
    regime_aware_backtest_summary.loc[
        REGIME_AWARE_STRATEGY_NAME,
        "Ending Value (₹)",
    ],
    rtol=1e-10,
)

assert np.isclose(
    selector_robustness_summary.loc[
        base_variant_name,
        "CAGR",
    ],
    regime_aware_backtest_summary.loc[
        REGIME_AWARE_STRATEGY_NAME,
        "CAGR",
    ],
    rtol=1e-10,
)


# ---------------------------------------------------------
# 6. General leakage and data-quality validation
# ---------------------------------------------------------

variant_weight_totals = (
    selector_variant_weights
    .groupby(
        [
            "Variant",
            "Signal Date",
            "Execution Date",
        ]
    )[
        "Target Weight"
    ]
    .sum()
)

assert (
    selector_variant_history[
        "Execution Date"
    ]
    > selector_variant_history[
        "Signal Date"
    ]
).all()

assert np.allclose(
    variant_weight_totals,
    1.0,
    atol=1e-10,
)

assert (
    selector_variant_weights[
        "Target Weight"
    ]
    .ge(
        0
    )
    .all()
)

assert (
    selector_variant_history
    .groupby(
        "Variant"
    )[
        "Signal Date"
    ]
    .nunique()
    .eq(
        101
    )
    .all()
)

assert (
    selector_robustness_summary[
        "Ending Value (₹)"
    ]
    .gt(
        0
    )
    .all()
)

assert np.isfinite(
    selector_robustness_summary[
        [
            "CAGR",
            "Annualised Volatility",
            "Sharpe Ratio",
            "Maximum Drawdown",
        ]
    ]
    .to_numpy()
).all()


# ---------------------------------------------------------
# 7. Save robustness datasets
# ---------------------------------------------------------

selector_variant_history.to_csv(
    PROCESSED_DATA_DIR
    / "regime_selector_variant_history.csv",
    index=False,
)

selector_variant_weights.to_csv(
    PROCESSED_DATA_DIR
    / "regime_selector_variant_weights.csv",
    index=False,
)

selector_robustness_summary.to_csv(
    PROCESSED_DATA_DIR
    / "regime_selector_robustness_summary.csv",
)

selector_selection_counts.to_csv(
    PROCESSED_DATA_DIR
    / "regime_selector_selection_counts.csv",
)

selector_performance_range.to_csv(
    PROCESSED_DATA_DIR
    / "regime_selector_performance_range.csv",
)


# ---------------------------------------------------------
# 8. Display results
# ---------------------------------------------------------

display_robustness_summary = (
    selector_robustness_summary.copy()
)

for column in [
    "Fallback Rate",
    "Agreement with Base Selector",
    "CAGR",
    "Annualised Volatility",
    "Maximum Drawdown",
    "Annualised Turnover",
]:

    display_robustness_summary[
        column
    ] = (
        display_robustness_summary[
            column
        ]
        * 100
    )


display_performance_range = (
    selector_performance_range.copy()
)

for row_name in [
    "CAGR",
    "Annualised Volatility",
    "Maximum Drawdown",
    "Annualised Turnover",
]:

    display_performance_range.loc[
        row_name
    ] = (
        display_performance_range.loc[
            row_name
        ]
        * 100
    )


print("REGIME-SELECTOR ROBUSTNESS AUDIT")
print("=" * 72)
print(
    "Selector variants:",
    len(
        selector_variant_specs
    ),
)
print(
    "Signal dates per variant:",
    101,
)
print(
    "Candidate strategies:",
    len(
        robustness_candidate_strategies
    ),
)
print(
    "Transaction cost:",
    f"{ONE_WAY_TRANSACTION_COST:.2%}",
)
print(
    "Base-selector reproduction:",
    "PASSED",
)
print(
    "Prior-date data only:",
    "PASSED",
)
print(
    "One-trading-day execution lag:",
    "PASSED",
)
print(
    "All portfolio weights sum to 100%:",
    "PASSED",
)
print(
    "Selector robustness audit:",
    "PASSED",
)

print("\nSELECTOR ROBUSTNESS SUMMARY")
display(
    display_robustness_summary.round(
        2
    )
)

print("\nSTRATEGY SELECTION COUNTS")
display(
    selector_selection_counts
)

print("\nPERFORMANCE RANGE ACROSS SELECTOR VARIANTS")
display(
    display_performance_range.round(
        2
    )
)

Completed selector variant: Expanding / 42 observations
Completed selector variant: Expanding / 63 observations — Base
Completed selector variant: Expanding / 126 observations
Completed selector variant: Trailing 504 days / 42 observations
Completed selector variant: Trailing 756 days / 63 observations
Completed selector variant: Trailing 1,008 days / 63 observations


KeyError: 'Annualised Volatility'

In [26]:
# =========================================================
# REPAIR STEP 58 PERFORMANCE-RANGE OUTPUT
# =========================================================

# Recreate the range table with Annualised Volatility included.
selector_performance_range = pd.DataFrame(
    {
        "Minimum":
            selector_robustness_summary[
                [
                    "CAGR",
                    "Annualised Volatility",
                    "Sharpe Ratio",
                    "Maximum Drawdown",
                    "Annualised Turnover",
                ]
            ].min(),

        "Median":
            selector_robustness_summary[
                [
                    "CAGR",
                    "Annualised Volatility",
                    "Sharpe Ratio",
                    "Maximum Drawdown",
                    "Annualised Turnover",
                ]
            ].median(),

        "Maximum":
            selector_robustness_summary[
                [
                    "CAGR",
                    "Annualised Volatility",
                    "Sharpe Ratio",
                    "Maximum Drawdown",
                    "Annualised Turnover",
                ]
            ].max(),
    }
)


# Save the corrected table.
selector_performance_range.to_csv(
    PROCESSED_DATA_DIR
    / "regime_selector_performance_range.csv",
)


# Prepare the selector summary for display.
display_robustness_summary = (
    selector_robustness_summary.copy()
)

for column in [
    "Fallback Rate",
    "Agreement with Base Selector",
    "CAGR",
    "Annualised Volatility",
    "Maximum Drawdown",
    "Annualised Turnover",
]:

    display_robustness_summary[
        column
    ] = (
        display_robustness_summary[
            column
        ]
        * 100
    )


# Prepare the performance-range table for display.
display_performance_range = (
    selector_performance_range.copy()
)

for row_name in [
    "CAGR",
    "Annualised Volatility",
    "Maximum Drawdown",
    "Annualised Turnover",
]:

    display_performance_range.loc[
        row_name
    ] = (
        display_performance_range.loc[
            row_name
        ]
        * 100
    )


# Final validation.
assert (
    "Annualised Volatility"
    in selector_performance_range.index
)

assert np.isfinite(
    selector_performance_range.to_numpy()
).all()


print("REGIME-SELECTOR ROBUSTNESS AUDIT")
print("=" * 72)
print(
    "Selector variants:",
    len(
        selector_variant_specs
    ),
)
print(
    "Signal dates per variant:",
    101,
)
print(
    "Candidate strategies:",
    len(
        robustness_candidate_strategies
    ),
)
print(
    "Transaction cost:",
    f"{ONE_WAY_TRANSACTION_COST:.2%}",
)
print(
    "Base-selector reproduction:",
    "PASSED",
)
print(
    "Prior-date data only:",
    "PASSED",
)
print(
    "One-trading-day execution lag:",
    "PASSED",
)
print(
    "All portfolio weights sum to 100%:",
    "PASSED",
)
print(
    "Performance-range repair:",
    "PASSED",
)
print(
    "Selector robustness audit:",
    "PASSED",
)

print("\nSELECTOR ROBUSTNESS SUMMARY")
display(
    display_robustness_summary.round(
        2
    )
)

print("\nSTRATEGY SELECTION COUNTS")
display(
    selector_selection_counts
)

print("\nPERFORMANCE RANGE ACROSS SELECTOR VARIANTS")
display(
    display_performance_range.round(
        2
    )
)

REGIME-SELECTOR ROBUSTNESS AUDIT
Selector variants: 6
Signal dates per variant: 101
Candidate strategies: 4
Transaction cost: 0.15%
Base-selector reproduction: PASSED
Prior-date data only: PASSED
One-trading-day execution lag: PASSED
All portfolio weights sum to 100%: PASSED
Performance-range repair: PASSED
Selector robustness audit: PASSED

SELECTOR ROBUSTNESS SUMMARY


,Lookback Trading Days,Minimum Same-Regime Observations,Fallback Selections,Fallback Rate,Agreement with Base Selector,1-Month Reversal Selections,12-Month Momentum Selections,Gradient Boosting Selections,Equal-Weight Selections,Ending Value (₹),CAGR,Annualised Volatility,Sharpe Ratio,Maximum Drawdown,Annualised Turnover,Transaction Costs (₹)
Variant,,,,,,,,,,,,,,,,
Expanding / 42 observations,NaN,42,9,8.91,95.05,44,43,5,9,11612329.89,34.25,21.42,1.22,-27.69,666.58,351489.80
Expanding / 63 observations — Base,NaN,63,14,13.86,100.00,41,41,5,14,11752022.41,34.44,21.15,1.24,-27.69,652.35,354506.83
Expanding / 126 observations,NaN,126,26,25.74,88.12,36,37,2,26,11747625.07,34.43,20.63,1.26,-28.90,571.35,341432.10
Trailing 504 days / 42 observations,504.0,42,12,11.88,66.34,45,26,9,21,8331520.91,29.00,19.91,1.09,-27.69,682.64,295401.21
Trailing 756 days / 63 observations,756.0,63,17,16.83,76.24,48,22,7,24,7562522.19,27.51,19.62,1.04,-27.69,733.40,333698.40
"Trailing 1,008 days / 63 observations",1008.0,63,15,14.85,83.17,44,28,6,23,8456905.81,29.23,19.96,1.10,-27.69,706.50,324327.59



STRATEGY SELECTION COUNTS


Selected Strategy,1-Month Reversal,12-Month Momentum,Gradient Boosting,Equal-Weight India 10
Variant,,,,
Expanding / 42 observations,44,43,5,9
Expanding / 63 observations — Base,41,41,5,14
Expanding / 126 observations,36,37,2,26
Trailing 504 days / 42 observations,45,26,9,21
Trailing 756 days / 63 observations,48,22,7,24
"Trailing 1,008 days / 63 observations",44,28,6,23



PERFORMANCE RANGE ACROSS SELECTOR VARIANTS


,Minimum,Median,Maximum
CAGR,27.51,31.74,34.44
Annualised Volatility,19.62,20.29,21.42
Sharpe Ratio,1.04,1.16,1.26
Maximum Drawdown,-28.90,-27.69,-27.69
Annualised Turnover,571.35,674.61,733.40


## Indian Financial-Year and Rolling Three-Year Stability

This section tests whether strategy performance is consistent through time.

The analysis includes:

- Indian financial years running from April to March
- Complete and partial financial-year identification
- Financial-year returns, volatility, Sharpe ratio and maximum drawdown
- Frequency of outperforming the Nifty 50
- Frequency of outperforming the equal-weight India 10 portfolio
- Rolling 756-trading-day performance, representing approximately three years
- Worst, median and best rolling three-year outcomes

All returns are transaction-cost adjusted and use the same out-of-sample backtest histories.

In [27]:
# =========================================================
# INDIAN FINANCIAL-YEAR AND ROLLING 3-YEAR STABILITY
# =========================================================

required_objects = [
    "final_strategy_daily_net_returns",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required previous-step objects are missing:\n"
        + "\n".join(
            missing_objects
        )
    )


# ---------------------------------------------------------
# 1. Select the main strategies for stability analysis
# ---------------------------------------------------------

stability_strategy_names = [
    "Regime-Aware Walk-Forward",
    "1-Month Reversal",
    "12-Month Momentum",
    "Gradient Boosting",
    "Equal-Weight India 10",
    "Nifty 50",
]

missing_stability_strategies = sorted(
    set(
        stability_strategy_names
    )
    - set(
        final_strategy_daily_net_returns.columns
    )
)

if missing_stability_strategies:
    raise RuntimeError(
        "Required strategy histories are missing:\n"
        + "\n".join(
            missing_stability_strategies
        )
    )


stability_daily_returns = (
    final_strategy_daily_net_returns[
        stability_strategy_names
    ]
    .copy()
)

stability_daily_returns.index = pd.to_datetime(
    stability_daily_returns.index
)

stability_daily_returns.index.name = "Date"

stability_daily_returns = (
    stability_daily_returns
    .sort_index()
)


daily_risk_free_rate_stability = (
    (
        1
        + RISK_FREE_RATE
    )
    ** (
        1
        / TRADING_DAYS_PER_YEAR
    )
    - 1
)


# ---------------------------------------------------------
# 2. Assign Indian financial-year labels
# ---------------------------------------------------------

def get_indian_financial_year_start(
    trading_date,
):

    return (
        trading_date.year
        if trading_date.month >= 4
        else trading_date.year - 1
    )


def format_indian_financial_year(
    starting_year,
):

    return (
        f"FY{starting_year}-"
        f"{str(starting_year + 1)[-2:]}"
    )


financial_year_start_series = pd.Series(
    [
        get_indian_financial_year_start(
            trading_date
        )
        for trading_date in (
            stability_daily_returns.index
        )
    ],
    index=stability_daily_returns.index,
    name="Financial Year Start",
)

financial_year_label_series = (
    financial_year_start_series
    .map(
        format_indian_financial_year
    )
    .rename(
        "Financial Year"
    )
)


financial_year_calendar = pd.DataFrame(
    {
        "Financial Year":
            financial_year_label_series,

        "Financial Year Start":
            financial_year_start_series,
    },
    index=stability_daily_returns.index,
)

financial_year_calendar_summary = (
    financial_year_calendar
    .reset_index()
    .groupby(
        [
            "Financial Year",
            "Financial Year Start",
        ],
        as_index=False,
    )
    .agg(
        First_Trading_Date=(
            "Date",
            "min",
        ),

        Last_Trading_Date=(
            "Date",
            "max",
        ),

        Trading_Days=(
            "Date",
            "count",
        ),
    )
    .sort_values(
        "Financial Year Start"
    )
    .reset_index(
        drop=True
    )
)

financial_year_calendar_summary[
    "Complete Financial Year"
] = (
    financial_year_calendar_summary[
        "First_Trading_Date"
    ].dt.month.eq(
        4
    )
    & financial_year_calendar_summary[
        "Last_Trading_Date"
    ].dt.month.eq(
        3
    )
    & financial_year_calendar_summary[
        "Trading_Days"
    ].ge(
        220
    )
)

complete_financial_year_map = (
    financial_year_calendar_summary
    .set_index(
        "Financial Year"
    )[
        "Complete Financial Year"
    ]
    .to_dict()
)


# ---------------------------------------------------------
# 3. Calculate performance within each financial year
# ---------------------------------------------------------

financial_year_performance_records = []

for financial_year in (
    financial_year_calendar_summary[
        "Financial Year"
    ]
):

    financial_year_dates = (
        financial_year_label_series.index[
            financial_year_label_series
            == financial_year
        ]
    )

    financial_year_returns = (
        stability_daily_returns
        .loc[
            financial_year_dates
        ]
    )

    for strategy_name in (
        stability_strategy_names
    ):

        strategy_returns = (
            financial_year_returns[
                strategy_name
            ]
            .dropna()
        )

        observations = len(
            strategy_returns
        )

        if observations == 0:
            continue

        period_return = (
            (
                1
                + strategy_returns
            )
            .prod()
            - 1
        )

        elapsed_years = (
            (
                strategy_returns.index.max()
                - strategy_returns.index.min()
            ).days
            / 365.25
        )

        annualised_return = (
            (
                1
                + period_return
            )
            ** (
                1
                / elapsed_years
            )
            - 1
            if elapsed_years > 0
            else np.nan
        )

        annualised_volatility = (
            strategy_returns.std(
                ddof=1
            )
            * np.sqrt(
                TRADING_DAYS_PER_YEAR
            )
        )

        annualised_excess_return = (
            (
                strategy_returns.mean()
                - daily_risk_free_rate_stability
            )
            * TRADING_DAYS_PER_YEAR
        )

        sharpe_ratio = (
            annualised_excess_return
            / annualised_volatility
            if annualised_volatility > 0
            else np.nan
        )

        financial_year_wealth = (
            (
                1
                + strategy_returns
            )
            .cumprod()
        )

        maximum_drawdown = (
            financial_year_wealth
            / financial_year_wealth.cummax()
            - 1
        ).min()

        financial_year_performance_records.append(
            {
                "Financial Year":
                    financial_year,

                "Financial Year Start":
                    get_indian_financial_year_start(
                        strategy_returns.index[
                            0
                        ]
                    ),

                "Complete Financial Year":
                    complete_financial_year_map[
                        financial_year
                    ],

                "Strategy":
                    strategy_name,

                "First Trading Date":
                    strategy_returns.index.min(),

                "Last Trading Date":
                    strategy_returns.index.max(),

                "Trading Days":
                    observations,

                "Period Return":
                    period_return,

                "Annualised Return":
                    annualised_return,

                "Annualised Volatility":
                    annualised_volatility,

                "Sharpe Ratio":
                    sharpe_ratio,

                "Maximum Drawdown":
                    maximum_drawdown,

                "Positive Day Rate":
                    strategy_returns.gt(
                        0
                    ).mean(),
            }
        )


financial_year_performance = (
    pd.DataFrame(
        financial_year_performance_records
    )
    .sort_values(
        [
            "Financial Year Start",
            "Strategy",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 4. Add return comparisons against benchmarks
# ---------------------------------------------------------

nifty_financial_year_returns = (
    financial_year_performance.loc[
        financial_year_performance[
            "Strategy"
        ]
        == "Nifty 50"
    ]
    .set_index(
        "Financial Year"
    )[
        "Period Return"
    ]
)

equal_weight_financial_year_returns = (
    financial_year_performance.loc[
        financial_year_performance[
            "Strategy"
        ]
        == "Equal-Weight India 10"
    ]
    .set_index(
        "Financial Year"
    )[
        "Period Return"
    ]
)

financial_year_performance[
    "Return vs Nifty"
] = (
    financial_year_performance[
        "Period Return"
    ]
    - financial_year_performance[
        "Financial Year"
    ]
    .map(
        nifty_financial_year_returns
    )
)

financial_year_performance[
    "Return vs Equal Weight"
] = (
    financial_year_performance[
        "Period Return"
    ]
    - financial_year_performance[
        "Financial Year"
    ]
    .map(
        equal_weight_financial_year_returns
    )
)


financial_year_return_pivot = (
    financial_year_performance
    .pivot(
        index="Financial Year",
        columns="Strategy",
        values="Period Return",
    )
    .reindex(
        columns=stability_strategy_names
    )
)


# ---------------------------------------------------------
# 5. Summarise complete Indian financial years
# ---------------------------------------------------------

complete_financial_year_performance = (
    financial_year_performance.loc[
        financial_year_performance[
            "Complete Financial Year"
        ]
    ]
    .copy()
)

financial_year_stability_summary = (
    complete_financial_year_performance
    .groupby(
        "Strategy"
    )
    .agg(
        Complete_Financial_Years=(
            "Financial Year",
            "nunique",
        ),

        Average_FY_Return=(
            "Period Return",
            "mean",
        ),

        Median_FY_Return=(
            "Period Return",
            "median",
        ),

        Worst_FY_Return=(
            "Period Return",
            "min",
        ),

        Best_FY_Return=(
            "Period Return",
            "max",
        ),

        Positive_FY_Rate=(
            "Period Return",
            lambda values: (
                values
                > 0
            ).mean(),
        ),

        Beats_Nifty_FY_Rate=(
            "Return vs Nifty",
            lambda values: (
                values
                > 0
            ).mean(),
        ),

        Beats_Equal_Weight_FY_Rate=(
            "Return vs Equal Weight",
            lambda values: (
                values
                > 0
            ).mean(),
        ),

        Average_FY_Sharpe=(
            "Sharpe Ratio",
            "mean",
        ),

        Worst_FY_Sharpe=(
            "Sharpe Ratio",
            "min",
        ),

        Worst_FY_Drawdown=(
            "Maximum Drawdown",
            "min",
        ),
    )
    .reindex(
        stability_strategy_names
    )
)


# ---------------------------------------------------------
# 6. Calculate rolling 756-trading-day performance
# ---------------------------------------------------------

ROLLING_WINDOW_TRADING_DAYS = 756

monthly_evaluation_dates = (
    pd.Series(
        stability_daily_returns.index,
        index=stability_daily_returns.index,
    )
    .groupby(
        stability_daily_returns
        .index
        .to_period(
            "M"
        )
    )
    .max()
    .tolist()
)

rolling_performance_records = []

for window_end_date in monthly_evaluation_dates:

    available_dates = (
        stability_daily_returns.index[
            stability_daily_returns.index
            <= window_end_date
        ]
    )

    if len(
        available_dates
    ) < ROLLING_WINDOW_TRADING_DAYS:
        continue

    window_dates = (
        available_dates[
            -ROLLING_WINDOW_TRADING_DAYS:
        ]
    )

    window_start_date = (
        window_dates[
            0
        ]
    )

    window_returns = (
        stability_daily_returns.loc[
            window_dates
        ]
    )

    elapsed_years = (
        (
            window_end_date
            - window_start_date
        ).days
        / 365.25
    )

    for strategy_name in (
        stability_strategy_names
    ):

        strategy_returns = (
            window_returns[
                strategy_name
            ]
            .dropna()
        )

        cumulative_return = (
            (
                1
                + strategy_returns
            )
            .prod()
            - 1
        )

        rolling_cagr = (
            (
                1
                + cumulative_return
            )
            ** (
                1
                / elapsed_years
            )
            - 1
        )

        rolling_volatility = (
            strategy_returns.std(
                ddof=1
            )
            * np.sqrt(
                TRADING_DAYS_PER_YEAR
            )
        )

        rolling_excess_return = (
            (
                strategy_returns.mean()
                - daily_risk_free_rate_stability
            )
            * TRADING_DAYS_PER_YEAR
        )

        rolling_sharpe = (
            rolling_excess_return
            / rolling_volatility
            if rolling_volatility > 0
            else np.nan
        )

        rolling_wealth = (
            (
                1
                + strategy_returns
            )
            .cumprod()
        )

        rolling_maximum_drawdown = (
            rolling_wealth
            / rolling_wealth.cummax()
            - 1
        ).min()

        rolling_performance_records.append(
            {
                "Window Start":
                    window_start_date,

                "Window End":
                    window_end_date,

                "Trading Days":
                    len(
                        strategy_returns
                    ),

                "Strategy":
                    strategy_name,

                "Rolling CAGR":
                    rolling_cagr,

                "Rolling Annualised Volatility":
                    rolling_volatility,

                "Rolling Sharpe Ratio":
                    rolling_sharpe,

                "Rolling Maximum Drawdown":
                    rolling_maximum_drawdown,
            }
        )


rolling_3y_performance = (
    pd.DataFrame(
        rolling_performance_records
    )
    .sort_values(
        [
            "Window End",
            "Strategy",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 7. Add rolling benchmark comparisons
# ---------------------------------------------------------

rolling_nifty_cagr = (
    rolling_3y_performance.loc[
        rolling_3y_performance[
            "Strategy"
        ]
        == "Nifty 50"
    ]
    .set_index(
        "Window End"
    )[
        "Rolling CAGR"
    ]
)

rolling_equal_weight_cagr = (
    rolling_3y_performance.loc[
        rolling_3y_performance[
            "Strategy"
        ]
        == "Equal-Weight India 10"
    ]
    .set_index(
        "Window End"
    )[
        "Rolling CAGR"
    ]
)

rolling_3y_performance[
    "Rolling CAGR vs Nifty"
] = (
    rolling_3y_performance[
        "Rolling CAGR"
    ]
    - rolling_3y_performance[
        "Window End"
    ]
    .map(
        rolling_nifty_cagr
    )
)

rolling_3y_performance[
    "Rolling CAGR vs Equal Weight"
] = (
    rolling_3y_performance[
        "Rolling CAGR"
    ]
    - rolling_3y_performance[
        "Window End"
    ]
    .map(
        rolling_equal_weight_cagr
    )
)


rolling_3y_stability_summary = (
    rolling_3y_performance
    .groupby(
        "Strategy"
    )
    .agg(
        Rolling_Windows=(
            "Window End",
            "nunique",
        ),

        Minimum_Rolling_CAGR=(
            "Rolling CAGR",
            "min",
        ),

        Median_Rolling_CAGR=(
            "Rolling CAGR",
            "median",
        ),

        Maximum_Rolling_CAGR=(
            "Rolling CAGR",
            "max",
        ),

        Positive_Rolling_CAGR_Rate=(
            "Rolling CAGR",
            lambda values: (
                values
                > 0
            ).mean(),
        ),

        Beats_Nifty_Rolling_Rate=(
            "Rolling CAGR vs Nifty",
            lambda values: (
                values
                > 0
            ).mean(),
        ),

        Beats_Equal_Weight_Rolling_Rate=(
            "Rolling CAGR vs Equal Weight",
            lambda values: (
                values
                > 0
            ).mean(),
        ),

        Minimum_Rolling_Sharpe=(
            "Rolling Sharpe Ratio",
            "min",
        ),

        Median_Rolling_Sharpe=(
            "Rolling Sharpe Ratio",
            "median",
        ),

        Maximum_Rolling_Sharpe=(
            "Rolling Sharpe Ratio",
            "max",
        ),

        Worst_Rolling_Drawdown=(
            "Rolling Maximum Drawdown",
            "min",
        ),
    )
    .reindex(
        stability_strategy_names
    )
)


# ---------------------------------------------------------
# 8. Validate stability analysis
# ---------------------------------------------------------

complete_financial_year_count = int(
    financial_year_calendar_summary[
        "Complete Financial Year"
    ].sum()
)

assert complete_financial_year_count >= 8

assert (
    financial_year_stability_summary[
        "Complete_Financial_Years"
    ]
    .eq(
        complete_financial_year_count
    )
    .all()
)

assert (
    rolling_3y_performance[
        "Trading Days"
    ]
    .eq(
        ROLLING_WINDOW_TRADING_DAYS
    )
    .all()
)

assert (
    rolling_3y_stability_summary[
        "Rolling_Windows"
    ]
    .nunique()
    == 1
)

assert (
    rolling_3y_stability_summary[
        "Rolling_Windows"
    ]
    .iloc[
        0
    ]
    >= 50
)

assert np.isfinite(
    financial_year_performance[
        [
            "Period Return",
            "Annualised Volatility",
            "Sharpe Ratio",
            "Maximum Drawdown",
        ]
    ]
    .to_numpy()
).all()

assert np.isfinite(
    rolling_3y_performance[
        [
            "Rolling CAGR",
            "Rolling Annualised Volatility",
            "Rolling Sharpe Ratio",
            "Rolling Maximum Drawdown",
        ]
    ]
    .to_numpy()
).all()


# ---------------------------------------------------------
# 9. Save stability outputs
# ---------------------------------------------------------

financial_year_calendar_summary.to_csv(
    PROCESSED_DATA_DIR
    / "indian_financial_year_calendar.csv",
    index=False,
)

financial_year_performance.to_csv(
    PROCESSED_DATA_DIR
    / "indian_financial_year_performance.csv",
    index=False,
)

financial_year_return_pivot.to_csv(
    PROCESSED_DATA_DIR
    / "indian_financial_year_return_pivot.csv",
)

financial_year_stability_summary.to_csv(
    PROCESSED_DATA_DIR
    / "indian_financial_year_stability_summary.csv",
)

rolling_3y_performance.to_csv(
    PROCESSED_DATA_DIR
    / "rolling_3y_strategy_performance.csv",
    index=False,
)

rolling_3y_stability_summary.to_csv(
    PROCESSED_DATA_DIR
    / "rolling_3y_strategy_stability_summary.csv",
)


# ---------------------------------------------------------
# 10. Display results
# ---------------------------------------------------------

display_financial_year_returns = (
    financial_year_return_pivot.copy()
    * 100
)

display_financial_year_summary = (
    financial_year_stability_summary.copy()
)

for column in [
    "Average_FY_Return",
    "Median_FY_Return",
    "Worst_FY_Return",
    "Best_FY_Return",
    "Positive_FY_Rate",
    "Beats_Nifty_FY_Rate",
    "Beats_Equal_Weight_FY_Rate",
    "Worst_FY_Drawdown",
]:

    display_financial_year_summary[
        column
    ] = (
        display_financial_year_summary[
            column
        ]
        * 100
    )


display_rolling_summary = (
    rolling_3y_stability_summary.copy()
)

for column in [
    "Minimum_Rolling_CAGR",
    "Median_Rolling_CAGR",
    "Maximum_Rolling_CAGR",
    "Positive_Rolling_CAGR_Rate",
    "Beats_Nifty_Rolling_Rate",
    "Beats_Equal_Weight_Rolling_Rate",
    "Worst_Rolling_Drawdown",
]:

    display_rolling_summary[
        column
    ] = (
        display_rolling_summary[
            column
        ]
        * 100
    )


print("INDIAN FINANCIAL-YEAR AND ROLLING 3-YEAR STABILITY")
print("=" * 72)
print(
    "Analysis period:",
    stability_daily_returns
    .index.min()
    .date(),
    "to",
    stability_daily_returns
    .index.max()
    .date(),
)
print(
    "Financial years observed:",
    financial_year_calendar_summary[
        "Financial Year"
    ].nunique(),
)
print(
    "Complete financial years:",
    complete_financial_year_count,
)
print(
    "Partial financial years:",
    int(
        (
            ~financial_year_calendar_summary[
                "Complete Financial Year"
            ]
        ).sum()
    ),
)
print(
    "Rolling window:",
    ROLLING_WINDOW_TRADING_DAYS,
    "trading days",
)
print(
    "Rolling evaluation windows:",
    rolling_3y_stability_summary[
        "Rolling_Windows"
    ]
    .iloc[
        0
    ],
)
print(
    "Strategies analysed:",
    len(
        stability_strategy_names
    ),
)
print(
    "Financial-year validation:",
    "PASSED",
)
print(
    "Rolling-period validation:",
    "PASSED",
)
print(
    "Time-stability analysis:",
    "PASSED",
)

print("\nFINANCIAL-YEAR RETURNS (%)")
display(
    display_financial_year_returns.round(
        2
    )
)

print("\nCOMPLETE FINANCIAL-YEAR STABILITY")
display(
    display_financial_year_summary.round(
        2
    )
)

print("\nROLLING 3-YEAR STABILITY")
display(
    display_rolling_summary.round(
        2
    )
)

INDIAN FINANCIAL-YEAR AND ROLLING 3-YEAR STABILITY
Analysis period: 2018-04-02 to 2026-07-30
Financial years observed: 9
Complete financial years: 8
Partial financial years: 1
Rolling window: 756 trading days
Rolling evaluation windows: 64
Strategies analysed: 6
Financial-year validation: PASSED
Rolling-period validation: PASSED
Time-stability analysis: PASSED

FINANCIAL-YEAR RETURNS (%)


Strategy,Regime-Aware Walk-Forward,1-Month Reversal,12-Month Momentum,Gradient Boosting,Equal-Weight India 10,Nifty 50
Financial Year,,,,,,
FY2018-19,1.43,8.49,11.69,-6.38,4.09,13.30
FY2019-20,5.32,-6.44,15.39,-12.29,-8.56,-25.69
FY2020-21,98.95,124.92,11.39,60.02,68.87,70.87
FY2021-22,28.55,50.76,36.09,29.63,31.03,18.88
FY2022-23,24.80,29.29,8.23,18.28,16.84,-0.60
FY2023-24,103.27,60.41,101.58,51.83,58.89,28.61
FY2024-25,33.63,8.70,27.19,43.21,21.45,5.34
FY2025-26,9.81,-4.00,5.57,3.51,-7.48,-5.05
FY2026-27,15.56,13.58,9.33,29.44,9.61,8.89



COMPLETE FINANCIAL-YEAR STABILITY


,Complete_Financial_Years,Average_FY_Return,Median_FY_Return,Worst_FY_Return,Best_FY_Return,Positive_FY_Rate,Beats_Nifty_FY_Rate,Beats_Equal_Weight_FY_Rate,Average_FY_Sharpe,Worst_FY_Sharpe,Worst_FY_Drawdown
Strategy,,,,,,,,,,,
Regime-Aware Walk-Forward,8,38.22,26.67,1.43,103.27,100.0,87.5,75.0,1.18,-0.26,-27.69
1-Month Reversal,8,34.02,19.00,-6.44,124.92,75.0,87.5,87.5,1.15,-0.43,-33.56
12-Month Momentum,8,27.14,13.54,5.57,101.58,100.0,75.0,75.0,0.89,0.05,-30.47
Gradient Boosting,8,23.47,23.95,-12.29,60.02,75.0,75.0,37.5,0.78,-0.62,-30.39
Equal-Weight India 10,8,23.14,19.15,-8.56,68.87,75.0,62.5,0.0,0.97,-0.93,-30.47
Nifty 50,8,13.21,9.32,-25.69,70.87,62.5,0.0,37.5,0.41,-1.21,-38.44



ROLLING 3-YEAR STABILITY


,Rolling_Windows,Minimum_Rolling_CAGR,Median_Rolling_CAGR,Maximum_Rolling_CAGR,Positive_Rolling_CAGR_Rate,Beats_Nifty_Rolling_Rate,Beats_Equal_Weight_Rolling_Rate,Minimum_Rolling_Sharpe,Median_Rolling_Sharpe,Maximum_Rolling_Sharpe,Worst_Rolling_Drawdown
Strategy,,,,,,,,,,,
Regime-Aware Walk-Forward,64,29.14,47.68,62.00,100.0,100.00,100.00,0.98,1.74,2.17,-27.69
1-Month Reversal,64,16.76,43.58,65.49,100.0,100.00,87.50,0.61,1.49,2.34,-33.56
12-Month Momentum,64,11.23,32.61,54.74,100.0,79.69,50.00,0.30,1.22,1.88,-30.47
Gradient Boosting,64,7.63,32.35,46.08,100.0,95.31,45.31,0.17,1.25,1.81,-30.39
Equal-Weight India 10,64,16.36,29.13,39.84,100.0,100.00,0.00,0.56,1.37,2.05,-30.47
Nifty 50,64,7.65,14.17,26.26,100.0,0.00,0.00,0.16,0.51,1.16,-38.44


## Block-Bootstrap Statistical Significance Audit

Strong historical performance does not automatically prove that a strategy has genuine predictive value.

This section tests whether each strategy’s daily return advantage over the Nifty 50 and equal-weight India 10 portfolio is statistically distinguishable from zero.

Methodology:

- Paired daily net returns
- Circular moving-block bootstrap
- 21-trading-day blocks to preserve short-term return dependence
- 3,000 bootstrap simulations
- 95% confidence intervals for annualised excess return
- One-sided hypothesis test for positive excess return
- Benjamini–Hochberg false-discovery-rate adjustment across comparisons

The test remains conditional on the strategies and specifications already researched. It reduces the risk of mistaking sampling noise for skill but does not completely eliminate researcher-selection or data-mining bias.

In [28]:
# =========================================================
# BLOCK-BOOTSTRAP STATISTICAL SIGNIFICANCE AUDIT
# =========================================================

required_objects = [
    "final_strategy_daily_net_returns",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required previous-step objects are missing:\n"
        + "\n".join(
            missing_objects
        )
    )


# ---------------------------------------------------------
# 1. Configure the statistical audit
# ---------------------------------------------------------

BOOTSTRAP_ITERATIONS = 3_000
BOOTSTRAP_BLOCK_LENGTH = 21
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95

audit_strategy_names = [
    "Regime-Aware Walk-Forward",
    "1-Month Reversal",
    "12-Month Momentum",
    "Gradient Boosting",
    "Equal-Weight India 10",
    "Nifty 50",
]

missing_audit_strategies = sorted(
    set(
        audit_strategy_names
    )
    - set(
        final_strategy_daily_net_returns.columns
    )
)

if missing_audit_strategies:
    raise RuntimeError(
        "Required strategy histories are missing:\n"
        + "\n".join(
            missing_audit_strategies
        )
    )


audit_daily_returns = (
    final_strategy_daily_net_returns[
        audit_strategy_names
    ]
    .copy()
    .sort_index()
)

audit_daily_returns.index = pd.to_datetime(
    audit_daily_returns.index
)

audit_daily_returns.index.name = "Date"


comparison_definitions = []

# Compare investable strategies with the Nifty 50.
for strategy_name in [
    "Regime-Aware Walk-Forward",
    "1-Month Reversal",
    "12-Month Momentum",
    "Gradient Boosting",
    "Equal-Weight India 10",
]:

    comparison_definitions.append(
        {
            "Strategy":
                strategy_name,

            "Benchmark":
                "Nifty 50",
        }
    )

# Compare concentrated strategies with equal-weight India 10.
for strategy_name in [
    "Regime-Aware Walk-Forward",
    "1-Month Reversal",
    "12-Month Momentum",
    "Gradient Boosting",
]:

    comparison_definitions.append(
        {
            "Strategy":
                strategy_name,

            "Benchmark":
                "Equal-Weight India 10",
        }
    )


# ---------------------------------------------------------
# 2. Create circular moving-block bootstrap indices
# ---------------------------------------------------------

def create_circular_block_indices(
    number_of_observations,
    block_length,
    bootstrap_iterations,
    random_seed,
):

    if number_of_observations <= block_length:
        raise ValueError(
            "The return history must be longer than "
            "the selected bootstrap block."
        )

    random_generator = np.random.default_rng(
        random_seed
    )

    blocks_required = int(
        np.ceil(
            number_of_observations
            / block_length
        )
    )

    block_start_positions = (
        random_generator.integers(
            low=0,
            high=number_of_observations,
            size=(
                bootstrap_iterations,
                blocks_required,
            ),
        )
    )

    block_offsets = np.arange(
        block_length
    )

    bootstrap_indices = (
        (
            block_start_positions[
                :,
                :,
                None,
            ]
            + block_offsets[
                None,
                None,
                :,
            ]
        )
        % number_of_observations
    )

    bootstrap_indices = (
        bootstrap_indices
        .reshape(
            bootstrap_iterations,
            -1,
        )[
            :,
            :number_of_observations,
        ]
    )

    return bootstrap_indices


number_of_audit_observations = len(
    audit_daily_returns
)

bootstrap_indices = (
    create_circular_block_indices(
        number_of_observations=(
            number_of_audit_observations
        ),
        block_length=(
            BOOTSTRAP_BLOCK_LENGTH
        ),
        bootstrap_iterations=(
            BOOTSTRAP_ITERATIONS
        ),
        random_seed=(
            RANDOM_SEED
        ),
    )
)


# ---------------------------------------------------------
# 3. Helper functions
# ---------------------------------------------------------

def calculate_cagr_from_returns(
    daily_returns,
    start_date,
    end_date,
):

    ending_wealth = float(
        np.prod(
            1
            + np.asarray(
                daily_returns,
                dtype=float,
            )
        )
    )

    elapsed_years = (
        (
            pd.Timestamp(
                end_date
            )
            - pd.Timestamp(
                start_date
            )
        ).days
        / 365.25
    )

    if elapsed_years <= 0:
        return np.nan

    return (
        ending_wealth
        ** (
            1
            / elapsed_years
        )
        - 1
    )


def benjamini_hochberg_adjustment(
    p_values,
):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(
        p_values
    )

    sorted_positions = np.argsort(
        p_values
    )

    sorted_p_values = (
        p_values[
            sorted_positions
        ]
    )

    adjusted_sorted = (
        sorted_p_values
        * number_of_tests
        / np.arange(
            1,
            number_of_tests + 1,
        )
    )

    adjusted_sorted = (
        np.minimum.accumulate(
            adjusted_sorted[
                ::-1
            ]
        )[
            ::-1
        ]
    )

    adjusted_p_values = np.empty_like(
        adjusted_sorted
    )

    adjusted_p_values[
        sorted_positions
    ] = np.clip(
        adjusted_sorted,
        0,
        1,
    )

    return adjusted_p_values


# ---------------------------------------------------------
# 4. Run paired block-bootstrap comparisons
# ---------------------------------------------------------

confidence_tail = (
    1
    - BOOTSTRAP_CONFIDENCE_LEVEL
)

lower_percentile = (
    100
    * confidence_tail
    / 2
)

upper_percentile = (
    100
    * (
        1
        - confidence_tail
        / 2
    )
)

bootstrap_audit_records = []

for comparison in comparison_definitions:

    strategy_name = (
        comparison[
            "Strategy"
        ]
    )

    benchmark_name = (
        comparison[
            "Benchmark"
        ]
    )

    strategy_returns = (
        audit_daily_returns[
            strategy_name
        ]
        .to_numpy(
            dtype=float
        )
    )

    benchmark_returns = (
        audit_daily_returns[
            benchmark_name
        ]
        .to_numpy(
            dtype=float
        )
    )

    excess_returns = (
        strategy_returns
        - benchmark_returns
    )

    observed_mean_daily_excess = float(
        excess_returns.mean()
    )

    observed_annualised_alpha = (
        observed_mean_daily_excess
        * TRADING_DAYS_PER_YEAR
    )

    annualised_tracking_error = (
        excess_returns.std(
            ddof=1
        )
        * np.sqrt(
            TRADING_DAYS_PER_YEAR
        )
    )

    information_ratio = (
        observed_annualised_alpha
        / annualised_tracking_error
        if annualised_tracking_error > 0
        else np.nan
    )

    strategy_cagr = (
        calculate_cagr_from_returns(
            daily_returns=(
                strategy_returns
            ),
            start_date=(
                audit_daily_returns.index.min()
            ),
            end_date=(
                audit_daily_returns.index.max()
            ),
        )
    )

    benchmark_cagr = (
        calculate_cagr_from_returns(
            daily_returns=(
                benchmark_returns
            ),
            start_date=(
                audit_daily_returns.index.min()
            ),
            end_date=(
                audit_daily_returns.index.max()
            ),
        )
    )

    observed_cagr_advantage = (
        strategy_cagr
        - benchmark_cagr
    )

    # Raw block bootstrap estimates the sampling distribution
    # and confidence interval around annualised alpha.
    raw_bootstrap_alpha = (
        excess_returns[
            bootstrap_indices
        ]
        .mean(
            axis=1
        )
        * TRADING_DAYS_PER_YEAR
    )

    bootstrap_alpha_lower = float(
        np.percentile(
            raw_bootstrap_alpha,
            lower_percentile,
        )
    )

    bootstrap_alpha_upper = float(
        np.percentile(
            raw_bootstrap_alpha,
            upper_percentile,
        )
    )

    bootstrap_probability_positive = float(
        (
            raw_bootstrap_alpha
            > 0
        ).mean()
    )

    # Centre the observations to create the null hypothesis
    # distribution where expected excess return equals zero.
    centred_excess_returns = (
        excess_returns
        - observed_mean_daily_excess
    )

    null_bootstrap_alpha = (
        centred_excess_returns[
            bootstrap_indices
        ]
        .mean(
            axis=1
        )
        * TRADING_DAYS_PER_YEAR
    )

    one_sided_p_value = (
        1
        + np.sum(
            null_bootstrap_alpha
            >= observed_annualised_alpha
        )
    ) / (
        BOOTSTRAP_ITERATIONS
        + 1
    )

    bootstrap_audit_records.append(
        {
            "Strategy":
                strategy_name,

            "Benchmark":
                benchmark_name,

            "Observations":
                len(
                    excess_returns
                ),

            "Observed Strategy CAGR":
                strategy_cagr,

            "Observed Benchmark CAGR":
                benchmark_cagr,

            "Observed CAGR Advantage":
                observed_cagr_advantage,

            "Annualised Arithmetic Alpha":
                observed_annualised_alpha,

            "Annualised Tracking Error":
                annualised_tracking_error,

            "Information Ratio":
                information_ratio,

            "Bootstrap Alpha Median":
                float(
                    np.median(
                        raw_bootstrap_alpha
                    )
                ),

            "Bootstrap Alpha 95% Lower":
                bootstrap_alpha_lower,

            "Bootstrap Alpha 95% Upper":
                bootstrap_alpha_upper,

            "Bootstrap Probability Alpha > 0":
                bootstrap_probability_positive,

            "One-Sided Bootstrap P-Value":
                one_sided_p_value,

            "Block Length":
                BOOTSTRAP_BLOCK_LENGTH,

            "Bootstrap Iterations":
                BOOTSTRAP_ITERATIONS,
        }
    )


block_bootstrap_alpha_audit = (
    pd.DataFrame(
        bootstrap_audit_records
    )
)


# ---------------------------------------------------------
# 5. Apply multiple-comparison adjustment
# ---------------------------------------------------------

block_bootstrap_alpha_audit[
    "FDR-Adjusted P-Value"
] = (
    benjamini_hochberg_adjustment(
        block_bootstrap_alpha_audit[
            "One-Sided Bootstrap P-Value"
        ]
    )
)

block_bootstrap_alpha_audit[
    "Raw Significant at 5%"
] = (
    block_bootstrap_alpha_audit[
        "One-Sided Bootstrap P-Value"
    ]
    < 0.05
)

block_bootstrap_alpha_audit[
    "FDR Significant at 5%"
] = (
    block_bootstrap_alpha_audit[
        "FDR-Adjusted P-Value"
    ]
    < 0.05
)

block_bootstrap_alpha_audit[
    "95% Alpha Interval Entirely Positive"
] = (
    block_bootstrap_alpha_audit[
        "Bootstrap Alpha 95% Lower"
    ]
    > 0
)

block_bootstrap_alpha_audit[
    "Statistical Conclusion"
] = np.select(
    [
        (
            block_bootstrap_alpha_audit[
                "FDR Significant at 5%"
            ]
            & block_bootstrap_alpha_audit[
                "95% Alpha Interval Entirely Positive"
            ]
        ),

        (
            block_bootstrap_alpha_audit[
                "Raw Significant at 5%"
            ]
            & ~block_bootstrap_alpha_audit[
                "FDR Significant at 5%"
            ]
        ),
    ],
    [
        "Positive after FDR adjustment",
        "Positive before adjustment only",
    ],
    default="Insufficient evidence of positive alpha",
)

block_bootstrap_alpha_audit = (
    block_bootstrap_alpha_audit
    .sort_values(
        [
            "Benchmark",
            "FDR-Adjusted P-Value",
            "Annualised Arithmetic Alpha",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 6. Create concise significance summaries
# ---------------------------------------------------------

significance_count_summary = (
    block_bootstrap_alpha_audit
    .groupby(
        "Benchmark"
    )
    .agg(
        Comparisons=(
            "Strategy",
            "count",
        ),

        Positive_Observed_Alpha=(
            "Annualised Arithmetic Alpha",
            lambda values: (
                values
                > 0
            ).sum(),
        ),

        Positive_95pct_Intervals=(
            "95% Alpha Interval Entirely Positive",
            "sum",
        ),

        Raw_5pct_Significant=(
            "Raw Significant at 5%",
            "sum",
        ),

        FDR_5pct_Significant=(
            "FDR Significant at 5%",
            "sum",
        ),
    )
)


regime_aware_significance = (
    block_bootstrap_alpha_audit.loc[
        block_bootstrap_alpha_audit[
            "Strategy"
        ]
        == "Regime-Aware Walk-Forward"
    ]
    .set_index(
        "Benchmark"
    )
    .sort_index()
)


# ---------------------------------------------------------
# 7. Validate the statistical audit
# ---------------------------------------------------------

assert (
    block_bootstrap_alpha_audit[
        "Observations"
    ]
    .eq(
        number_of_audit_observations
    )
    .all()
)

assert (
    block_bootstrap_alpha_audit[
        "One-Sided Bootstrap P-Value"
    ]
    .between(
        0,
        1,
    )
    .all()
)

assert (
    block_bootstrap_alpha_audit[
        "FDR-Adjusted P-Value"
    ]
    .between(
        0,
        1,
    )
    .all()
)

assert (
    block_bootstrap_alpha_audit[
        "Bootstrap Probability Alpha > 0"
    ]
    .between(
        0,
        1,
    )
    .all()
)

assert (
    block_bootstrap_alpha_audit[
        "Bootstrap Alpha 95% Lower"
    ]
    .le(
        block_bootstrap_alpha_audit[
            "Bootstrap Alpha 95% Upper"
        ]
    )
    .all()
)

assert np.isfinite(
    block_bootstrap_alpha_audit[
        [
            "Observed Strategy CAGR",
            "Observed Benchmark CAGR",
            "Observed CAGR Advantage",
            "Annualised Arithmetic Alpha",
            "Annualised Tracking Error",
            "Information Ratio",
            "Bootstrap Alpha Median",
            "Bootstrap Alpha 95% Lower",
            "Bootstrap Alpha 95% Upper",
        ]
    ]
    .to_numpy()
).all()

assert len(
    block_bootstrap_alpha_audit
) == len(
    comparison_definitions
)


# ---------------------------------------------------------
# 8. Save statistical-audit outputs
# ---------------------------------------------------------

block_bootstrap_alpha_audit.to_csv(
    PROCESSED_DATA_DIR
    / "block_bootstrap_alpha_audit.csv",
    index=False,
)

significance_count_summary.to_csv(
    PROCESSED_DATA_DIR
    / "block_bootstrap_significance_counts.csv",
)

regime_aware_significance.to_csv(
    PROCESSED_DATA_DIR
    / "regime_aware_bootstrap_significance.csv",
)


# ---------------------------------------------------------
# 9. Display results
# ---------------------------------------------------------

display_bootstrap_audit = (
    block_bootstrap_alpha_audit.copy()
)

percentage_columns = [
    "Observed Strategy CAGR",
    "Observed Benchmark CAGR",
    "Observed CAGR Advantage",
    "Annualised Arithmetic Alpha",
    "Annualised Tracking Error",
    "Bootstrap Alpha Median",
    "Bootstrap Alpha 95% Lower",
    "Bootstrap Alpha 95% Upper",
    "Bootstrap Probability Alpha > 0",
    "One-Sided Bootstrap P-Value",
    "FDR-Adjusted P-Value",
]

for column in percentage_columns:

    display_bootstrap_audit[
        column
    ] = (
        display_bootstrap_audit[
            column
        ]
        * 100
    )


display_regime_aware_significance = (
    regime_aware_significance.copy()
)

for column in percentage_columns:

    display_regime_aware_significance[
        column
    ] = (
        display_regime_aware_significance[
            column
        ]
        * 100
    )


print("BLOCK-BOOTSTRAP STATISTICAL SIGNIFICANCE AUDIT")
print("=" * 72)
print(
    "Analysis period:",
    audit_daily_returns
    .index.min()
    .date(),
    "to",
    audit_daily_returns
    .index.max()
    .date(),
)
print(
    "Daily observations:",
    number_of_audit_observations,
)
print(
    "Strategy-benchmark comparisons:",
    len(
        comparison_definitions
    ),
)
print(
    "Bootstrap simulations:",
    BOOTSTRAP_ITERATIONS,
)
print(
    "Circular block length:",
    BOOTSTRAP_BLOCK_LENGTH,
    "trading days",
)
print(
    "Confidence level:",
    f"{BOOTSTRAP_CONFIDENCE_LEVEL:.0%}",
)
print(
    "Multiple-testing adjustment:",
    "Benjamini–Hochberg FDR",
)
print(
    "Paired return alignment:",
    "PASSED",
)
print(
    "Bootstrap validation:",
    "PASSED",
)
print(
    "Statistical significance audit:",
    "PASSED",
)

print("\nALL STRATEGY-BENCHMARK COMPARISONS")
display(
    display_bootstrap_audit.round(
        2
    )
)

print("\nREGIME-AWARE SIGNIFICANCE")
display(
    display_regime_aware_significance.round(
        2
    )
)

print("\nSIGNIFICANCE COUNTS")
display(
    significance_count_summary
)

BLOCK-BOOTSTRAP STATISTICAL SIGNIFICANCE AUDIT
Analysis period: 2018-04-02 to 2026-07-30
Daily observations: 2054
Strategy-benchmark comparisons: 9
Bootstrap simulations: 3000
Circular block length: 21 trading days
Confidence level: 95%
Multiple-testing adjustment: Benjamini–Hochberg FDR
Paired return alignment: PASSED
Bootstrap validation: PASSED
Statistical significance audit: PASSED

ALL STRATEGY-BENCHMARK COMPARISONS


,Strategy,Benchmark,Observations,Observed Strategy CAGR,Observed Benchmark CAGR,Observed CAGR Advantage,Annualised Arithmetic Alpha,Annualised Tracking Error,Information Ratio,Bootstrap Alpha Median,...,Bootstrap Alpha 95% Upper,Bootstrap Probability Alpha > 0,One-Sided Bootstrap P-Value,Block Length,Bootstrap Iterations,FDR-Adjusted P-Value,Raw Significant at 5%,FDR Significant at 5%,95% Alpha Interval Entirely Positive,Statistical Conclusion
0,Regime-Aware Walk-Forward,Equal-Weight India 10,2054,34.44,20.78,13.66,11.80,12.30,0.96,11.72,...,21.06,99.67,0.67,21,3000,1.20,True,True,True,Positive after FDR adjustment
1,1-Month Reversal,Equal-Weight India 10,2054,29.24,20.78,8.46,7.85,13.16,0.60,7.75,...,16.55,96.63,3.87,21,3000,4.97,True,True,False,Insufficient evidence of positive alpha
2,12-Month Momentum,Equal-Weight India 10,2054,24.64,20.78,3.86,4.31,13.41,0.32,4.15,...,13.30,81.83,18.13,21,3000,18.13,False,False,False,Insufficient evidence of positive alpha
3,Gradient Boosting,Equal-Weight India 10,2054,23.68,20.78,2.90,3.27,12.54,0.26,3.18,...,10.17,82.47,17.86,21,3000,18.13,False,False,False,Insufficient evidence of positive alpha
4,Regime-Aware Walk-Forward,Nifty 50,2054,34.44,10.98,23.46,20.37,15.02,1.36,20.28,...,32.04,99.97,0.10,21,3000,0.40,True,True,True,Positive after FDR adjustment
5,1-Month Reversal,Nifty 50,2054,29.24,10.98,18.25,16.42,15.67,1.05,16.30,...,26.98,99.90,0.13,21,3000,0.40,True,True,True,Positive after FDR adjustment
6,Equal-Weight India 10,Nifty 50,2054,20.78,10.98,9.80,8.57,8.02,1.07,8.54,...,13.60,99.93,0.10,21,3000,0.40,True,True,True,Positive after FDR adjustment
7,Gradient Boosting,Nifty 50,2054,23.68,10.98,12.70,11.84,14.63,0.81,11.83,...,20.68,99.63,0.57,21,3000,1.20,True,True,True,Positive after FDR adjustment
8,12-Month Momentum,Nifty 50,2054,24.64,10.98,13.65,12.89,16.49,0.78,12.73,...,24.20,98.77,1.07,21,3000,1.60,True,True,True,Positive after FDR adjustment



REGIME-AWARE SIGNIFICANCE


,Strategy,Observations,Observed Strategy CAGR,Observed Benchmark CAGR,Observed CAGR Advantage,Annualised Arithmetic Alpha,Annualised Tracking Error,Information Ratio,Bootstrap Alpha Median,Bootstrap Alpha 95% Lower,Bootstrap Alpha 95% Upper,Bootstrap Probability Alpha > 0,One-Sided Bootstrap P-Value,Block Length,Bootstrap Iterations,FDR-Adjusted P-Value,Raw Significant at 5%,FDR Significant at 5%,95% Alpha Interval Entirely Positive,Statistical Conclusion
Benchmark,,,,,,,,,,,,,,,,,,,,
Equal-Weight India 10,Regime-Aware Walk-Forward,2054,34.44,20.78,13.66,11.80,12.30,0.96,11.72,2.98,21.06,99.67,0.67,21,3000,1.2,True,True,True,Positive after FDR adjustment
Nifty 50,Regime-Aware Walk-Forward,2054,34.44,10.98,23.46,20.37,15.02,1.36,20.28,9.37,32.04,99.97,0.10,21,3000,0.4,True,True,True,Positive after FDR adjustment



SIGNIFICANCE COUNTS


,Comparisons,Positive_Observed_Alpha,Positive_95pct_Intervals,Raw_5pct_Significant,FDR_5pct_Significant
Benchmark,,,,,
Equal-Weight India 10,4,4,1,2,2
Nifty 50,5,5,5,5,5


## Walk-Forward Lasso Regression

Lasso Regression extends the linear model by shrinking weak feature coefficients and setting some coefficients exactly to zero.

This provides:

- Automatic feature selection
- Reduced model complexity
- A check on whether sparse linear relationships outperform ordinary linear and Ridge Regression
- The same expanding-window, leakage-safe evaluation methodology used by every previous model

A fixed, untuned regularisation parameter is used to avoid introducing another retrospective hyperparameter search.

In [29]:
# =========================================================
# WALK-FORWARD LASSO REGRESSION AND STRATEGY BACKTEST
# =========================================================

from sklearn.linear_model import Lasso
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


required_objects = [
    "regression_panel",
    "rebalance_schedule",
    "feature_columns",
    "market_calendar",
    "regression_evaluation",
    "baseline_evaluation",
    "backtest_monthly_target_weights",
    "calculate_backtest_statistics",
    "asset_daily_returns",
    "nifty_daily_returns",
    "final_strategy_daily_net_returns",
    "final_strategy_daily_gross_returns",
    "final_strategy_daily_values",
    "final_strategy_daily_turnover",
    "final_strategy_daily_costs",
    "final_all_strategy_backtest_summary",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required previous-step objects are missing:\n"
        + "\n".join(missing_objects)
    )


LASSO_MODEL_NAME = "Lasso Regression"
LASSO_ALPHA = 0.001
TOP_STOCK_COUNT = 3

market_calendar = pd.DatetimeIndex(
    market_calendar
).sort_values()


# ---------------------------------------------------------
# 1. Define the standardised Lasso model
# ---------------------------------------------------------

lasso_pipeline = Pipeline(
    steps=[
        (
            "standard_scaler",
            StandardScaler(),
        ),
        (
            "regression_model",
            Lasso(
                alpha=LASSO_ALPHA,
                max_iter=20_000,
                tol=1e-5,
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)


# ---------------------------------------------------------
# 2. Generate monthly leakage-safe predictions
# ---------------------------------------------------------

lasso_prediction_records = []

for rebalance_date, training_cutoff in (
    rebalance_schedule.items()
):

    training_sample = (
        regression_panel.loc[
            regression_panel["Date"]
            <= training_cutoff
        ]
        .copy()
    )

    prediction_sample = (
        regression_panel.loc[
            regression_panel["Date"]
            == rebalance_date
        ]
        .copy()
    )

    if len(prediction_sample) != len(
        INDIA_10_TICKERS
    ):
        continue

    if len(training_sample) < (
        len(INDIA_10_TICKERS) * 252
    ):
        continue

    X_train = (
        training_sample[
            feature_columns
        ]
        .astype(float)
    )

    y_train = (
        training_sample[
            "forward_excess_return_21d"
        ]
        .astype(float)
    )

    X_predict = (
        prediction_sample[
            feature_columns
        ]
        .astype(float)
    )

    lasso_pipeline.fit(
        X_train,
        y_train,
    )

    predicted_excess_return = (
        lasso_pipeline.predict(
            X_predict
        )
    )

    fitted_lasso = (
        lasso_pipeline.named_steps[
            "regression_model"
        ]
    )

    non_zero_coefficients = int(
        np.count_nonzero(
            np.abs(
                fitted_lasso.coef_
            )
            > 1e-12
        )
    )

    prediction_records = (
        prediction_sample[
            [
                "Date",
                "Ticker",
                "forward_return_21d",
                "forward_excess_return_21d",
                "positive_return_target",
                "outperform_target",
            ]
        ]
        .copy()
    )

    prediction_records[
        "Model"
    ] = LASSO_MODEL_NAME

    prediction_records[
        "Predicted Excess Return"
    ] = predicted_excess_return

    prediction_records[
        "Training Cutoff"
    ] = training_cutoff

    prediction_records[
        "Training Observations"
    ] = len(
        training_sample
    )

    prediction_records[
        "Non-Zero Coefficients"
    ] = non_zero_coefficients

    lasso_prediction_records.append(
        prediction_records
    )


if not lasso_prediction_records:
    raise RuntimeError(
        "No Lasso predictions were generated."
    )


lasso_predictions = (
    pd.concat(
        lasso_prediction_records,
        ignore_index=True,
    )
    .sort_values(
        [
            "Date",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 3. Evaluate prediction and ranking performance
# ---------------------------------------------------------

actual_values = (
    lasso_predictions[
        "forward_excess_return_21d"
    ]
)

predicted_values = (
    lasso_predictions[
        "Predicted Excess Return"
    ]
)

monthly_rank_ic_values = []
selected_forward_returns = []
selected_excess_returns = []
selected_positive_rates = []
selected_outperformance_rates = []

for prediction_date, cross_section in (
    lasso_predictions.groupby("Date")
):

    if (
        cross_section[
            "Predicted Excess Return"
        ].nunique()
        > 1
    ):

        rank_ic = (
            cross_section[
                [
                    "Predicted Excess Return",
                    "forward_excess_return_21d",
                ]
            ]
            .corr(
                method="spearman"
            )
            .iloc[0, 1]
        )

    else:
        rank_ic = np.nan

    monthly_rank_ic_values.append(
        rank_ic
    )

    selected_stocks = (
        cross_section
        .sort_values(
            by=[
                "Predicted Excess Return",
                "Ticker",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .head(
            TOP_STOCK_COUNT
        )
    )

    selected_forward_returns.append(
        selected_stocks[
            "forward_return_21d"
        ].mean()
    )

    selected_excess_returns.append(
        selected_stocks[
            "forward_excess_return_21d"
        ].mean()
    )

    selected_positive_rates.append(
        selected_stocks[
            "positive_return_target"
        ].mean()
    )

    selected_outperformance_rates.append(
        selected_stocks[
            "outperform_target"
        ].mean()
    )


valid_rank_ic = (
    pd.Series(
        monthly_rank_ic_values,
        dtype=float,
    )
    .dropna()
)

directional_accuracy = (
    (
        predicted_values > 0
    )
    == (
        actual_values > 0
    )
).mean()


lasso_evaluation = (
    pd.DataFrame(
        [
            {
                "Model":
                    LASSO_MODEL_NAME,

                "Rebalances":
                    lasso_predictions[
                        "Date"
                    ].nunique(),

                "Predictions":
                    len(
                        lasso_predictions
                    ),

                "Mean Absolute Error":
                    mean_absolute_error(
                        actual_values,
                        predicted_values,
                    ),

                "Root Mean Squared Error":
                    np.sqrt(
                        mean_squared_error(
                            actual_values,
                            predicted_values,
                        )
                    ),

                "Pooled R-Squared":
                    r2_score(
                        actual_values,
                        predicted_values,
                    ),

                "Mean Rank IC":
                    valid_rank_ic.mean(),

                "Median Rank IC":
                    valid_rank_ic.median(),

                "Positive Rank IC Rate":
                    valid_rank_ic.gt(
                        0
                    ).mean(),

                "Directional Accuracy":
                    directional_accuracy,

                "Mean Selected Forward Return":
                    np.mean(
                        selected_forward_returns
                    ),

                "Mean Selected Excess Return":
                    np.mean(
                        selected_excess_returns
                    ),

                "Selected Positive Return Rate":
                    np.mean(
                        selected_positive_rates
                    ),

                "Selected Outperformance Rate":
                    np.mean(
                        selected_outperformance_rates
                    ),

                "Median Non-Zero Coefficients":
                    lasso_predictions[
                        "Non-Zero Coefficients"
                    ].median(),

                "Minimum Non-Zero Coefficients":
                    lasso_predictions[
                        "Non-Zero Coefficients"
                    ].min(),

                "Maximum Non-Zero Coefficients":
                    lasso_predictions[
                        "Non-Zero Coefficients"
                    ].max(),
            }
        ]
    )
    .set_index(
        "Model"
    )
)


# ---------------------------------------------------------
# 4. Convert Lasso predictions into top-three weights
# ---------------------------------------------------------

lasso_weight_records = []

for signal_date, cross_section in (
    lasso_predictions.groupby("Date")
):

    calendar_position = (
        market_calendar.searchsorted(
            signal_date,
            side="right",
        )
    )

    if calendar_position >= len(
        market_calendar
    ):
        continue

    execution_date = (
        market_calendar[
            calendar_position
        ]
    )

    ranked_cross_section = (
        cross_section
        .sort_values(
            by=[
                "Predicted Excess Return",
                "Ticker",
            ],
            ascending=[
                False,
                True,
            ],
        )
    )

    selected_tickers = (
        ranked_cross_section[
            "Ticker"
        ]
        .head(
            TOP_STOCK_COUNT
        )
        .tolist()
    )

    score_map = (
        ranked_cross_section
        .set_index(
            "Ticker"
        )[
            "Predicted Excess Return"
        ]
        .to_dict()
    )

    for ticker in INDIA_10_TICKERS:

        lasso_weight_records.append(
            {
                "Strategy":
                    LASSO_MODEL_NAME,

                "Signal Type":
                    "Sparse Regression Model",

                "Signal Date":
                    signal_date,

                "Execution Date":
                    execution_date,

                "Ticker":
                    ticker,

                "Score":
                    score_map[
                        ticker
                    ],

                "Selected":
                    ticker
                    in selected_tickers,

                "Target Weight":
                    (
                        1 / TOP_STOCK_COUNT
                        if ticker
                        in selected_tickers
                        else 0.0
                    ),
            }
        )


lasso_strategy_weights = (
    pd.DataFrame(
        lasso_weight_records
    )
    .sort_values(
        [
            "Execution Date",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 5. Run the transaction-cost-adjusted Lasso backtest
# ---------------------------------------------------------

lasso_backtest_result = (
    backtest_monthly_target_weights(
        strategy_weights=(
            lasso_strategy_weights
        ),

        daily_asset_returns=(
            asset_daily_returns
        ),

        initial_investment_inr=(
            INITIAL_INVESTMENT_INR
        ),

        one_way_transaction_cost=(
            ONE_WAY_TRANSACTION_COST
        ),
    )
)


lasso_nifty_returns = (
    nifty_daily_returns
    .reindex(
        lasso_backtest_result.index
    )
    .copy()
)

if lasso_nifty_returns.isna().any():
    raise RuntimeError(
        "Missing Nifty observations during "
        "the Lasso backtest."
    )

lasso_nifty_returns.iloc[0] = 0.0


lasso_backtest_statistics = (
    calculate_backtest_statistics(
        strategy_name=(
            LASSO_MODEL_NAME
        ),

        daily_returns=(
            lasso_backtest_result[
                "Net Return"
            ]
        ),

        portfolio_values=(
            lasso_backtest_result[
                "Portfolio Value (₹)"
            ]
        ),

        benchmark_returns=(
            lasso_nifty_returns
        ),

        daily_turnover=(
            lasso_backtest_result[
                "One-Way Turnover"
            ]
        ),

        daily_costs=(
            lasso_backtest_result[
                "Transaction Cost (₹)"
            ]
        ),
    )
)


lasso_backtest_summary = (
    pd.DataFrame(
        [
            lasso_backtest_statistics
        ]
    )
    .set_index(
        "Strategy"
    )
)


# ---------------------------------------------------------
# 6. Compare Lasso with linear models and baselines
# ---------------------------------------------------------

lasso_model_comparison = (
    pd.concat(
        [
            regression_evaluation.loc[
                [
                    "Linear Regression",
                    "Ridge Regression",
                ]
            ],

            lasso_evaluation[
                regression_evaluation.columns
            ],
        ],
        axis=0,
    )
)

for baseline_name in [
    "1-Month Reversal",
    "12-Month Momentum",
]:

    baseline_row = (
        baseline_evaluation.loc[
            baseline_name
        ]
    )

    lasso_model_comparison.loc[
        baseline_name,
        [
            "Rebalances",
            "Mean Rank IC",
            "Positive Rank IC Rate",
            "Mean Selected Forward Return",
            "Mean Selected Excess Return",
            "Selected Positive Return Rate",
            "Selected Outperformance Rate",
        ],
    ] = [
        baseline_row[
            "Rebalances"
        ],
        baseline_row[
            "Mean Rank IC"
        ],
        baseline_row[
            "Positive Rank IC Rate"
        ],
        baseline_row[
            "Mean Selected Forward Return"
        ],
        baseline_row[
            "Mean Selected Excess Return"
        ],
        baseline_row[
            "Selected Positive Return Rate"
        ],
        baseline_row[
            "Selected Outperformance Rate"
        ],
    ]


lasso_model_comparison = (
    lasso_model_comparison
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


lasso_backtest_comparison = (
    pd.concat(
        [
            final_all_strategy_backtest_summary.loc[
                [
                    "Regime-Aware Walk-Forward",
                    "1-Month Reversal",
                    "12-Month Momentum",
                    "Linear Regression",
                    "Ridge Regression",
                    "Equal-Weight India 10",
                    "Nifty 50",
                ]
            ],

            lasso_backtest_summary,
        ],
        axis=0,
    )
    .sort_values(
        "Sharpe Ratio",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 7. Add Lasso to the final research datasets
# ---------------------------------------------------------

final_strategy_daily_net_returns = (
    final_strategy_daily_net_returns
    .drop(
        columns=[
            LASSO_MODEL_NAME
        ],
        errors="ignore",
    )
)

final_strategy_daily_gross_returns = (
    final_strategy_daily_gross_returns
    .drop(
        columns=[
            LASSO_MODEL_NAME
        ],
        errors="ignore",
    )
)

final_strategy_daily_values = (
    final_strategy_daily_values
    .drop(
        columns=[
            LASSO_MODEL_NAME
        ],
        errors="ignore",
    )
)

final_strategy_daily_turnover = (
    final_strategy_daily_turnover
    .drop(
        columns=[
            LASSO_MODEL_NAME
        ],
        errors="ignore",
    )
)

final_strategy_daily_costs = (
    final_strategy_daily_costs
    .drop(
        columns=[
            LASSO_MODEL_NAME
        ],
        errors="ignore",
    )
)


final_strategy_daily_net_returns[
    LASSO_MODEL_NAME
] = (
    lasso_backtest_result[
        "Net Return"
    ]
)

final_strategy_daily_gross_returns[
    LASSO_MODEL_NAME
] = (
    lasso_backtest_result[
        "Gross Return"
    ]
)

final_strategy_daily_values[
    LASSO_MODEL_NAME
] = (
    lasso_backtest_result[
        "Portfolio Value (₹)"
    ]
)

final_strategy_daily_turnover[
    LASSO_MODEL_NAME
] = (
    lasso_backtest_result[
        "One-Way Turnover"
    ]
)

final_strategy_daily_costs[
    LASSO_MODEL_NAME
] = (
    lasso_backtest_result[
        "Transaction Cost (₹)"
    ]
)


final_all_strategy_backtest_summary = (
    pd.concat(
        [
            final_all_strategy_backtest_summary.drop(
                index=LASSO_MODEL_NAME,
                errors="ignore",
            ),
            lasso_backtest_summary,
        ],
        axis=0,
    )
    .sort_values(
        "Sharpe Ratio",
        ascending=False,
    )
)


monthly_strategy_weights_with_lasso = (
    pd.concat(
        [
            monthly_strategy_weights.loc[
                monthly_strategy_weights[
                    "Strategy"
                ]
                != LASSO_MODEL_NAME
            ],
            lasso_strategy_weights,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "Execution Date",
            "Strategy",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 8. Save outputs
# ---------------------------------------------------------

lasso_predictions.to_csv(
    PROCESSED_DATA_DIR
    / "walk_forward_lasso_predictions.csv",
    index=False,
)

lasso_evaluation.to_csv(
    PROCESSED_DATA_DIR
    / "walk_forward_lasso_evaluation.csv",
)

lasso_strategy_weights.to_csv(
    PROCESSED_DATA_DIR
    / "lasso_strategy_weights.csv",
    index=False,
)

lasso_backtest_result.to_csv(
    PROCESSED_DATA_DIR
    / "lasso_daily_backtest.csv",
    index_label="Date",
)

lasso_backtest_summary.to_csv(
    PROCESSED_DATA_DIR
    / "lasso_backtest_summary.csv",
)

lasso_model_comparison.to_csv(
    PROCESSED_DATA_DIR
    / "lasso_model_comparison.csv",
)

lasso_backtest_comparison.to_csv(
    PROCESSED_DATA_DIR
    / "lasso_backtest_comparison.csv",
)

monthly_strategy_weights_with_lasso.to_csv(
    PROCESSED_DATA_DIR
    / "monthly_strategy_weights_with_lasso.csv",
    index=False,
)

final_strategy_daily_net_returns.to_csv(
    PROCESSED_DATA_DIR
    / "final_strategy_daily_net_returns.csv",
    index_label="Date",
)

final_strategy_daily_values.to_csv(
    PROCESSED_DATA_DIR
    / "final_strategy_daily_portfolio_values.csv",
    index_label="Date",
)

final_all_strategy_backtest_summary.to_csv(
    PROCESSED_DATA_DIR
    / "final_all_strategy_backtest_summary.csv",
)


# ---------------------------------------------------------
# 9. Validate Lasso analysis
# ---------------------------------------------------------

lasso_weight_totals = (
    lasso_strategy_weights
    .groupby(
        [
            "Signal Date",
            "Execution Date",
        ]
    )[
        "Target Weight"
    ]
    .sum()
)

lasso_active_holdings = (
    lasso_strategy_weights
    .groupby(
        [
            "Signal Date",
            "Execution Date",
        ]
    )[
        "Selected"
    ]
    .sum()
)

assert (
    lasso_predictions[
        "Training Cutoff"
    ]
    < lasso_predictions[
        "Date"
    ]
).all()

assert (
    lasso_predictions[
        "Predicted Excess Return"
    ]
    .notna()
    .all()
)

assert np.isfinite(
    lasso_predictions[
        "Predicted Excess Return"
    ]
).all()

assert (
    lasso_predictions[
        "Date"
    ].nunique()
    == 101
)

assert np.allclose(
    lasso_weight_totals,
    1.0,
    atol=1e-10,
)

assert (
    lasso_active_holdings
    .eq(
        TOP_STOCK_COUNT
    )
    .all()
)

assert (
    lasso_strategy_weights[
        "Execution Date"
    ]
    > lasso_strategy_weights[
        "Signal Date"
    ]
).all()

assert (
    lasso_backtest_result[
        "Portfolio Value (₹)"
    ]
    .gt(
        0
    )
    .all()
)

assert (
    LASSO_MODEL_NAME
    in final_strategy_daily_net_returns.columns
)

assert (
    LASSO_MODEL_NAME
    in final_all_strategy_backtest_summary.index
)


# ---------------------------------------------------------
# 10. Display results
# ---------------------------------------------------------

display_lasso_evaluation = (
    lasso_evaluation.copy()
)

evaluation_percentage_columns = [
    "Mean Absolute Error",
    "Root Mean Squared Error",
    "Pooled R-Squared",
    "Mean Rank IC",
    "Median Rank IC",
    "Positive Rank IC Rate",
    "Directional Accuracy",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in evaluation_percentage_columns:

    display_lasso_evaluation[
        column
    ] = (
        display_lasso_evaluation[
            column
        ]
        * 100
    )


display_lasso_backtest = (
    lasso_backtest_summary.copy()
)

backtest_percentage_columns = [
    "Total Return",
    "CAGR",
    "Annualised Volatility",
    "Maximum Drawdown",
    "Correlation vs Nifty",
    "Total One-Way Turnover",
    "Annualised Turnover",
]

for column in backtest_percentage_columns:

    display_lasso_backtest[
        column
    ] = (
        display_lasso_backtest[
            column
        ]
        * 100
    )


display_lasso_comparison = (
    lasso_backtest_comparison[
        [
            "Ending Value (₹)",
            "CAGR",
            "Annualised Volatility",
            "Sharpe Ratio",
            "Maximum Drawdown",
            "Annualised Turnover",
            "Transaction Costs (₹)",
        ]
    ]
    .copy()
)

for column in [
    "CAGR",
    "Annualised Volatility",
    "Maximum Drawdown",
    "Annualised Turnover",
]:

    display_lasso_comparison[
        column
    ] = (
        display_lasso_comparison[
            column
        ]
        * 100
    )


print("WALK-FORWARD LASSO REGRESSION")
print("=" * 72)
print(
    "Evaluation period:",
    lasso_predictions[
        "Date"
    ].min().date(),
    "to",
    lasso_predictions[
        "Date"
    ].max().date(),
)
print(
    "Monthly rebalances:",
    lasso_predictions[
        "Date"
    ].nunique(),
)
print(
    "Features supplied:",
    len(
        feature_columns
    ),
)
print(
    "Lasso alpha:",
    LASSO_ALPHA,
)
print(
    "Median non-zero coefficients:",
    int(
        lasso_predictions[
            "Non-Zero Coefficients"
        ].median()
    ),
)
print(
    "Training-only standardisation:",
    "PASSED",
)
print(
    "Leakage-safe cutoffs:",
    "PASSED",
)
print(
    "One-trading-day execution lag:",
    "PASSED",
)
print(
    "Transaction-cost-adjusted backtest:",
    "PASSED",
)
print(
    "Lasso scope completion:",
    "PASSED",
)

print("\nLASSO PREDICTION METRICS")
display(
    display_lasso_evaluation.round(
        2
    )
)

print("\nLASSO STRATEGY PERFORMANCE")
display(
    display_lasso_backtest.round(
        2
    )
)

print("\nLASSO VERSUS MAJOR STRATEGIES")
display(
    display_lasso_comparison.round(
        2
    )
)

WALK-FORWARD LASSO REGRESSION
Evaluation period: 2018-03-28 to 2026-07-01
Monthly rebalances: 101
Features supplied: 27
Lasso alpha: 0.001
Median non-zero coefficients: 10
Training-only standardisation: PASSED
Leakage-safe cutoffs: PASSED
One-trading-day execution lag: PASSED
Transaction-cost-adjusted backtest: PASSED
Lasso scope completion: PASSED

LASSO PREDICTION METRICS


,Rebalances,Predictions,Mean Absolute Error,Root Mean Squared Error,Pooled R-Squared,Mean Rank IC,Median Rank IC,Positive Rank IC Rate,Directional Accuracy,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate,Median Non-Zero Coefficients,Minimum Non-Zero Coefficients,Maximum Non-Zero Coefficients
Model,,,,,,,,,,,,,,,,
Lasso Regression,101,1010,5.48,7.31,-1.42,0.77,-0.61,48.51,50.99,2.32,1.3,57.76,50.83,10.0,8,14



LASSO STRATEGY PERFORMANCE


,Start Date,End Date,Years,Ending Value (₹),Total Return,CAGR,Annualised Volatility,Sharpe Ratio,Maximum Drawdown,Calmar Ratio,Beta vs Nifty,Correlation vs Nifty,Total One-Way Turnover,Annualised Turnover,Transaction Costs (₹),Rebalances
Strategy,,,,,,,,,,,,,,,,
Lasso Regression,2018-04-02,2026-07-30,8.33,5702946.41,470.29,23.26,21.04,0.82,-29.82,0.78,0.85,69.27,5444.92,653.98,228612.54,101



LASSO VERSUS MAJOR STRATEGIES


,Ending Value (₹),CAGR,Annualised Volatility,Sharpe Ratio,Maximum Drawdown,Annualised Turnover,Transaction Costs (₹)
Strategy,,,,,,,
Regime-Aware Walk-Forward,11752022.41,34.44,21.15,1.24,-27.69,652.35,354506.83
1-Month Reversal,8460873.01,29.24,21.54,1.03,-33.56,865.42,449638.41
Equal-Weight India 10,4816198.61,20.78,16.70,0.86,-30.47,41.41,10846.57
Linear Regression,5984914.98,23.97,21.22,0.84,-29.82,730.44,253682.90
Ridge Regression,5984914.98,23.97,21.22,0.84,-29.82,730.44,253682.90
12-Month Momentum,6256936.61,24.64,22.35,0.84,-30.47,288.58,90850.69
Lasso Regression,5702946.41,23.26,21.04,0.82,-29.82,653.98,228612.54
Nifty 50,2381279.58,10.98,17.09,0.34,-38.44,0.00,0.00


## Final v0.6 Machine-Learning Research Scorecard

This section consolidates the complete v0.6 research into:

- Predictive-model scorecard
- Final strategy leaderboard
- Latest historical signal allocations
- Latest model rankings
- Robustness and statistical-evidence summary
- Research conclusions and limitations

The latest allocation is the final signal available in the notebook dataset. It is a historical research output, not a live investment recommendation.

In [31]:
# =========================================================
# FINAL V0.6 MACHINE-LEARNING RESEARCH SCORECARD
# =========================================================

required_objects = [
    "baseline_evaluation",
    "regression_evaluation",
    "lasso_evaluation",
    "classification_evaluation",
    "tree_regression_evaluation",
    "final_all_strategy_backtest_summary",
    "monthly_strategy_weights_with_lasso",
    "advanced_strategy_weights",
    "regime_aware_strategy_weights",
    "combined_strategy_signals",
    "lasso_predictions",
    "market_regime_frame",
    "regime_selector_history",
    "selector_performance_range",
    "regime_aware_significance",
    "financial_year_stability_summary",
    "rolling_3y_stability_summary",
    "cost_robustness_summary",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required previous-step objects are missing:\n"
        + "\n".join(
            missing_objects
        )
    )


PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------
# 1. Build a unified predictive-model scorecard
# ---------------------------------------------------------

predictive_scorecard_records = []


def append_predictive_evaluation(
    evaluation_frame,
    model_family,
    prediction_task,
):

    for model_name, model_row in (
        evaluation_frame.iterrows()
    ):

        predictive_scorecard_records.append(
            {
                "Model":
                    model_name,

                "Model Family":
                    model_family,

                "Prediction Task":
                    prediction_task,

                "Rebalances":
                    model_row.get(
                        "Rebalances",
                        np.nan,
                    ),

                "Predictions":
                    model_row.get(
                        "Predictions",
                        np.nan,
                    ),

                "Mean Rank IC":
                    model_row.get(
                        "Mean Rank IC",
                        np.nan,
                    ),

                "Median Rank IC":
                    model_row.get(
                        "Median Rank IC",
                        np.nan,
                    ),

                "Positive Rank IC Rate":
                    model_row.get(
                        "Positive Rank IC Rate",
                        np.nan,
                    ),

                "Mean Selected Forward Return":
                    model_row.get(
                        "Mean Selected Forward Return",
                        np.nan,
                    ),

                "Mean Selected Excess Return":
                    model_row.get(
                        "Mean Selected Excess Return",
                        np.nan,
                    ),

                "Selected Positive Return Rate":
                    model_row.get(
                        "Selected Positive Return Rate",
                        np.nan,
                    ),

                "Selected Outperformance Rate":
                    model_row.get(
                        "Selected Outperformance Rate",
                        np.nan,
                    ),

                "Mean Absolute Error":
                    model_row.get(
                        "Mean Absolute Error",
                        np.nan,
                    ),

                "Root Mean Squared Error":
                    model_row.get(
                        "Root Mean Squared Error",
                        np.nan,
                    ),

                "Pooled R-Squared":
                    model_row.get(
                        "Pooled R-Squared",
                        np.nan,
                    ),

                "Directional Accuracy":
                    model_row.get(
                        "Directional Accuracy",
                        np.nan,
                    ),

                "Accuracy":
                    model_row.get(
                        "Accuracy",
                        np.nan,
                    ),

                "Balanced Accuracy":
                    model_row.get(
                        "Balanced Accuracy",
                        np.nan,
                    ),

                "ROC AUC":
                    model_row.get(
                        "ROC AUC",
                        np.nan,
                    ),

                "Brier Score":
                    model_row.get(
                        "Brier Score",
                        np.nan,
                    ),

                "Median Non-Zero Coefficients":
                    model_row.get(
                        "Median Non-Zero Coefficients",
                        np.nan,
                    ),
            }
        )


append_predictive_evaluation(
    evaluation_frame=baseline_evaluation,
    model_family="Rule-Based Baseline",
    prediction_task="Cross-Sectional Return Ranking",
)

append_predictive_evaluation(
    evaluation_frame=regression_evaluation,
    model_family="Linear Regression",
    prediction_task="Forward 21-Day Excess Return",
)

append_predictive_evaluation(
    evaluation_frame=lasso_evaluation,
    model_family="Sparse Linear Regression",
    prediction_task="Forward 21-Day Excess Return",
)

append_predictive_evaluation(
    evaluation_frame=classification_evaluation.loc[
        [
            "Positive Return Logistic"
        ]
    ],
    model_family="Logistic Classification",
    prediction_task="Probability of Positive Return",
)

append_predictive_evaluation(
    evaluation_frame=classification_evaluation.loc[
        [
            "Nifty Outperformance Logistic"
        ]
    ],
    model_family="Logistic Classification",
    prediction_task="Probability of Nifty Outperformance",
)

append_predictive_evaluation(
    evaluation_frame=tree_regression_evaluation,
    model_family="Tree-Based Regression",
    prediction_task="Forward 21-Day Excess Return",
)


predictive_model_scorecard = (
    pd.DataFrame(
        predictive_scorecard_records
    )
)


# ---------------------------------------------------------
# 2. Attach backtest results to predictive methods
# ---------------------------------------------------------

backtest_metrics_for_merge = (
    final_all_strategy_backtest_summary[
        [
            "Ending Value (₹)",
            "CAGR",
            "Annualised Volatility",
            "Sharpe Ratio",
            "Maximum Drawdown",
            "Annualised Turnover",
            "Transaction Costs (₹)",
        ]
    ]
    .reset_index()
)

backtest_metrics_for_merge = (
    backtest_metrics_for_merge.rename(
        columns={
            backtest_metrics_for_merge.columns[
                0
            ]:
                "Model",
        }
    )
)

predictive_model_scorecard = (
    predictive_model_scorecard
    .merge(
        backtest_metrics_for_merge,
        on="Model",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "Sharpe Ratio",
            "Mean Selected Excess Return",
        ],
        ascending=[
            False,
            False,
        ],
        na_position="last",
    )
    .reset_index(
        drop=True
    )
)

predictive_model_scorecard[
    "Backtested"
] = (
    predictive_model_scorecard[
        "Sharpe Ratio"
    ]
    .notna()
)


# ---------------------------------------------------------
# 3. Create the final strategy leaderboard
# ---------------------------------------------------------

def classify_strategy_type(
    strategy_name,
):

    if strategy_name == (
        "Regime-Aware Walk-Forward"
    ):
        return "Dynamic Regime Selector"

    if strategy_name in [
        "1-Month Reversal",
        "12-Month Momentum",
    ]:
        return "Rule-Based Signal"

    if strategy_name in [
        "Equal-Weight India 10",
        "Nifty 50",
    ]:
        return "Benchmark"

    if (
        "Probability Weighted"
        in strategy_name
        or "Inverse Volatility"
        in strategy_name
    ):
        return "Advanced ML Construction"

    if strategy_name in [
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression",
    ]:
        return "Linear ML Strategy"

    if "Logistic" in strategy_name:
        return "Classification ML Strategy"

    if strategy_name in [
        "Random Forest",
        "Gradient Boosting",
    ]:
        return "Tree-Based ML Strategy"

    return "Other Research Strategy"


final_strategy_scorecard = (
    final_all_strategy_backtest_summary
    .copy()
)

final_strategy_scorecard[
    "Strategy Type"
] = [
    classify_strategy_type(
        strategy_name
    )
    for strategy_name in (
        final_strategy_scorecard.index
    )
]

final_strategy_scorecard[
    "CAGR Rank"
] = (
    final_strategy_scorecard[
        "CAGR"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

final_strategy_scorecard[
    "Sharpe Rank"
] = (
    final_strategy_scorecard[
        "Sharpe Ratio"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

# Less negative drawdown is better.
final_strategy_scorecard[
    "Drawdown Rank"
] = (
    final_strategy_scorecard[
        "Maximum Drawdown"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

final_strategy_scorecard[
    "Turnover Rank"
] = (
    final_strategy_scorecard[
        "Annualised Turnover"
    ]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

nifty_cagr = (
    final_strategy_scorecard.loc[
        "Nifty 50",
        "CAGR",
    ]
)

equal_weight_cagr = (
    final_strategy_scorecard.loc[
        "Equal-Weight India 10",
        "CAGR",
    ]
)

final_strategy_scorecard[
    "CAGR Advantage vs Nifty"
] = (
    final_strategy_scorecard[
        "CAGR"
    ]
    - nifty_cagr
)

final_strategy_scorecard[
    "CAGR Advantage vs Equal Weight"
] = (
    final_strategy_scorecard[
        "CAGR"
    ]
    - equal_weight_cagr
)

final_strategy_scorecard[
    "Beats Nifty on CAGR"
] = (
    final_strategy_scorecard[
        "CAGR"
    ]
    > nifty_cagr
)

final_strategy_scorecard[
    "Beats Equal Weight on CAGR"
] = (
    final_strategy_scorecard[
        "CAGR"
    ]
    > equal_weight_cagr
)

final_strategy_scorecard = (
    final_strategy_scorecard
    .sort_values(
        "Sharpe Ratio",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 4. Combine every latest portfolio allocation
# ---------------------------------------------------------

base_allocation_source = (
    monthly_strategy_weights_with_lasso
    .copy()
)

base_allocation_source[
    "Source Strategy"
] = (
    base_allocation_source[
        "Strategy"
    ]
)

base_allocation_source[
    "Construction Method"
] = np.where(
    base_allocation_source[
        "Strategy"
    ]
    == "Equal-Weight India 10",
    "Equal Weight — 10 Stocks",
    "Top-3 Equal Weight",
)

base_allocation_source[
    "Regime"
] = np.nan


advanced_allocation_source = (
    advanced_strategy_weights
    .copy()
)

advanced_allocation_source[
    "Source Strategy"
] = (
    advanced_allocation_source[
        "Strategy"
    ]
)

advanced_allocation_source[
    "Regime"
] = np.nan


regime_allocation_source = (
    regime_aware_strategy_weights
    .copy()
)

regime_allocation_source[
    "Score"
] = (
    regime_allocation_source[
        "Selection Score"
    ]
)

regime_allocation_source[
    "Construction Method"
] = (
    "Prior Same-Regime Sharpe Selection"
)


allocation_columns = [
    "Strategy",
    "Signal Date",
    "Execution Date",
    "Ticker",
    "Score",
    "Selected",
    "Target Weight",
    "Source Strategy",
    "Construction Method",
    "Regime",
]


# Important correction:
# Each sliced DataFrame is copied before its date columns
# are modified, preventing SettingWithCopyWarning.
allocation_sources = [
    base_allocation_source[
        allocation_columns
    ].copy(),

    advanced_allocation_source[
        allocation_columns
    ].copy(),

    regime_allocation_source[
        allocation_columns
    ].copy(),
]


for allocation_frame in allocation_sources:

    allocation_frame.loc[
        :,
        "Signal Date",
    ] = pd.to_datetime(
        allocation_frame[
            "Signal Date"
        ]
    )

    allocation_frame.loc[
        :,
        "Execution Date",
    ] = pd.to_datetime(
        allocation_frame[
            "Execution Date"
        ]
    )


latest_common_signal_date = min(
    allocation_frame[
        "Signal Date"
    ].max()
    for allocation_frame in (
        allocation_sources
    )
)

latest_portfolio_allocations = (
    pd.concat(
        [
            allocation_frame.loc[
                allocation_frame[
                    "Signal Date"
                ]
                == latest_common_signal_date
            ]
            .copy()
            for allocation_frame in (
                allocation_sources
            )
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "Strategy",
            "Target Weight",
            "Ticker",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)

latest_market_regime = (
    market_regime_frame.loc[
        latest_common_signal_date,
        "Regime",
    ]
)

latest_portfolio_allocations[
    "Market Regime"
] = (
    latest_market_regime
)


# ---------------------------------------------------------
# 5. Create a readable latest-allocation summary
# ---------------------------------------------------------

latest_portfolio_summary_records = []

for strategy_name, strategy_weights in (
    latest_portfolio_allocations.groupby(
        "Strategy"
    )
):

    selected_weights = (
        strategy_weights.loc[
            strategy_weights[
                "Target Weight"
            ]
            > 0
        ]
        .sort_values(
            "Target Weight",
            ascending=False,
        )
        .copy()
    )

    holdings_text = "; ".join(
        (
            f"{row['Ticker']} "
            f"({row['Target Weight']:.1%})"
        )
        for _, row in (
            selected_weights.iterrows()
        )
    )

    source_strategies = (
        strategy_weights[
            "Source Strategy"
        ]
        .dropna()
        .astype(str)
        .unique()
    )

    source_strategy = (
        source_strategies[
            0
        ]
        if len(
            source_strategies
        ) > 0
        else strategy_name
    )

    construction_methods = (
        strategy_weights[
            "Construction Method"
        ]
        .dropna()
        .astype(str)
        .unique()
    )

    construction_method = (
        construction_methods[
            0
        ]
        if len(
            construction_methods
        ) > 0
        else "Not Available"
    )

    latest_portfolio_summary_records.append(
        {
            "Strategy":
                strategy_name,

            "Signal Date":
                latest_common_signal_date,

            "Execution Date":
                strategy_weights[
                    "Execution Date"
                ].iloc[
                    0
                ],

            "Market Regime":
                latest_market_regime,

            "Source Strategy":
                source_strategy,

            "Construction Method":
                construction_method,

            "Active Holdings":
                len(
                    selected_weights
                ),

            "Largest Weight":
                selected_weights[
                    "Target Weight"
                ].max(),

            "Holdings":
                holdings_text,
        }
    )


latest_portfolio_summary = (
    pd.DataFrame(
        latest_portfolio_summary_records
    )
    .set_index(
        "Strategy"
    )
    .sort_index()
)


# ---------------------------------------------------------
# 6. Create the latest direct signal rankings
# ---------------------------------------------------------

lasso_signal_frame = (
    lasso_predictions[
        [
            "Date",
            "Ticker",
            "Predicted Excess Return",
        ]
    ]
    .rename(
        columns={
            "Predicted Excess Return":
                "Score",
        }
    )
    .copy()
)

lasso_signal_frame[
    "Strategy"
] = "Lasso Regression"

lasso_signal_frame[
    "Signal Type"
] = (
    "Sparse Regression Model"
)


all_direct_signal_history = (
    pd.concat(
        [
            combined_strategy_signals[
                [
                    "Date",
                    "Ticker",
                    "Score",
                    "Strategy",
                    "Signal Type",
                ]
            ],
            lasso_signal_frame[
                [
                    "Date",
                    "Ticker",
                    "Score",
                    "Strategy",
                    "Signal Type",
                ]
            ],
        ],
        ignore_index=True,
    )
)

all_direct_signal_history[
    "Date"
] = pd.to_datetime(
    all_direct_signal_history[
        "Date"
    ]
)

latest_signal_rankings = (
    all_direct_signal_history.loc[
        all_direct_signal_history[
            "Date"
        ]
        == latest_common_signal_date
    ]
    .copy()
)

latest_signal_rankings[
    "Signal Rank"
] = (
    latest_signal_rankings
    .groupby(
        "Strategy"
    )[
        "Score"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

latest_base_selection_lookup = (
    latest_portfolio_allocations[
        [
            "Strategy",
            "Ticker",
            "Selected",
            "Target Weight",
        ]
    ]
    .drop_duplicates(
        subset=[
            "Strategy",
            "Ticker",
        ]
    )
    .copy()
)

latest_signal_rankings = (
    latest_signal_rankings
    .merge(
        latest_base_selection_lookup,
        on=[
            "Strategy",
            "Ticker",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "Strategy",
            "Signal Rank",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 7. Derive final evidence-based research conclusions
# ---------------------------------------------------------

best_overall_strategy = (
    final_strategy_scorecard[
        "Sharpe Ratio"
    ]
    .idxmax()
)

simple_strategy_names = [
    "1-Month Reversal",
    "12-Month Momentum",
]

best_simple_strategy = (
    final_strategy_scorecard.loc[
        simple_strategy_names,
        "Sharpe Ratio",
    ]
    .idxmax()
)

base_ml_strategy_names = [
    "Linear Regression",
    "Ridge Regression",
    "Lasso Regression",
    "Positive Return Logistic",
    "Nifty Outperformance Logistic",
    "Random Forest",
    "Gradient Boosting",
]

best_ml_backtest_strategy = (
    final_strategy_scorecard.loc[
        base_ml_strategy_names,
        "Sharpe Ratio",
    ]
    .idxmax()
)

ml_prediction_rows = (
    predictive_model_scorecard.loc[
        predictive_model_scorecard[
            "Model"
        ]
        .isin(
            base_ml_strategy_names
        )
    ]
)

best_ml_forward_row = (
    ml_prediction_rows
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
    .iloc[
        0
    ]
)

reversal_forward_excess = (
    predictive_model_scorecard.loc[
        predictive_model_scorecard[
            "Model"
        ]
        == "1-Month Reversal",
        "Mean Selected Excess Return",
    ]
    .iloc[
        0
    ]
)

highest_ml_forward_excess = (
    best_ml_forward_row[
        "Mean Selected Excess Return"
    ]
)

most_cost_robust_strategy = (
    cost_robustness_summary
    .drop(
        index=[
            "Nifty 50"
        ],
        errors="ignore",
    )[
        "100bps CAGR"
    ]
    .idxmax()
)

selector_minimum_cagr = (
    selector_performance_range.loc[
        "CAGR",
        "Minimum",
    ]
)

selector_median_cagr = (
    selector_performance_range.loc[
        "CAGR",
        "Median",
    ]
)

selector_maximum_cagr = (
    selector_performance_range.loc[
        "CAGR",
        "Maximum",
    ]
)

alpha_vs_equal_weight = (
    regime_aware_significance.loc[
        "Equal-Weight India 10",
        "Annualised Arithmetic Alpha",
    ]
)

alpha_vs_nifty = (
    regime_aware_significance.loc[
        "Nifty 50",
        "Annualised Arithmetic Alpha",
    ]
)

fdr_vs_equal_weight = bool(
    regime_aware_significance.loc[
        "Equal-Weight India 10",
        "FDR Significant at 5%",
    ]
)

fdr_vs_nifty = bool(
    regime_aware_significance.loc[
        "Nifty 50",
        "FDR Significant at 5%",
    ]
)

regime_aware_fy_row = (
    financial_year_stability_summary.loc[
        "Regime-Aware Walk-Forward"
    ]
)

regime_aware_rolling_row = (
    rolling_3y_stability_summary.loc[
        "Regime-Aware Walk-Forward"
    ]
)

latest_regime_choice = (
    regime_selector_history.loc[
        regime_selector_history[
            "Signal Date"
        ]
        == latest_common_signal_date,
        "Selected Strategy",
    ]
    .iloc[
        0
    ]
)


research_conclusions = pd.DataFrame(
    [
        {
            "Finding":
                "Overall historical leader",

            "Evidence":
                (
                    f"{best_overall_strategy} produced "
                    f"{final_strategy_scorecard.loc[best_overall_strategy, 'CAGR']:.2%} "
                    f"CAGR and a "
                    f"{final_strategy_scorecard.loc[best_overall_strategy, 'Sharpe Ratio']:.2f} "
                    f"Sharpe ratio."
                ),

            "Interpretation":
                (
                    "The dynamic regime selector was the strongest "
                    "historical strategy after transaction costs."
                ),
        },

        {
            "Finding":
                "Strongest simple signal",

            "Evidence":
                (
                    f"{best_simple_strategy} produced "
                    f"{final_strategy_scorecard.loc[best_simple_strategy, 'CAGR']:.2%} "
                    f"CAGR and a "
                    f"{final_strategy_scorecard.loc[best_simple_strategy, 'Sharpe Ratio']:.2f} "
                    f"Sharpe ratio."
                ),

            "Interpretation":
                (
                    "Simple price-based signals remained difficult "
                    "for the ML models to beat."
                ),
        },

        {
            "Finding":
                "Machine-learning comparison",

            "Evidence":
                (
                    f"The best ML forward-selection result was "
                    f"{best_ml_forward_row['Model']} at "
                    f"{highest_ml_forward_excess:.2%} mean excess return, "
                    f"versus {reversal_forward_excess:.2%} for "
                    f"1-Month Reversal."
                ),

            "Interpretation":
                (
                    "No tested ML model exceeded the reversal "
                    "baseline on the primary forward-selection measure."
                ),
        },

        {
            "Finding":
                "Best backtested ML strategy",

            "Evidence":
                (
                    f"{best_ml_backtest_strategy} had the highest "
                    f"Sharpe ratio among the base ML strategies at "
                    f"{final_strategy_scorecard.loc[best_ml_backtest_strategy, 'Sharpe Ratio']:.2f}."
                ),

            "Interpretation":
                (
                    "ML added useful diversification but was not "
                    "the standalone research winner."
                ),
        },

        {
            "Finding":
                "Regime-selector robustness",

            "Evidence":
                (
                    f"Six selector variants produced "
                    f"{selector_minimum_cagr:.2%} to "
                    f"{selector_maximum_cagr:.2%} CAGR, with a "
                    f"{selector_median_cagr:.2%} median."
                ),

            "Interpretation":
                (
                    "Performance did not depend entirely on the "
                    "base 63-observation selector specification."
                ),
        },

        {
            "Finding":
                "Statistical evidence",

            "Evidence":
                (
                    f"Annualised arithmetic alpha was "
                    f"{alpha_vs_equal_weight:.2%} versus equal weight "
                    f"and {alpha_vs_nifty:.2%} versus Nifty 50. "
                    f"FDR significance: equal weight={fdr_vs_equal_weight}, "
                    f"Nifty={fdr_vs_nifty}."
                ),

            "Interpretation":
                (
                    "The regime-aware return advantage remained "
                    "positive after the block-bootstrap multiple-test adjustment."
                ),
        },

        {
            "Finding":
                "Time stability",

            "Evidence":
                (
                    f"The regime-aware strategy was positive in "
                    f"{regime_aware_fy_row['Positive_FY_Rate']:.0%} "
                    f"of complete Indian financial years and had a "
                    f"{regime_aware_rolling_row['Minimum_Rolling_CAGR']:.2%} "
                    f"minimum rolling three-year CAGR."
                ),

            "Interpretation":
                (
                    "The result was not concentrated entirely in "
                    "one isolated financial year."
                ),
        },

        {
            "Finding":
                "Severe-cost resilience",

            "Evidence":
                (
                    f"{most_cost_robust_strategy} had the highest "
                    f"100-basis-point cost-scenario CAGR among the "
                    f"tested original strategies."
                ),

            "Interpretation":
                (
                    "Longer-term momentum was more resilient than "
                    "high-turnover reversal under severe trading costs."
                ),
        },

        {
            "Finding":
                "Latest historical regime allocation",

            "Evidence":
                (
                    f"On {latest_common_signal_date.date()}, the "
                    f"market regime was {latest_market_regime} and "
                    f"the regime selector chose {latest_regime_choice}."
                ),

            "Interpretation":
                (
                    "This is the final historical notebook signal, "
                    "not a live recommendation."
                ),
        },

        {
            "Finding":
                "Material research limitation",

            "Evidence":
                (
                    "The universe contains a fixed India 10 stock set "
                    "and the candidate strategies were researched on "
                    "the same broad historical sample."
                ),

            "Interpretation":
                (
                    "Survivorship, universe-selection and researcher-"
                    "choice bias may remain despite walk-forward and "
                    "bootstrap controls."
                ),
        },
    ]
)


# ---------------------------------------------------------
# 8. Save final v0.6 scorecard outputs
# ---------------------------------------------------------

final_output_frames = {
    "v06_predictive_model_scorecard.csv":
        (
            predictive_model_scorecard,
            False,
        ),

    "v06_strategy_scorecard.csv":
        (
            final_strategy_scorecard,
            True,
        ),

    "v06_latest_portfolio_allocations.csv":
        (
            latest_portfolio_allocations,
            False,
        ),

    "v06_latest_portfolio_summary.csv":
        (
            latest_portfolio_summary,
            True,
        ),

    "v06_latest_signal_rankings.csv":
        (
            latest_signal_rankings,
            False,
        ),

    "v06_research_conclusions.csv":
        (
            research_conclusions,
            False,
        ),
}

saved_output_paths = []

for output_directory in [
    PROCESSED_DATA_DIR,
    OUTPUT_DIR,
]:

    for filename, (
        output_frame,
        include_index,
    ) in final_output_frames.items():

        output_path = (
            output_directory
            / filename
        )

        output_frame.to_csv(
            output_path,
            index=include_index,
        )

        saved_output_paths.append(
            output_path
        )


# ---------------------------------------------------------
# 9. Final validations
# ---------------------------------------------------------

latest_weight_totals = (
    latest_portfolio_allocations
    .groupby(
        "Strategy"
    )[
        "Target Weight"
    ]
    .sum()
)

expected_latest_strategy_count = (
    monthly_strategy_weights_with_lasso[
        "Strategy"
    ].nunique()
    + advanced_strategy_weights[
        "Strategy"
    ].nunique()
    + regime_aware_strategy_weights[
        "Strategy"
    ].nunique()
)

assert (
    predictive_model_scorecard[
        "Model"
    ].nunique()
    == 13
)

assert (
    final_strategy_scorecard.index
    .nunique()
    == 15
)

assert (
    latest_portfolio_allocations[
        "Strategy"
    ].nunique()
    == expected_latest_strategy_count
)

assert np.allclose(
    latest_weight_totals,
    1.0,
    atol=1e-10,
)

assert not latest_portfolio_allocations.duplicated(
    subset=[
        "Strategy",
        "Ticker",
    ]
).any()

assert (
    latest_portfolio_allocations[
        "Execution Date"
    ]
    > latest_portfolio_allocations[
        "Signal Date"
    ]
).all()

assert (
    latest_signal_rankings[
        "Strategy"
    ].nunique()
    == 9
)

assert (
    latest_signal_rankings[
        "Signal Rank"
    ]
    .between(
        1,
        len(
            INDIA_10_TICKERS
        ),
    )
    .all()
)

assert len(
    research_conclusions
) == 10

assert all(
    output_path.exists()
    for output_path in (
        saved_output_paths
    )
)


# ---------------------------------------------------------
# 10. Display the final v0.6 research scorecard
# ---------------------------------------------------------

display_strategy_leaderboard = (
    final_strategy_scorecard[
        [
            "Strategy Type",
            "Ending Value (₹)",
            "CAGR",
            "Annualised Volatility",
            "Sharpe Ratio",
            "Maximum Drawdown",
            "Annualised Turnover",
            "CAGR Rank",
            "Sharpe Rank",
        ]
    ]
    .copy()
)

for column in [
    "CAGR",
    "Annualised Volatility",
    "Maximum Drawdown",
    "Annualised Turnover",
]:

    display_strategy_leaderboard[
        column
    ] = (
        display_strategy_leaderboard[
            column
        ]
        * 100
    )


display_latest_portfolio_summary = (
    latest_portfolio_summary.copy()
)

display_latest_portfolio_summary[
    "Largest Weight"
] = (
    display_latest_portfolio_summary[
        "Largest Weight"
    ]
    * 100
)


print(
    "FINAL V0.6 MACHINE-LEARNING RESEARCH SCORECARD"
)
print(
    "=" * 76
)
print(
    "Predictive methods evaluated:",
    predictive_model_scorecard[
        "Model"
    ].nunique(),
)
print(
    "Transaction-cost-adjusted strategies:",
    final_strategy_scorecard.index.nunique(),
)
print(
    "Latest historical signal date:",
    latest_common_signal_date.date(),
)
print(
    "Latest execution date:",
    latest_portfolio_allocations[
        "Execution Date"
    ].max().date(),
)
print(
    "Latest market regime:",
    latest_market_regime,
)
print(
    "Latest regime-aware selection:",
    latest_regime_choice,
)
print(
    "Best overall strategy:",
    best_overall_strategy,
)
print(
    "Best simple strategy:",
    best_simple_strategy,
)
print(
    "Final output files:",
    len(
        final_output_frames
    ),
)
print(
    "Predictive-model consolidation:",
    "PASSED",
)
print(
    "Latest-allocation validation:",
    "PASSED",
)
print(
    "Warning-free allocation processing:",
    "PASSED",
)
print(
    "Final research scorecard:",
    "PASSED",
)

print(
    "\nFINAL STRATEGY LEADERBOARD"
)
display(
    display_strategy_leaderboard.round(
        2
    )
)

print(
    "\nLATEST HISTORICAL PORTFOLIO SUMMARY"
)
display(
    display_latest_portfolio_summary.round(
        2
    )
)

print(
    "\nFINAL RESEARCH CONCLUSIONS"
)
display(
    research_conclusions
)

FINAL V0.6 MACHINE-LEARNING RESEARCH SCORECARD
Predictive methods evaluated: 13
Transaction-cost-adjusted strategies: 15
Latest historical signal date: 2026-07-01
Latest execution date: 2026-07-02
Latest market regime: Bear / Higher Volatility
Latest regime-aware selection: 1-Month Reversal
Best overall strategy: Regime-Aware Walk-Forward
Best simple strategy: 1-Month Reversal
Final output files: 6
Predictive-model consolidation: PASSED
Latest-allocation validation: PASSED
Warning-free allocation processing: PASSED
Final research scorecard: PASSED

FINAL STRATEGY LEADERBOARD


,Strategy Type,Ending Value (₹),CAGR,Annualised Volatility,Sharpe Ratio,Maximum Drawdown,Annualised Turnover,CAGR Rank,Sharpe Rank
Strategy,,,,,,,,,
Regime-Aware Walk-Forward,Dynamic Regime Selector,11752022.41,34.44,21.15,1.24,-27.69,652.35,1,1
1-Month Reversal,Rule-Based Signal,8460873.01,29.24,21.54,1.03,-33.56,865.42,2,2
Equal-Weight India 10,Benchmark,4816198.61,20.78,16.70,0.86,-30.47,41.41,11,3
Ridge Regression,Linear ML Strategy,5984914.98,23.97,21.22,0.84,-29.82,730.44,4,4
Linear Regression,Linear ML Strategy,5984914.98,23.97,21.22,0.84,-29.82,730.44,4,4
12-Month Momentum,Rule-Based Signal,6256936.61,24.64,22.35,0.84,-30.47,288.58,3,6
Gradient Boosting,Tree-Based ML Strategy,5868523.31,23.68,21.14,0.84,-30.39,738.93,6,7
Positive Return Logistic,Classification ML Strategy,5505116.77,22.74,20.25,0.82,-31.09,669.06,8,8
Lasso Regression,Linear ML Strategy,5702946.41,23.26,21.04,0.82,-29.82,653.98,7,9



LATEST HISTORICAL PORTFOLIO SUMMARY


,Signal Date,Execution Date,Market Regime,Source Strategy,Construction Method,Active Holdings,Largest Weight,Holdings
Strategy,,,,,,,,
1-Month Reversal,2026-07-01,2026-07-02,Bear / Higher Volatility,1-Month Reversal,Top-3 Equal Weight,3,33.33,LT.NS (33.3%); POWERGRID.NS (33.3%); TCS.NS (3...
12-Month Momentum,2026-07-01,2026-07-02,Bear / Higher Volatility,12-Month Momentum,Top-3 Equal Weight,3,33.33,BEL.NS (33.3%); LT.NS (33.3%); SUNPHARMA.NS (3...
Equal-Weight India 10,2026-07-01,2026-07-02,Bear / Higher Volatility,Equal-Weight India 10,Equal Weight — 10 Stocks,10,10.00,BEL.NS (10.0%); BHARTIARTL.NS (10.0%); HDFCBAN...
Gradient Boosting,2026-07-01,2026-07-02,Bear / Higher Volatility,Gradient Boosting,Top-3 Equal Weight,3,33.33,LT.NS (33.3%); SUNPHARMA.NS (33.3%); TCS.NS (3...
Gradient Boosting — Inverse Volatility,2026-07-01,2026-07-02,Bear / Higher Volatility,Gradient Boosting — Inverse Volatility,Top-3 Inverse Volatility,3,39.05,SUNPHARMA.NS (39.0%); LT.NS (33.6%); TCS.NS (2...
Lasso Regression,2026-07-01,2026-07-02,Bear / Higher Volatility,Lasso Regression,Top-3 Equal Weight,3,33.33,LT.NS (33.3%); POWERGRID.NS (33.3%); TCS.NS (3...
Linear Regression,2026-07-01,2026-07-02,Bear / Higher Volatility,Linear Regression,Top-3 Equal Weight,3,33.33,POWERGRID.NS (33.3%); SUNPHARMA.NS (33.3%); TC...
Nifty Outperformance Logistic,2026-07-01,2026-07-02,Bear / Higher Volatility,Nifty Outperformance Logistic,Top-3 Equal Weight,3,33.33,BEL.NS (33.3%); POWERGRID.NS (33.3%); TCS.NS (...
Nifty Outperformance Logistic — Probability Weighted,2026-07-01,2026-07-02,Bear / Higher Volatility,Nifty Outperformance Logistic — Probability We...,Probability Weighted,3,35.39,TCS.NS (35.4%); POWERGRID.NS (32.4%); BEL.NS (...



FINAL RESEARCH CONCLUSIONS


,Finding,Evidence,Interpretation
0,Overall historical leader,Regime-Aware Walk-Forward produced 34.44% CAGR...,The dynamic regime selector was the strongest ...
1,Strongest simple signal,1-Month Reversal produced 29.24% CAGR and a 1....,Simple price-based signals remained difficult ...
2,Machine-learning comparison,The best ML forward-selection result was Linea...,No tested ML model exceeded the reversal basel...
3,Best backtested ML strategy,Linear Regression had the highest Sharpe ratio...,ML added useful diversification but was not th...
4,Regime-selector robustness,Six selector variants produced 27.51% to 34.44...,Performance did not depend entirely on the bas...
5,Statistical evidence,Annualised arithmetic alpha was 11.80% versus ...,The regime-aware return advantage remained pos...
6,Time stability,The regime-aware strategy was positive in 100%...,The result was not concentrated entirely in on...
7,Severe-cost resilience,12-Month Momentum had the highest 100-basis-po...,Longer-term momentum was more resilient than h...
8,Latest historical regime allocation,"On 2026-07-01, the market regime was Bear / Hi...","This is the final historical notebook signal, ..."
9,Material research limitation,The universe contains a fixed India 10 stock s...,"Survivorship, universe-selection and researche..."
